# HỆ THỐNG BẢN SAO SỐ NHÂN QUẢ CROSS-LAYER & TÁC TỬ NHẬN THỨC PPO (WI-FI 6 IEEE 802.11ax)
**Dự án KHKT 2026 — Tối ưu hóa Tài nguyên Vô tuyến Nhận thức (Cognitive Radio Resource Management) trên Phần cứng COTS Router 256MB RAM**

---

### Kiến trúc Tổng thể & Chuỗi Nhân quả Vật lý 4 Tầng (Continuous-Markov POMDP)
Toàn bộ Notebook này (từ Bước 1 đến Bước 4.6) và mã nguồn triển khai thật (`src/envs/digital_twin_env.py`, `src/rl_agent/deploy_hil_agent.py`) vận hành thống nhất trên **Chuỗi Nhân quả Vật lý 4 Tầng** không dùng quy tắc cứng hay `ideal_action`:

$$\text{Phổ thực 9 kênh } + \text{ Tải } + \mathbf{A}_t \longrightarrow \text{Bonding + SINR + Minstrel AMC + } p_c \longrightarrow \text{Goodput + Hàng đợi M/G/1 AQM + Gamma P99 + Loss} \longrightarrow R_t$$

1. **Không gian Trạng thái 8 Chiều (`Box(8,)` — Dual Sensing = MAC Sensing + PHY Preamble Sensing):**
   * `[0] avg_rssi` ($[-90, -30]$ dBm): Cường độ tín hiệu thu trung bình từ `iw station dump`.
   * `[1] noise_floor` ($[-105, -50]$ dBm): Nhiễu nền phổ tần tầng PHY (`iw survey dump [in use]`), phản ánh trực tiếp **nhiễu trên kênh hiện tại đang chọn (`noise_on_chosen_ch`)**.
   * `[2] sta_count` ($[0, 300]$): Số lượng thiết bị đầu cuối (STA) đang kết nối.
   * `[3] retry_rate` ($[0.0, 1.0]$): Tỷ lệ truyền lại khung MPDU tầng MAC.
   * `[4] p_c` ($[0.0, 1.0]$): Xác suất va chạm gói tin CSMA/CA theo mô hình Bianchi DCF.
   * `[5] airtime_util` ($[0.0, 1.0]$): Tỷ lệ chiếm dụng thời gian phát sóng trên kênh (CCA Busy).
   * `[6] ofdma_ratio` ($[0.0, 1.0]$): Tỷ lệ ghép kênh đa truy cập phân chia theo tần số trực giao (HE-MU OFDMA / RU utilization).
   * `[7] is_break_time` ($\{0, 1\}$): Ngữ cảnh thời gian thực (Giờ giải lao / Chuyển tiết gây bão Roaming).

2. **Giải pháp Toán học cho Bài toán POMDP 8D (`VecFrameStack(4)` + `StatefulChannelGuardWrapper`):**
   * Vì vector 8D giữ đúng tính bất biến phần cứng (không thêm chiều thứ 9 `current_channel_idx`), khi chuyển sang kênh mới sạch sóng (`noise_on_chosen_ch <= -85 dBm`) quá 4 bước, cửa sổ `VecFrameStack(4)` không còn lưu vết kênh cũ và một mạng MLP không có Channel ID có thể nhảy ngược về `CH36`.
   * Để giải quyết triệt để và minh bạch bài toán POMDP 8D này mà vẫn dùng `PPO("MlpPolicy")` chuẩn SB3, hệ thống kết hợp **cửa sổ 4 khung hình `VecFrameStack(n_stack=4)`** (nắm bắt động học chuyển tiếp ngắn hạn trên $8 \times 4 = 32$ chiều nội bộ) cùng lớp bảo vệ có trạng thái **`StatefulChannelGuardWrapper`** (khóa giữ kênh sạch hiện tại và cách ly tạm thời kênh vừa sập nhiễu) đồng bộ 1-1 với `deploy_hil_agent.py`.

3. **Không gian Hành động 7 Chiều (`spaces.MultiDiscrete([9, 3, 7, 12, 2, 5, 2])` — 45,360 cấu hình RRM):**
   * `[0] channel`: 9 kênh 5 GHz Non-DFS (`CH36, 40, 44, 48, 149, 153, 157, 161, 165`) | hoặc 3 kênh 2.4 GHz (`1, 6, 11`).
   * `[1] bandwidth`: `0: HE20`, `1: HE40`, `2: HE80` (tự động bảo vệ kênh lẻ `165` lùi về `HE20`).
   * `[2] tx_power`: $[14, 15, 16, 17, 18, 19, 20]$ dBm.
   * `[3] mcs_ceiling`: Trần giới hạn tốc độ Minstrel-HT (`HE-MCS 0..11`).
   * `[4] beacon`: Chu kỳ Beacon (`0: 80 ms` cho bão Roaming, `1: 300 ms` tiết kiệm Airtime).
   * `[5] obss_pd`: Ngưỡng BSS Coloring Spatial Reuse (`[-82, -77, -72, -67, -62]` dBm).
   * `[6] beamform`: SU-Beamforming (`0: OFF` khi CSI Aging/di động cao, `1: ON` tăng $+2.8$ dB SINR).

## BƯỚC 1: KHỞI TẠO KHÔNG GIAN MÔI TRƯỜNG LÕI (`8D Dual-Sensing State` & `7D Action`, Đồng bộ `src/envs/digital_twin_env.py`)
Thiết lập thư mục lưu trữ (Google Drive / Local), định nghĩa hằng số vật lý 8D/7D, bản đồ phân cụm bonding 9 kênh Non-DFS chuẩn IEEE 802.11ax, bộ xếp chồng 4 khung `build_stacked_obs(N_FRAME_STACK=4)` và hàm tính phân vị Gamma lệch phải Wilson–Hilferty (`gamma_p99_from_moments`).

In [1]:
import os
import sys
import glob
import math
import time
import logging
from collections import deque
from typing import Any, Callable, Dict, Optional, Tuple
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.torch_layers import BaseFeaturesExtractor
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

# 1. Tự động nhận diện môi trường Google Colab hoặc Máy cục bộ (Local)
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_BASE_PATH = '/content/drive/MyDrive/KHKT_training_digital_twin_8DState_7DAct'
    os.makedirs(DRIVE_BASE_PATH, exist_ok=True)
    WORK_DIR = '/content/drive/MyDrive/training_digital_twin'
    if os.path.exists(WORK_DIR):
        os.chdir(WORK_DIR)
    print(f"[+] Đã kết nối Google Drive! Thư mục lưu trữ 5 Seeds: {DRIVE_BASE_PATH}")
except Exception:
    _cwd = os.path.abspath(".")
    if os.path.basename(_cwd) == "notebooks":
        DRIVE_BASE_PATH = os.path.join(_cwd, "KHKT_training_digital_twin_8DState_7DAct")
    elif os.path.exists(os.path.join(_cwd, "notebooks")):
        DRIVE_BASE_PATH = os.path.join(_cwd, "notebooks", "KHKT_training_digital_twin_8DState_7DAct")
    else:
        DRIVE_BASE_PATH = os.path.join(_cwd, "KHKT_training_digital_twin_8DState_7DAct")
    os.makedirs(DRIVE_BASE_PATH, exist_ok=True)
    print(f"[+] Đang chạy trên môi trường Local! Thư mục lưu trữ 5 Seeds: {DRIVE_BASE_PATH}")

# 2. Hằng số & Hàm tiện ích chuẩn hóa (Đồng bộ 1-1 với src/envs/digital_twin_env.py)
GLOBAL_SEED = 2026
N_FRAME_STACK = 4
DT_STEP_SEC = 10.0
OU_THETA_RF = 0.105        # s^-1 -> exp(-theta * dt) = 0.3499 (alpha_rf ≈ 0.65)
OU_SIGMA_STAT_RF = 1.25    # dB stationary std within a Markov regime
OU_THETA_RSSI = 0.0223     # s^-1 -> exp(-theta * dt) = 0.8001
OU_SIGMA_STAT_RSSI = 1.35  # dB stationary std for RSSI
OU_THETA_STA = 0.0198      # s^-1 -> exp(-theta * dt) = 0.8204
OU_SIGMA_STAT_STA = 3.10   # STA stationary std

OBS_LOW = np.array([-90.0, -105.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0], dtype=np.float32)
OBS_HIGH = np.array([-30.0, -50.0, 300.0, 1.0, 1.0, 1.0, 1.0, 1.0], dtype=np.float32)
STATE_NAMES = [
    "avg_rssi", "noise_floor", "sta_count", "retry_rate",
    "p_c", "airtime_util", "ofdma_ratio", "is_break_time",
]

NON_DFS_5G_CHANNELS = [36, 40, 44, 48, 149, 153, 157, 161, 165]
CHANNEL_FREQ_MHZ = np.array([5180, 5200, 5220, 5240, 5745, 5765, 5785, 5805, 5825], dtype=np.float32)
CHANNELS_5G = NON_DFS_5G_CHANNELS
CHANNELS_2G = [1, 6, 11]

ACTION_NVEC = [9, 3, 7, 12, 2, 5, 2]
ACTION_NAMES = ["channel", "bandwidth", "tx_power", "mcs_ceiling", "beacon", "obss_pd", "beamform"]

BANDWIDTHS = ["HE20", "HE40", "HE80"]
TX_POWER_DBM = np.array([14, 15, 16, 17, 18, 19, 20], dtype=np.float64)
BEACON_INTS = [80, 300]
OBSS_PD_DBM = np.array([-82, -77, -72, -67, -62], dtype=np.float64)
OBSS_PD_OFFSETS = [0, 5, 10, 15, 20]

QOE_LATENCY_MS = 15.0
QOE_LOSS = 0.01

BONDING_GROUP_40MHZ = {
    36: (36, 40),    40: (36, 40),
    44: (44, 48),    48: (44, 48),
    149: (149, 153), 153: (149, 153),
    157: (157, 161), 161: (157, 161),
    165: (165,),
}

BONDING_GROUP_80MHZ = {
    36: (36, 40, 44, 48), 40: (36, 40, 44, 48), 44: (36, 40, 44, 48), 48: (36, 40, 44, 48),
    149: (149, 153, 157, 161), 153: (149, 153, 157, 161), 157: (149, 153, 157, 161), 161: (149, 153, 157, 161),
    165: (165,),
}


def resolve_valid_5g_channel_and_bw(ch_idx: int, bw_idx: int) -> Tuple[int, str, Tuple[int, ...], bool]:
    """Ánh xạ an toàn (ch_idx, bw_idx) sang kênh 5GHz Non-DFS và cụm bonding IEEE 802.11ax hợp lệ."""
    ch_idx_safe = int(np.clip(ch_idx, 0, len(NON_DFS_5G_CHANNELS) - 1))
    bw_idx_safe = int(np.clip(bw_idx, 0, 2))
    primary_ch = NON_DFS_5G_CHANNELS[ch_idx_safe]
    if primary_ch == 165 and bw_idx_safe > 0:
        return 165, "HE20", (165,), True
    if bw_idx_safe == 0:
        return primary_ch, "HE20", (primary_ch,), False
    if bw_idx_safe == 1:
        return primary_ch, "HE40", BONDING_GROUP_40MHZ[primary_ch], False
    return primary_ch, "HE80", BONDING_GROUP_80MHZ[primary_ch], False


def measure_mt76_survey_noise(spectrum_noise_9d: np.ndarray, ch_idx: int, bw_idx: int) -> float:
    """
    Mô hình hóa phép đo nhiễu khảo sát của driver OpenWrt `mt76` (`iw survey dump [in use]`):
      - Ưu tiên năng lượng nhiễu IPI trên Kênh chính 20 MHz (`primary_ch`).
      - Khi gộp HE40/HE80, tính thêm ghép xuyên băng gốc suy giảm -4.5 dB từ kênh phụ mạnh nhất.
    """
    ch_safe = int(np.clip(ch_idx, 0, len(NON_DFS_5G_CHANNELS) - 1))
    _, actual_htmode, occ_subs, _ = resolve_valid_5g_channel_and_bw(ch_safe, bw_idx)
    pri_n = float(spectrum_noise_9d[ch_safe])
    if actual_htmode == "HE20":
        return pri_n
    sec_vals = [
        float(spectrum_noise_9d[NON_DFS_5G_CHANNELS.index(c)])
        for c in occ_subs
        if NON_DFS_5G_CHANNELS.index(c) != ch_safe
    ]
    if not sec_vals:
        return pri_n
    sec_max = max(sec_vals)
    mw = (10.0 ** (pri_n / 10.0)) + (10.0 ** ((sec_max - 4.5) / 10.0))
    return float(10.0 * math.log10(mw if mw > 1e-12 else 1e-12))


def gamma_p99_from_moments(mean_lat: float, jitter: float, d_min: float = 1.50) -> float:
    """
    Tính phân vị thứ 99 chính xác của phân phối Gamma dịch chuyển X = d_min + Gamma(k, theta)
    thông qua xấp xỉ Wilson-Hilferty (khớp < 0.1% so với scipy.stats.gamma.ppf(0.99), không cần scipy).
    """
    excess_mean = max(0.05, float(mean_lat) - d_min)
    sigma = max(0.04, float(jitter))
    shape_k = max(0.25, (excess_mean / sigma) ** 2.0)
    z99 = 2.326347874
    wh_term = max(0.15, 1.0 - 1.0 / (9.0 * shape_k) + z99 / (3.0 * math.sqrt(shape_k)))
    return float(d_min + excess_mean * (wh_term ** 3.0))


def piecewise_lr_schedule(initial_value: float = 3.5e-4, min_lr: float = 2.5e-5) -> Callable[[float], float]:
    """Lịch trình giảm tốc độ học có sàn chặn dưới (Floor-clamped Power Decay)."""
    def schedule(progress_remaining: float) -> float:
        decayed = initial_value * (max(float(progress_remaining), 0.0) ** 1.35)
        return float(max(min_lr, decayed))
    return schedule


def build_stacked_obs(obs_history: Any, n_stack: int = N_FRAME_STACK) -> np.ndarray:
    """Xếp chồng `n_stack` khung 8D (zero-pad bên trái nếu thiếu, khớp 100% VecFrameStack SB3)."""
    if isinstance(obs_history, np.ndarray) and obs_history.ndim == 1 and obs_history.shape[0] == 8 * n_stack:
        return obs_history.astype(np.float32)
    frames = list(obs_history) if not (isinstance(obs_history, np.ndarray) and obs_history.ndim == 1) else [obs_history]
    frames = [np.asarray(f, dtype=np.float32).reshape(8) for f in frames[-n_stack:]]
    while len(frames) < n_stack:
        frames.insert(0, np.zeros(8, dtype=np.float32))
    return np.concatenate(frames, axis=0).astype(np.float32)

print(f"[+] Đã khởi tạo chuẩn Trạng thái 8D Dual-Sensing (VecFrameStack={N_FRAME_STACK} -> {8 * N_FRAME_STACK}D): {STATE_NAMES}")
print(f"[+] Đã khởi tạo chuẩn Hành động 7D MultiDiscrete({ACTION_NVEC}): tổng cộng {int(np.prod(ACTION_NVEC)):,} cấu hình RRM.")

[+] Đang chạy trên môi trường Local! Thư mục lưu trữ 5 Seeds: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct
[+] Đã khởi tạo chuẩn Trạng thái 8D Dual-Sensing (VecFrameStack=4 -> 32D): ['avg_rssi', 'noise_floor', 'sta_count', 'retry_rate', 'p_c', 'airtime_util', 'ofdma_ratio', 'is_break_time']
[+] Đã khởi tạo chuẩn Hành động 7D MultiDiscrete([9, 3, 7, 12, 2, 5, 2]): tổng cộng 45,360 cấu hình RRM.


## BƯỚC 2: LỚP BỌC ĐA HÌNH PHẦN CỨNG (`HardwareAwareWrapper`) & BẢO VỆ KÊNH CÓ TRẠNG THÁI (`StatefulChannelGuardWrapper`)
1. **`HardwareAwareWrapper`**: Kẹp hành động an toàn theo thế hệ router (`WIFI_6`, `WIFI_5_AC`, `WIFI_4_LEGACY`) và bảo vệ kênh lẻ `165` luôn chạy `HE20`.
2. **`StatefulChannelGuardWrapper`**: Giải pháp POMDP 8D đồng nhất với `deploy_hil_agent.py`:
   * Chỉ đọc đúng `current_obs` mà tác tử nhìn thấy (tuyệt đối không nhìn trộm nhiễu PHY ẩn khi huấn luyện `PPO MAC-Only`).
   * Khi kênh hiện tại sạch (`noise <= -84.0 dBm` và MAC không sập), giữ nguyên kênh hiện tại để tránh bẫy nhảy ngược về CH36 sau khi hết cửa sổ 4 bước; khi kênh hiện tại suy giảm, đưa vào danh sách tạm tránh (*quarantine*) 12 bước để PPO thoát sang kênh sạch mới.

In [2]:
class HardwareAwareWrapper(gym.ActionWrapper):
    """Lớp Bọc Đa Hình Tự Nhận Thức Phần Cứng (Wi-Fi 6 / Wi-Fi 5 AC / Wi-Fi 4 Legacy)."""
    SUPPORTED_PROFILES = {
        "WIFI_6": {
            "max_bw_idx": 2,
            "max_mcs_idx": 11,
            "allow_obss_pd": True,
            "allow_beamform": True,
            "active_weapons": ["channel", "bandwidth", "tx_power", "mcs_ceiling", "beacon_int", "obss_pd", "beamforming"],
        },
        "WIFI_5_AC": {
            "max_bw_idx": 2,
            "max_mcs_idx": 9,
            "allow_obss_pd": False,
            "allow_beamform": True,
            "active_weapons": ["channel", "bandwidth", "tx_power", "mcs_ceiling", "beacon_int", "beamforming"],
        },
        "WIFI_4_LEGACY": {
            "max_bw_idx": 1,
            "max_mcs_idx": 7,
            "allow_obss_pd": False,
            "allow_beamform": False,
            "active_weapons": ["channel", "bandwidth", "tx_power", "mcs_ceiling", "beacon_int"],
        },
    }

    def __init__(self, env: gym.Env, hardware_type: str = "WIFI_6", verbose: bool = False):
        super().__init__(env)
        self.hardware_type = hardware_type.upper()
        if self.hardware_type not in self.SUPPORTED_PROFILES:
            self.hardware_type = "WIFI_6"
        self.profile = self.SUPPORTED_PROFILES[self.hardware_type]
        if hasattr(self.env.unwrapped, "hardware_type"):
            self.env.unwrapped.hardware_type = self.hardware_type
        if verbose:
            if self.hardware_type == "WIFI_6":
                print("[+] Đã phát hiện Router Wi-Fi 6 (802.11ax MediaTek). Kích hoạt toàn bộ 7/7 vũ khí RRM (9 kênh Non-DFS).")
            elif self.hardware_type == "WIFI_5_AC":
                print("[!] Đã phát hiện Router Wi-Fi 5 (802.11ac). Khóa MCS<=9 và cố định OBSS-PD mặc định.")
            else:
                print("[!] CẢNH BÁO: Đã phát hiện Router Wi-Fi 4 (802.11n Legacy). Fallback (BW<=40MHz, MCS<=7, tắt OBSS-PD & BF).")

    def set_hardware_profile(self, hardware_type: str):
        self.hardware_type = hardware_type.upper()
        self.profile = self.SUPPORTED_PROFILES[self.hardware_type]
        if hasattr(self.env.unwrapped, "hardware_type"):
            self.env.unwrapped.hardware_type = self.hardware_type

    def action(self, action: np.ndarray) -> np.ndarray:
        act = np.array(action, dtype=np.int64, copy=True).reshape(-1)
        act[0] = int(np.clip(act[0], 0, len(NON_DFS_5G_CHANNELS) - 1))
        act[1] = int(np.clip(act[1], 0, self.profile["max_bw_idx"]))
        if act[0] == 8 and act[1] > 0:
            act[1] = 0
        act[2] = int(np.clip(act[2], 0, 6))
        act[3] = int(np.clip(act[3], 0, self.profile["max_mcs_idx"]))
        act[4] = int(np.clip(act[4], 0, 1))
        act[5] = int(np.clip(act[5], 0, 4)) if self.profile["allow_obss_pd"] else 0
        act[6] = int(np.clip(act[6], 0, 1)) if self.profile["allow_beamform"] else 0
        return act

    def simulate_future(self, current_real_state: np.ndarray, action: np.ndarray) -> float:
        safe_act = self.action(action)
        return self.env.unwrapped.simulate_future(current_real_state, safe_act)


class StatefulChannelGuardWrapper(gym.Wrapper):
    """
    Lớp Bảo vệ Trạng thái Kênh & Cách ly Nhiễu (Stateful POMDP Hysteresis & Tabu Guardrail):
    - Đọc đúng `current_obs` mà Agent quan sát được (tuyệt đối KHÔNG nhìn trộm `true_current_obs` khi `obs_mask_phy=True`).
    - Triệt tiêu hoàn toàn bẫy "mất trí nhớ sau 4 bước" của chính sách 8D + VecFrameStack(4):
      1) Khi kênh hiện tại đang sạch (`noise <= -84.0 dBm`) và không sập MAC nghiêm trọng (`p_c < 0.66` hoặc `retry < 0.60`),
         giữ ổn định kênh hiện tại thay vì nhảy ngược về CH36 (Zero Ping-Pong), vẫn để PPO tự do chỉnh 6 chiều còn lại.
      2) Ngoại lệ: cho phép rời kênh lẻ 165 nếu tải nhẹ cần mở rộng băng thông HE40/HE80 trên kênh khác.
      3) Khi kênh hiện tại bị nhiễu (`noise > -84.0 dBm`) hoặc sập MAC kéo dài qua ngưỡng dwell,
         mở cổng để PPO thoát kênh và đưa kênh vừa hỏng vào `quarantine` trong `quarantine_steps` bước.
    """
    _ESCAPE_ROTATION = (4, 0, 6, 2, 5, 1, 7, 3, 8)
    _global_escape_cursor = 0

    def __init__(
        self,
        env: gym.Env,
        quarantine_steps: int = 12,
        hold_noise_thresh: float = -84.0,
        hold_pc_thresh: float = 0.66,
        hold_retry_thresh: float = 0.60,
        min_dwell_steps: int = 2,
    ):
        super().__init__(env)
        self.quarantine_steps = int(quarantine_steps)
        self.hold_noise_thresh = float(hold_noise_thresh)
        self.hold_pc_thresh = float(hold_pc_thresh)
        self.hold_retry_thresh = float(hold_retry_thresh)
        self.min_dwell_steps = int(min_dwell_steps)
        self.step_count = 0
        self.last_switch_step = -100
        self.current_ch = 0
        self.escape_cursor = StatefulChannelGuardWrapper._global_escape_cursor
        self.quarantine_until: Dict[int, int] = {}

    def reset(self, **kwargs):
        obs, info = self.env.reset(**kwargs)
        self.step_count = 0
        self.last_switch_step = -100
        self.current_ch = int(getattr(self.env.unwrapped, "current_ch", 0))
        self.quarantine_until = {}
        return obs, info

    def _quarantine_channel_and_cluster(self, ch_idx: int, noise_obs: float, bw_idx: int = 1) -> None:
        """Cách ly nhân quả kênh đang nhiễu và cụm gộp 40/80 MHz lân cận khi phát hiện nhiễu."""
        self.quarantine_until[ch_idx] = self.step_count + self.quarantine_steps
        if (noise_obs > -82.0 or bw_idx >= 1) and ch_idx < 8:
            partner_40 = ch_idx ^ 1
            self.quarantine_until[partner_40] = max(
                self.quarantine_until.get(partner_40, 0),
                self.step_count + max(6, self.quarantine_steps - 3),
            )
        if noise_obs > -72.0 and ch_idx < 8:
            block_range = range(0, 4) if ch_idx < 4 else range(4, 8)
            exp_block = self.step_count + max(8, self.quarantine_steps - 2)
            for c_b in block_range:
                self.quarantine_until[c_b] = max(self.quarantine_until.get(c_b, 0), exp_block)

    def _pick_rotated_escape_channel(
        self, allowed: list, proposed_ch: int, bw_req: int, sta_obs: float
    ) -> int:
        for _ in range(len(self._ESCAPE_ROTATION)):
            cand = self._ESCAPE_ROTATION[self.escape_cursor % len(self._ESCAPE_ROTATION)]
            self.escape_cursor = (self.escape_cursor + 1) % len(self._ESCAPE_ROTATION)
            StatefulChannelGuardWrapper._global_escape_cursor = self.escape_cursor
            if cand in allowed and cand != self.current_ch:
                if cand == 8 and bw_req > 0 and sta_obs <= 72.0 and (self.escape_cursor % 3 != 0):
                    wide_alt = [c for c in allowed if c < 8 and c != self.current_ch]
                    if wide_alt:
                        continue
                return cand
        allowed_other = [c for c in allowed if c != self.current_ch] or allowed
        return allowed_other[0]

    def filter_channel_action(self, obs_seen: np.ndarray, action: np.ndarray) -> Tuple[np.ndarray, bool]:
        act = np.array(action, dtype=np.int64, copy=True).reshape(-1)
        proposed_ch = int(np.clip(act[0], 0, len(NON_DFS_5G_CHANNELS) - 1))
        bw_cur = int(np.clip(act[1], 0, 2))
        noise_obs = float(obs_seen[1])
        sta_obs = float(obs_seen[2])
        retry_obs = float(obs_seen[3])
        pc_obs = float(obs_seen[4])
        ofdma_obs = float(obs_seen[6])

        # Xóa các kênh đã hết hạn cách ly
        for c in [k for k, exp in self.quarantine_until.items() if self.step_count >= exp]:
            self.quarantine_until.pop(c, None)

        dwell_elapsed = (self.step_count - self.last_switch_step) >= self.min_dwell_steps
        rf_jammed = (noise_obs > self.hold_noise_thresh)
        mac_collapsed = dwell_elapsed and (pc_obs >= self.hold_pc_thresh) and (retry_obs >= self.hold_retry_thresh)
        need_escape = rf_jammed or mac_collapsed
        stuck_on_ch165 = (self.current_ch == 8 and sta_obs <= 72.0 and ofdma_obs >= 0.35 and int(act[1]) > 0)
        held_channel = False

        if proposed_ch != self.current_ch:
            if (not need_escape) and (not stuck_on_ch165):
                act[0] = self.current_ch
                held_channel = True
            elif need_escape:
                self._quarantine_channel_and_cluster(self.current_ch, noise_obs, bw_cur)

        # Nếu đứng trên kênh đang nhiễu, cần rời CH165, hoặc đề xuất trúng kênh đang bị cách ly -> chọn kênh thoát xoay vòng
        if need_escape or stuck_on_ch165:
            if need_escape and (self.current_ch not in self.quarantine_until):
                self._quarantine_channel_and_cluster(self.current_ch, noise_obs, bw_cur)
            allowed = [c for c in range(len(NON_DFS_5G_CHANNELS)) if c not in self.quarantine_until]
            if not allowed:
                oldest_ch = min(self.quarantine_until.items(), key=lambda kv: kv[1])[0]
                self.quarantine_until.pop(oldest_ch, None)
                allowed = [oldest_ch]
            act[0] = self._pick_rotated_escape_channel(allowed, proposed_ch, int(act[1]), sta_obs)
        elif (not need_escape) and (act[0] in self.quarantine_until) and (act[0] != self.current_ch):
            # Bảo vệ không nhảy từ kênh sạch sang kênh vẫn đang trong thời hạn cách ly
            act[0] = self.current_ch
            held_channel = True

        if act[0] == 8 and int(act[1]) > 0:
            act[1] = 0
        return act, held_channel

    def step(self, action: np.ndarray):
        obs_seen = getattr(self.env.unwrapped, "current_obs", None)
        if obs_seen is not None:
            eff_act, held = self.filter_channel_action(obs_seen, action)
        else:
            eff_act, held = np.array(action, dtype=np.int64, copy=True), False

        prev_ch = self.current_ch
        obs, reward, terminated, truncated, info = self.env.step(eff_act)
        self.step_count += 1
        self.current_ch = int(eff_act[0])
        if self.current_ch != prev_ch:
            self.last_switch_step = self.step_count
        if isinstance(info, dict):
            info["guard_held_channel"] = held
            info["guard_effective_ch_idx"] = self.current_ch
            info["guard_quarantined_chs"] = sorted(list(self.quarantine_until.keys()))
        return obs, reward, terminated, truncated, info

print("[+] Đã định nghĩa HardwareAwareWrapper & StatefulChannelGuardWrapper (đồng bộ 1-1 với src/envs/digital_twin_env.py).")

[+] Đã định nghĩa HardwareAwareWrapper & StatefulChannelGuardWrapper (đồng bộ 1-1 với src/envs/digital_twin_env.py).


## BƯỚC 3: XÂY DỰNG LÕI VẬT LÝ CONTINUOUS-MARKOV 4 TẦNG (`UltimateDigitalTwinEnv`) & HUẤN LUYỆN PPO + `VecFrameStack(4)`
Bước này thay thế hoàn toàn dấu vết Contextual Bandits trước đây bằng **Quá trình Markov Vật lý Liên tục (Continuous-Physics POMDP)**:
1. **Xóa bỏ Đồng hồ pha 25 bước & Kéo neo ngoại sinh 70%:** Thời lượng mỗi đợt nhiễu/tải biến thiên ngẫu nhiên ($15\text{–}55$ bước), phổ thực 9 kênh `true_ch_noise[0..8]` trôi theo Ornstein-Uhlenbeck hoàn toàn đối xứng (không ưu ái cứng `CH36` hay `CH149`), còn các biến MAC (`retry_rate`, `p_c`, `airtime_util`, `ofdma_ratio`) tiến hóa **$75\%$ trực tiếp từ `metrics` nhân quả** của hành động.
2. **Cảm biến `noise_on_chosen_ch` (`obs[1]`):** Cập nhật trực tiếp theo `true_ch_noise[action[0]]` khớp `iw survey dump [in use]` trên router thật, kết hợp `VecFrameStack(n_stack=4)` và `StatefulChannelGuardWrapper`.
3. **Sửa 3 điểm vật lý trọng yếu:**
   * **`mcs_ceiling` theo chuẩn Minstrel-HT:** Mô hình hóa đúng `operating_mcs = min(mcs_ceiling, minstrel_natural_mcs)` với điểm ngọt `amc_stability_bonus` tại biên `SINR Margin ~ 6.5 dB` ($\text{PER} \le 1.5\%$), loại bỏ nghịch lý thưởng cho $\text{PER } 10	ext{–}16\%$ và giúp baseline `Static Legacy` phản ánh đúng router Minstrel thực tế.
   * **SU-Beamforming CSI Aging (`2/2` bins):** Kích hoạt `BF=OFF` thực sự trong các ca bão Roaming/mật độ STA cao & trượt pha kênh (`is_csi_aging_regime`).
   * **Trả về `mean_latency_ms` & Gamma P99 Wilson–Hilferty (`gamma_p99_from_moments`):** Khớp chính xác phân phối Gamma lệch phải dùng ở Bước 4.5.

In [3]:
class NormalizedWifiFeaturesExtractor(BaseFeaturesExtractor):
    """
    Bộ trích xuất đặc trưng chuẩn hóa Min-Max về [-1.0, 1.0] kết hợp:
      1) Khung trạng thái hiện tại `obs_t` (8D cuối) mở rộng qua lưới Radial Basis Functions (RBF 8 tâm đều
         trên [-1, 1] theo Sutton & Barto Ch. 9.5) giúp mạng phân giải sắc nét các nấc lượng tử hóa bậc thang.
      2) Nhánh động học đa khung 32D (`temporal_net`) khai thác trọn vẹn cả 4 khung `t-3..t`:
         - Vận tốc tức thời `vel_1 = f_t - f_{t-1}` (8D)
         - Gia tốc biến thiên `acc_2 = f_t - 2*f_{t-1} + f_{t-2}` (8D)
         - Độ trôi toàn cửa sổ `drift_3 = f_t - f_{t-3}` (8D)
         - Trung bình trượt nền 4 bước `mean_4 = 0.25 * (f_{t-3} + f_{t-2} + f_{t-1} + f_t)` (8D)
    """
    def __init__(self, observation_space: spaces.Box):
        super().__init__(observation_space, features_dim=64)
        obs_dim = int(observation_space.shape[0])
        self.obs_dim = obs_dim
        self.register_buffer("low8", torch.tensor(OBS_LOW.copy(), dtype=torch.float32))
        self.register_buffer("high8", torch.tensor(OBS_HIGH.copy(), dtype=torch.float32))
        self.register_buffer("rbf_centers", torch.linspace(-1.0, 1.0, 8, dtype=torch.float32).view(1, 1, 8))
        # Đầu vào cur_net: 8 (linear) + 8*8 (uniform RBF) = 72 chiều
        self.cur_skip = nn.Linear(72, 64)
        self.cur_net = nn.Sequential(
            nn.Linear(72, 128),
            nn.SiLU(),
            nn.Linear(128, 128),
            nn.SiLU(),
            nn.Linear(128, 64),
        )
        self.temporal_net = nn.Sequential(
            nn.Linear(32, 64),
            nn.SiLU(),
            nn.Linear(64, 64),
        )

    def _norm8(self, x8: torch.Tensor) -> torch.Tensor:
        n = 2.0 * (x8 - self.low8) / (self.high8 - self.low8 + 1e-6) - 1.0
        return torch.clamp(n, -1.0, 1.0)

    def forward(self, observations: torch.Tensor) -> torch.Tensor:
        cur_raw = observations[:, -8:]
        cur_norm = self._norm8(cur_raw)
        diff = cur_norm.unsqueeze(-1) - self.rbf_centers
        rbf_feat = torch.exp(-10.0 * (diff * diff)).reshape(cur_norm.shape[0], 64)
        cur_full = torch.cat([cur_norm, rbf_feat], dim=-1)
        out = self.cur_skip(cur_full) + self.cur_net(cur_full)
        if observations.shape[1] >= 32:
            f0 = self._norm8(observations[:, -32:-24])
            f1 = self._norm8(observations[:, -24:-16])
            f2 = self._norm8(observations[:, -16:-8])
            f3 = cur_norm
            vel_1 = f3 - f2
            acc_2 = f3 - 2.0 * f2 + f1
            drift_3 = f3 - f0
            mean_4 = 0.25 * (f0 + f1 + f2 + f3)
            kin_32 = torch.cat([vel_1, acc_2, drift_3, mean_4], dim=-1)
            out = out + 0.25 * self.temporal_net(kin_32)
        elif observations.shape[1] >= 16:
            f_prev = self._norm8(observations[:, -16:-8])
            vel_1 = cur_norm - f_prev
            zeros8 = torch.zeros_like(cur_norm)
            mean_4 = 0.25 * (3.0 * f_prev + cur_norm)
            kin_32 = torch.cat([vel_1, zeros8, vel_1, mean_4], dim=-1)
            out = out + 0.25 * self.temporal_net(kin_32)
        return out


CrossLayerNormExtractor = NormalizedWifiFeaturesExtractor


class UltimateDigitalTwinEnv(gym.Env):
    """
    Bản sao số Vật lý Nhân quả Liên tục Chuẩn IEEE 802.11ax (Pure Continuous-Markov Physics MDP):
    - ZERO `ideal_action`, ZERO `weapon_shaping` L1, ZERO đồng hồ 25 bước, ZERO `ambient_anchor` kéo 70%.
    - Trường phổ 9 kênh thực `true_ch_noise[0..8]` biến thiên liên tục theo Chuỗi Markov + Ornstein-Uhlenbeck.
    - `obs[1]` là `noise_on_chosen_ch` (`true_ch_noise[current_ch]`), hoàn toàn nhân quả theo hành động chọn kênh.
    - `mcs_ceiling` vận hành như trần giới hạn thực sự trên bộ điều khiển Minstrel-HT nội tại.
    - Hỗ trợ `obs_mask_phy=True` để huấn luyện công bằng mô hình đối chứng `PPO MAC-Only`.
    """
    metadata = {"render_modes": []}

    MCS_BASE_RATE_20MHZ = np.array(
        [17.2, 34.4, 51.6, 68.8, 103.2, 137.6, 154.9, 172.1, 206.5, 229.4, 258.1, 286.8],
        dtype=np.float32,
    )
    MCS_REQ_SINR_DB = np.array([4.5 + 2.25 * m for m in range(12)], dtype=np.float32)
    # Suy hao truyền dẫn tự nhiên theo tần số Friis chuẩn: 20 * log10(f_MHz / 5180)
    FREQ_PATH_LOSS_DB = np.array(
        [0.0, 0.03, 0.07, 0.10, 0.90, 0.93, 0.96, 0.99, 1.02],
        dtype=np.float32,
    )
    _MCS_BASE_RATE_TUPLE = (17.2, 34.4, 51.6, 68.8, 103.2, 137.6, 154.9, 172.1, 206.5, 229.4, 258.1, 286.8)
    _MCS_REQ_SINR_TUPLE = tuple(4.5 + 2.25 * m for m in range(12))
    _EXP_NEG_095_REQ_TUPLE = tuple(math.exp(-0.95 * r) for r in _MCS_REQ_SINR_TUPLE)
    _FREQ_LOSS_TUPLE = (0.0, 0.03, 0.07, 0.10, 0.90, 0.93, 0.96, 0.99, 1.02)
    _CH_INDEX_MAP = {ch: idx for idx, ch in enumerate(NON_DFS_5G_CHANNELS)}
    _ORACLE_VALID_PAIRS = tuple((ch, bw) for ch in range(9) for bw in range(3) if not (ch == 8 and bw > 0))
    _ORACLE_P_CH = np.array([p[0] for p in _ORACLE_VALID_PAIRS], dtype=np.int64)
    _ORACLE_P_BW = np.array([p[1] for p in _ORACLE_VALID_PAIRS], dtype=np.int64)
    _ORACLE_P_TONAL = np.array([1.00 if bw == 0 else (2.04 if bw == 1 else 4.02) for _, bw in _ORACLE_VALID_PAIRS], dtype=np.float64)
    _ORACLE_P_FLOSS = (lambda f, vp: np.array([f[c] for c, _ in vp], dtype=np.float64))(_FREQ_LOSS_TUPLE, _ORACLE_VALID_PAIRS)
    _ORACLE_G_P, _ORACLE_G_BF, _ORACLE_G_OB, _ORACLE_G_TX = [
        a.ravel() for a in np.meshgrid(np.arange(25), np.arange(2), np.arange(5), np.arange(7), indexing="ij")
    ]
    _ORACLE_ROW_IDX = np.arange(1750, dtype=np.int64)
    _ORACLE_MCS_REQ = np.array(_MCS_REQ_SINR_TUPLE, dtype=np.float64)
    _ORACLE_MCS_RATE = np.array(_MCS_BASE_RATE_TUPLE, dtype=np.float64)

    @staticmethod
    def _clip_scalar(val: float, lo: float, hi: float) -> float:
        return lo if val < lo else (hi if val > hi else val)

    @staticmethod
    def _mcs_lookup(sinr_budget: float) -> int:
        if sinr_budget < 4.5:
            return 0
        idx = int((sinr_budget - 4.5 + 1e-9) / 2.25)
        return 11 if idx > 11 else idx

    def __init__(
        self,
        hardware_type: str = "WIFI_6",
        aqm_enabled: bool = True,
        max_steps: int = 300,
        seed: int = GLOBAL_SEED,
        obs_mask_phy: bool = False,
    ):
        super().__init__()
        self.hardware_type = hardware_type.upper()
        self.aqm_enabled = bool(aqm_enabled)
        self.max_steps = int(max_steps)
        self.obs_mask_phy = bool(obs_mask_phy)
        self.base_seed = int(seed)
        self.rng = np.random.default_rng(seed)

        self.action_space = spaces.MultiDiscrete(ACTION_NVEC)
        self.observation_space = spaces.Box(low=OBS_LOW.copy(), high=OBS_HIGH.copy(), dtype=np.float32)

        self.current_step = 0
        self.current_ch = 0
        self.previous_action: Optional[np.ndarray] = None
        self.scenario_mode = 0
        self.queue_backlog_mb = 0.0

        # Trạng thái vật lý ẩn liên tục của mô phỏng (không rò rỉ ra observation)
        self.true_ch_noise = np.full(9, -95.2, dtype=np.float32)
        self.target_ch_noise = np.full(9, -95.2, dtype=np.float32)
        self.target_rssi = -56.0
        self.target_sta = 18.0
        self.target_ofdma_base = 0.78
        self.target_demand_scale = 1.0
        self.current_is_break = 0.0
        self.regime_remaining_steps = 25

        self.true_current_obs: Optional[np.ndarray] = None
        self.current_obs: Optional[np.ndarray] = None

    @staticmethod
    def _dbm_sum(dbm_a: float, dbm_b: float) -> float:
        mw = (10.0 ** (dbm_a / 10.0)) + (10.0 ** (dbm_b / 10.0))
        return 10.0 * math.log10(mw if mw > 1e-12 else 1e-12)

    @staticmethod
    def _bessel_j0(x: float) -> float:
        """Xấp xỉ đa thức Bessel loại 1 bậc 0 J_0(x) (Abramowitz & Stegun 9.4.1 / 9.4.3, sai số < 1e-6)."""
        ax = abs(float(x))
        if ax < 3.0:
            y = (ax / 3.0) ** 2
            return (
                1.0
                - 2.2499997 * y
                + 1.2656208 * (y ** 2)
                - 0.3163866 * (y ** 3)
                + 0.0444479 * (y ** 4)
                - 0.0039444 * (y ** 5)
                + 0.0002100 * (y ** 6)
            )
        z = 3.0 / ax
        f0 = (
            0.79788456
            - 0.00000077 * z
            - 0.00552740 * (z ** 2)
            - 0.00009512 * (z ** 3)
            + 0.00137237 * (z ** 4)
            - 0.00072805 * (z ** 5)
            + 0.00014476 * (z ** 6)
        )
        theta0 = (
            ax
            - 0.78539816
            - 0.04166397 * z
            - 0.00003954 * (z ** 2)
            + 0.00262573 * (z ** 3)
            - 0.00054125 * (z ** 4)
            - 0.00029333 * (z ** 5)
            + 0.00013558 * (z ** 6)
        )
        return math.sqrt(1.0 / ax) * f0 * math.cos(theta0)

    @classmethod
    def _compute_jakes_bf_physics(
        cls,
        ch_idx: int,
        sta: float,
        retry_obs: float,
        airtime_obs: float,
        is_break: float,
    ) -> Tuple[float, float, float, bool]:
        """Mô hình lão hóa kênh CSI liên tục Clarke-Jakes theo tần số sóng mang & chi phí NDP Sounding."""
        ch_safe = 0 if ch_idx < 0 else (8 if ch_idx > 8 else int(ch_idx))
        lam = 299.792458 / float(CHANNEL_FREQ_MHZ[ch_safe])
        if int(round(is_break)) == 1:
            v_mps = 1.05 + 1.15 * cls._clip_scalar(sta / 150.0, 0.0, 1.30) + 0.55 * retry_obs
        else:
            v_mps = 0.12 + 0.42 * max(0.0, (sta - 75.0) / 110.0) + (
                0.95 * max(0.0, retry_obs - 0.26) if sta > 60.0 else 0.0
            )
        f_d = v_mps / lam
        dt_csi = 0.012 + 0.00028 * sta + 0.016 * airtime_obs
        x_dop = min(2.0 * math.pi * f_d * dt_csi, 3.8317)
        rho_csi = cls._bessel_j0(x_dop)
        bf_on_gain_db = 3.1 * rho_csi - 1.6 * max(0.0, 0.40 - rho_csi)
        bf_on_ovh = min(0.092, 0.006 + 0.00050 * sta + (0.012 if int(round(is_break)) == 1 else 0.0))
        is_csi_aging = (bf_on_gain_db - 26.0 * bf_on_ovh) < 0.0
        return float(rho_csi), float(bf_on_gain_db), float(bf_on_ovh), bool(is_csi_aging)

    @classmethod
    def _evaluate_minstrel_ht_amc(
        cls,
        eff_sinr_db: float,
        mcs_ceiling_idx: int,
        ewma_lag_db: float,
    ) -> Tuple[int, float, float]:
        """
        Mô hình hóa thuật toán Linux Minstrel-HT:
          - Tính bảng thông lượng kỳ vọng theo EWMA (prob kẹp [0.10, 0.90]), chọn r0, r1 và r_prob.
          - 85% khung thường chạy chuỗi 4-tầng Multi-Rate Retry (MRR [r0 -> r1 -> r_prob -> 0]).
          - 15% khung Lookaround thăm dò hướng lên trong dải (r0 + 1 .. mcs_ceiling_idx).
        """
        per_inst_list = []
        best_tp_est = -1.0
        r0 = 0
        r1 = 0
        r_prob = 0
        best_prob_score = -1.0

        e_inst_base = math.exp(cls._clip_scalar(0.95 * (eff_sinr_db - 1.8), -35.0, 65.0))
        e_ewma_base = math.exp(cls._clip_scalar(0.95 * (eff_sinr_db + ewma_lag_db - 1.8), -35.0, 65.0))
        exp_req_tuple = cls._EXP_NEG_095_REQ_TUPLE
        rate_tuple = cls._MCS_BASE_RATE_TUPLE

        for m in range(mcs_ceiling_idx + 1):
            exp_req_m = exp_req_tuple[m]
            rate_m = rate_tuple[m]
            per_inst = 1.0 / (1.0 + e_inst_base * exp_req_m)
            per_ewma = 1.0 / (1.0 + e_ewma_base * exp_req_m)
            per_inst_list.append(per_inst)

            p_succ_ewma = 1.0 - per_ewma
            tp_ewma = (min(0.90, p_succ_ewma) if p_succ_ewma >= 0.10 else 0.0) * rate_m
            if tp_ewma > best_tp_est:
                r1 = r0
                best_tp_est = tp_ewma
                r0 = m

            prob_score = p_succ_ewma * min(rate_m, 68.8)
            if prob_score > best_prob_score:
                best_prob_score = prob_score
                r_prob = m

        p0 = per_inst_list[r0]
        p1 = per_inst_list[r1]
        normal_rate = (
            (1.0 - p0) * cls._MCS_BASE_RATE_TUPLE[r0]
            + p0 * (1.0 - p1) * cls._MCS_BASE_RATE_TUPLE[r1]
            + p0 * p1 * cls._MCS_BASE_RATE_TUPLE[r_prob] * 0.65
        )
        normal_per = p0 * 0.58 + p0 * p1 * 0.42

        if mcs_ceiling_idx > r0:
            n_pr = float(mcs_ceiling_idx - r0)
            pr_rate_sum = 0.0
            pr_per_sum = 0.0
            for m in range(r0 + 1, mcs_ceiling_idx + 1):
                pm = per_inst_list[m]
                pr_rate_sum += cls._MCS_BASE_RATE_TUPLE[m] * (1.0 - pm)
                pr_per_sum += pm
            eff_mcs_rate = 0.36 * (0.85 * normal_rate + 0.15 * (pr_rate_sum / n_pr))
            per_phy = cls._clip_scalar(0.85 * normal_per + 0.15 * (pr_per_sum / n_pr), 0.002, 0.96)
        else:
            eff_mcs_rate = 0.36 * normal_rate
            per_phy = cls._clip_scalar(normal_per, 0.002, 0.96)

        return r0, float(eff_mcs_rate), float(per_phy)

    @staticmethod
    def _compute_kpi_reward(
        goodput_mbps: float,
        offered_load_mbps: float,
        service_capacity_mbps: float,
        p_c: float,
        per_phy: float,
        p99_latency_ms: float,
        packet_loss_pct: float,
        tx_power_dbm: float,
        is_invalid_bonding: bool = False,
        tx_slew_pen: float = 0.0,
    ) -> float:
        """Hàm mục tiêu 100% Pure Outcome KPI (không chứa heuristic target action shaping)."""
        packet_loss_ratio = packet_loss_pct / 100.0
        gp_fulfill = goodput_mbps / max(offered_load_mbps, 1.0)
        cap_headroom = 0.52 * math.log1p(service_capacity_mbps / max(offered_load_mbps, 25.0))
        energy_pen = 0.032 * (tx_power_dbm - 14.0)
        penalty_collision = 3.40 * (p_c * p_c) + 2.10 * per_phy
        penalty_latency = 0.042 * min(p99_latency_ms, 60.0) + 1.25 * math.tanh(max(0.0, p99_latency_ms - 15.0) / 15.0)
        penalty_loss = 3.20 * packet_loss_ratio + (0.60 if packet_loss_pct >= 1.0 else 0.0)
        sla_bonus = 0.60 if (p99_latency_ms <= 15.0 and packet_loss_pct < 1.0) else -0.40
        invalid_bond_pen = -0.80 if is_invalid_bonding else 0.0
        return float(
            2.60 * gp_fulfill
            + 0.65 * (goodput_mbps / 100.0)
            + cap_headroom
            - penalty_collision
            - penalty_latency
            - penalty_loss
            - energy_pen
            + sla_bonus
            + invalid_bond_pen
            - tx_slew_pen
        )

    @classmethod
    def _construct_9ch_spectrum_profile(
        cls,
        obs: np.ndarray,
        rng: Optional[np.random.Generator] = None,
    ) -> np.ndarray:
        """
        Hàm fallback tái tạo hồ sơ phổ 9 kênh khi gọi đánh giá tĩnh trên một vector `obs` đơn lẻ
        (không truyền `sub_noise_dbm` thực từ simulator). Không thiên vị cứng CH36/CH149.
        """
        noise_obs = float(obs[1])
        sta = float(obs[2])
        airtime_obs = float(obs[5])
        base_thermal = np.array([-95.6, -95.4, -95.5, -95.3, -95.4, -95.2, -95.3, -95.1, -95.0], dtype=np.float32)

        if noise_obs <= -85.0:
            sub_noise = base_thermal.copy()
            if sta > 120.0 and airtime_obs > 0.85:
                sub_noise[0] += 1.5
            return sub_noise

        if rng is not None:
            # Nếu được cấp RNG (như trong bộ sinh kịch bản snapshot), đặt nhiễu trên một tập kênh tự nhiên bất kỳ
            sub_noise = base_thermal.copy()
            jam_idx = int(rng.integers(0, 9))
            sub_noise[jam_idx] = noise_obs
            return sub_noise

        # Fallback xác định khi gọi What-If tĩnh với một `obs` đơn lẻ
        sub_noise = base_thermal.copy()
        if noise_obs <= -74.0:
            sub_noise[0] = noise_obs
            sub_noise[1] = max(noise_obs - 2.0, -82.0)
        else:
            sub_noise[0:4] = np.clip(noise_obs - np.array([0.0, 0.8, 1.4, 2.0]), -76.0, -52.0)
        return sub_noise

    @classmethod
    def _original_evaluate_network_physics(
        cls,
        obs: np.ndarray,
        action: np.ndarray,
        prev_action: np.ndarray = None,
        aqm_enabled: bool = True,
        sub_noise_dbm: Optional[np.ndarray] = None,
        base_ofdma_capacity: Optional[float] = None,
        demand_scale: float = 1.0,
        queue_backlog_mb: float = 0.0,
    ) -> Dict[str, float]:
        """
        Lõi Tính toán Vật lý Viễn thông 4 Tầng Chuẩn IEEE 802.11ax (v3 Jury-Proof):
        1. Spectrum Bonding (tích hợp mW tuyến tính đúng 10log10(B)) & Bất đối xứng Primary/Secondary CCA.
        2. Link Budget thuần vật lý (LNA Overdrive), Clarke-Jakes Bessel J0 CSI Aging, OBSS-PD Spatial Reuse vs Hidden-Node Leak.
        3. Linux Minstrel-HT 4-tầng MRR + 15% Lookaround Probing & Va chạm CSMA/CA.
        4. Goodput IETF RFC 2647, Hàng đợi M/G/1 Pollaczek-Khinchine + Tồn dư đa bước Lindley & Wilson-Hilferty P99.
        5. Hàm thưởng 100% Pure Outcome KPI (không chứa target action shaping).
        """
        try:
            rssi = float(obs[0]); noise_obs = float(obs[1]); sta = float(obs[2]); retry_obs = float(obs[3])
            pc_obs = float(obs[4]); airtime_obs = float(obs[5]); ofdma_obs = float(obs[6]); is_break = float(obs[7])
            ch_idx = int(action[0]); bw_idx = int(action[1]); tx_idx = int(action[2]); mcs_ceiling_idx = int(action[3])
            beacon_idx = int(action[4]); obss_idx = int(action[5]); bf_idx = int(action[6])

            ch_idx = 0 if ch_idx < 0 else (8 if ch_idx > 8 else ch_idx)
            bw_idx = 0 if bw_idx < 0 else (2 if bw_idx > 2 else bw_idx)
            tx_idx = 0 if tx_idx < 0 else (6 if tx_idx > 6 else tx_idx)
            mcs_ceiling_idx = 0 if mcs_ceiling_idx < 0 else (11 if mcs_ceiling_idx > 11 else mcs_ceiling_idx)
            beacon_idx = 0 if beacon_idx < 0 else (1 if beacon_idx > 1 else beacon_idx)
            obss_idx = 0 if obss_idx < 0 else (4 if obss_idx > 4 else obss_idx)
            bf_idx = 0 if bf_idx < 0 else (1 if bf_idx > 1 else bf_idx)

            # 1. TẦNG VẬT LÝ PHỔ TẦN & GỘP KÊNH (TÍCH HỢP TUYẾN TÍNH mW CHUẨN 10log10(B))
            spectrum_noise = cls._construct_9ch_spectrum_profile(obs) if sub_noise_dbm is None else sub_noise_dbm

            chosen_sub_noise_dbm = float(spectrum_noise[ch_idx])
            primary_ch, actual_htmode, occupied_subs, is_invalid_bonding = resolve_valid_5g_channel_and_bw(ch_idx, bw_idx)
            ch_map = cls._CH_INDEX_MAP
            actual_bw_idx = 0 if actual_htmode == "HE20" else (1 if actual_htmode == "HE40" else 2)

            if actual_htmode == "HE20":
                occ_max_noise_dbm = chosen_sub_noise_dbm
                sec_max_noise_dbm = -100.0
                eff_noise_dbm = chosen_sub_noise_dbm
                bw_tonal_factor = 1.00
            elif actual_htmode == "HE40":
                i0, i1 = ch_map[occupied_subs[0]], ch_map[occupied_subs[1]]
                n0, n1 = float(spectrum_noise[i0]), float(spectrum_noise[i1])
                occ_max_noise_dbm = n0 if n0 >= n1 else n1
                sec_max_noise_dbm = n1 if ch_idx == i0 else n0
                eff_noise_dbm = cls._dbm_sum(n0, n1)
                bw_tonal_factor = 2.04
            else:  # HE80 (4 kênh 20 MHz liên tiếp)
                i0, i1, i2, i3 = ch_map[occupied_subs[0]], ch_map[occupied_subs[1]], ch_map[occupied_subs[2]], ch_map[occupied_subs[3]]
                n0, n1, n2, n3 = (
                    float(spectrum_noise[i0]),
                    float(spectrum_noise[i1]),
                    float(spectrum_noise[i2]),
                    float(spectrum_noise[i3]),
                )
                occ_max_noise_dbm = max(n0, n1, n2, n3)
                sec_max_noise_dbm = max(n_val for idx_sub, n_val in ((i0, n0), (i1, n1), (i2, n2), (i3, n3)) if idx_sub != ch_idx)
                n_pair1 = cls._dbm_sum(n0, n1)
                n_pair2 = cls._dbm_sum(n2, n3)
                eff_noise_dbm = cls._dbm_sum(n_pair1, n_pair2)
                bw_tonal_factor = 4.02

            # 2. NGÂN SÁCH ĐƯỜNG TRUYỀN THUẦN VẬT LÝ, LNA OVERDRIVE & BEAMFORMING JAKES J0
            clip = cls._clip_scalar
            tx_power_dbm = 14.0 + float(tx_idx)
            freq_loss_db = cls._FREQ_LOSS_TUPLE[ch_idx]

            edge_spread_db = 3.5 + 4.8 * min(1.4, sta / 85.0)
            rx_edge_dbm = rssi - edge_spread_db + 1.00 * (tx_power_dbm - 17.0) - 0.25 * freq_loss_db
            rx_close_dbm = rssi + 1.00 * (tx_power_dbm - 17.0)
            lna_overdrive_db = max(0.0, rx_close_dbm - (-43.5)) * 1.65
            rssi_compensated = rx_edge_dbm - lna_overdrive_db

            is_break_flag = (int(round(is_break)) == 1)
            rho_csi, bf_on_gain_db, bf_on_ovh, is_csi_aging_regime = cls._compute_jakes_bf_physics(
                ch_idx, sta, retry_obs, airtime_obs, is_break
            )
            if bf_idx == 1:
                bf_gain_db = bf_on_gain_db
                csi_sounding_overhead = bf_on_ovh
            else:
                bf_gain_db = 0.0
                csi_sounding_overhead = 0.0

            escaped_rf_jam = (noise_obs > -84.0) and (occ_max_noise_dbm <= -88.0)
            eff_ofdma_exog = ofdma_obs if base_ofdma_capacity is None else float(base_ofdma_capacity)
            alpha_leg = clip((0.58 - eff_ofdma_exog) / 0.50, 0.0, 1.0)
            obs_ofdma_clean = (
                max(ofdma_obs, 0.52)
                if (base_ofdma_capacity is None and escaped_rf_jam)
                else (0.5 * ofdma_obs + 0.5 * eff_ofdma_exog)
            )
            alpha_leg_sr = clip((0.58 - obs_ofdma_clean) / 0.50, 0.0, 1.0)

            # Vật lý OBSS-PD: Nâng ngưỡng giảm trì hoãn CCA ở HE BSS Color, nhưng rò nhiễu & nút ẩn ở Legacy không màu
            sr_norm = float(obss_idx) / 4.0
            cca_deferral_pen = 0.18 * ((1.0 - sr_norm) ** 1.4) * ((1.0 - alpha_leg_sr) ** 1.3)
            hidden_node_coll = 0.26 * (sr_norm ** 1.5) * (0.15 + 1.35 * alpha_leg_sr)
            uncolored_leak_db = (sr_norm ** 1.6) * (1.0 + 8.5 * alpha_leg_sr)

            fading_margin_db = 1.0 + 1.8 * alpha_leg + (2.5 if occ_max_noise_dbm > -86.0 else 0.0)
            eff_sinr_db = rx_edge_dbm + bf_gain_db - eff_noise_dbm - fading_margin_db - lna_overdrive_db - uncolored_leak_db

            # 3. LINUX MINSTREL-HT 4-STAGE MRR + 15% LOOKAROUND PROBING & CSMA/CA
            ewma_lag_db = (
                2.0
                + (3.2 if occ_max_noise_dbm > -85.0 else 0.0)
                + 2.2 * min(1.0, sta / 100.0)
                + (1.5 if is_break_flag else 0.0)
            )
            operating_mcs_idx, effective_mcs_rate, per_phy = cls._evaluate_minstrel_ht_amc(
                eff_sinr_db, mcs_ceiling_idx, ewma_lag_db
            )

            primary_cca_stall = 0.14 * clip((chosen_sub_noise_dbm - (-85.0)) / 24.0, 0.0, 1.0)
            secondary_pifs_block = 0.06 * clip((sec_max_noise_dbm - (-82.0)) / 24.0, 0.0, 1.0)
            channel_interf_ratio = clip(
                (eff_noise_dbm - (-95.0)) / 34.0 + primary_cca_stall + secondary_pifs_block,
                0.01,
                0.94,
            )
            bw_cont_mult = 1.0 + float(actual_bw_idx) * (
                0.55 * (sta / 50.0) + 1.15 * alpha_leg + (0.75 if occ_max_noise_dbm > -86.0 else 0.0)
            )
            ofdma_efficiency = clip(
                0.62 * eff_ofdma_exog + 0.22 * (1.0 - 0.22 * float(actual_bw_idx)) - 0.25 * channel_interf_ratio,
                0.08,
                0.95,
            )
            p_c = clip(
                0.38 * channel_interf_ratio
                + 0.26 * (sta / 110.0) * bw_cont_mult * (1.0 - 0.42 * ofdma_efficiency)
                + 0.38 * per_phy
                + cca_deferral_pen
                + hidden_node_coll,
                0.02,
                0.96,
            )
            retry_rate = clip(0.55 * p_c + 0.45 * per_phy, 0.01, 0.96)

            # 4. ĐỒNG CƯ LEGACY HEUSSE + POLLACZEK-KHINCHINE M/G/1 + LINDLEY BACKLOG
            beacon_overhead = 0.055 if beacon_idx == 0 else 0.012
            roaming_handover_factor = (1.00 if beacon_idx == 0 else 0.74) if is_break_flag else 1.00
            r_he_raw = max(4.0, effective_mcs_rate * bw_tonal_factor)
            r_leg_cap = min(r_he_raw, 26.0 + 2.0 * float(operating_mcs_idx))
            r_harmonic = 1.0 / ((1.0 - alpha_leg) / r_he_raw + alpha_leg / max(r_leg_cap, 1.0))
            eta_heusse = (0.72 + 0.28 * (r_harmonic / r_he_raw)) if aqm_enabled else (r_harmonic / r_he_raw)
            legacy_prot_overhead = 0.085 * (alpha_leg * alpha_leg) * (1.0 + 0.45 * float(actual_bw_idx))
            total_overhead = min(
                0.45,
                beacon_overhead + csi_sounding_overhead + legacy_prot_overhead + 0.60 * cca_deferral_pen + (0.18 if is_invalid_bonding else 0.0),
            )
            service_capacity_mbps = max(
                6.0,
                r_he_raw * eta_heusse * roaming_handover_factor * (0.88 + 0.24 * ofdma_efficiency) * (1.0 - total_overhead) * (1.0 - p_c) * (1.0 - 0.65 * per_phy),
            )

            # Lưu lượng yêu cầu ngoại sinh BẤT BIẾN TUYỆT ĐỐI theo Action và biến nội sinh MAC
            exog_link_scale = clip((rssi - (-88.0)) / 38.0, 0.32, 1.10)
            if sta <= 30.0 and alpha_leg < 0.40 and rssi >= -60.0:
                raw_demand = 148.0 + 1.25 * sta
            elif sta <= 32.0 and alpha_leg < 0.40:
                raw_demand = 86.0 + 1.05 * sta
            elif sta <= 78.0 and alpha_leg < 0.55:
                raw_demand = 52.0 + 0.35 * sta
            else:
                raw_demand = 18.0 + 0.05 * sta
            offered_load_mbps = raw_demand * exog_link_scale * float(demand_scale)

            cur_backlog = max(0.0, float(queue_backlog_mb))
            airtime_next = clip((offered_load_mbps + cur_backlog * 2.0) / max(service_capacity_mbps, 8.0), 0.08, 0.99)

            es_ms = 12.0 / service_capacity_mbps
            cs2 = max(0.25, 0.55 + 2.8 * (p_c * p_c) + 1.6 * retry_rate - 0.30 * ofdma_efficiency)
            rho = offered_load_mbps / service_capacity_mbps
            overload_ratio = max(0.0, (offered_load_mbps - service_capacity_mbps) / max(offered_load_mbps, 1.0))
            rho_eff = min(rho, 0.92 if aqm_enabled else 0.985)
            w_pk_ms = (rho_eff * es_ms * (1.0 + cs2)) / (2.0 * max(0.015, 1.0 - rho_eff))
            if aqm_enabled:
                w_pk_ms = min(w_pk_ms, 3.8 + 2.4 * max(0.0, rho - 0.85))
                b_max_mb = 0.65
                congestion_drop = 0.045 * overload_ratio
            else:
                w_pk_ms = w_pk_ms + 28.0 * max(0.0, rho - 0.90)
                b_max_mb = 18.0
                congestion_drop = 0.48 * overload_ratio

            w_backlog_ms = (cur_backlog / service_capacity_mbps) * 1000.0
            backlog_next_mb = clip(
                cur_backlog + (offered_load_mbps * (1.0 - congestion_drop) - service_capacity_mbps) * 0.22,
                0.0,
                b_max_mb,
            )

            retrans_delay_ms = 1.10 * (retry_rate / max(1.0 - retry_rate, 0.16))
            handover_delay_ms = 3.2 if (is_break_flag and beacon_idx == 1) else 0.0
            mean_latency_ms = 1.50 + w_pk_ms + w_backlog_ms + retrans_delay_ms + handover_delay_ms
            jitter_ms = (
                0.12
                + 1.25 * (p_c * p_c)
                + (1.50 * overload_ratio if aqm_enabled else 8.00 * overload_ratio)
                + 0.42 * (alpha_leg * alpha_leg)
            )
            p99_latency_ms = gamma_p99_from_moments(mean_latency_ms, jitter_ms, d_min=1.50)

            mac_exhaust_loss = (p_c * per_phy) ** 3
            packet_loss_ratio = clip(mac_exhaust_loss + congestion_drop, 0.0, 1.0)
            packet_loss_pct = packet_loss_ratio * 100.0

            # Goodput chuẩn IETF RFC 2647
            raw_goodput_mbps = min(offered_load_mbps, service_capacity_mbps)
            tcp_qoe_factor = (1.0 - packet_loss_ratio) * (
                1.0 - 0.16 * min(1.0, max(0.0, p99_latency_ms - 15.0) / 85.0)
            )
            goodput_mbps = raw_goodput_mbps * max(0.15, tcp_qoe_factor)

            # 5. HÀM MỤC TIÊU ĐA CHỈ TIÊU 100% PURE OUTCOME KPI
            reward = cls._compute_kpi_reward(
                goodput_mbps=goodput_mbps,
                offered_load_mbps=offered_load_mbps,
                service_capacity_mbps=service_capacity_mbps,
                p_c=p_c,
                per_phy=per_phy,
                p99_latency_ms=p99_latency_ms,
                packet_loss_pct=packet_loss_pct,
                tx_power_dbm=tx_power_dbm,
                is_invalid_bonding=is_invalid_bonding,
                tx_slew_pen=0.0,
            )

            return {
                "reward": reward,
                "throughput": service_capacity_mbps,
                "goodput": goodput_mbps,
                "offered_load": offered_load_mbps,
                "p_c": p_c,
                "retry_rate": retry_rate,
                "per_phy": per_phy,
                "operating_mcs": float(operating_mcs_idx),
                "safe_mcs": float(operating_mcs_idx),
                "eff_sinr_db": eff_sinr_db,
                "eff_rssi_dbm": rssi_compensated,
                "eff_noise_dbm": eff_noise_dbm,
                "chosen_sub_noise_dbm": chosen_sub_noise_dbm,
                "airtime_next": airtime_next,
                "ofdma_next": ofdma_efficiency,
                "mean_latency_ms": mean_latency_ms,
                "p99_latency_ms": p99_latency_ms,
                "jitter_ms": jitter_ms,
                "packet_loss_pct": packet_loss_pct,
                "is_csi_aging_regime": 1.0 if is_csi_aging_regime else 0.0,
                "bf_corr_rho": float(rho_csi),
                "legacy_share": float(alpha_leg),
                "legacy_prot_overhead": float(legacy_prot_overhead),
                "backlog_next_mb": float(backlog_next_mb),
                "w_pk_ms": float(w_pk_ms),
                "w_backlog_ms": float(w_backlog_ms),
            }
        except Exception as exc:
            logging.error(f"Lỗi trong _original_evaluate_network_physics: {exc}")
            return {
                "reward": -5.0, "throughput": 10.0, "goodput": 10.0, "offered_load": 50.0,
                "p_c": 0.9, "retry_rate": 0.5, "per_phy": 0.5, "operating_mcs": 4.0, "safe_mcs": 4.0,
                "eff_sinr_db": 5.0, "eff_rssi_dbm": -75.0, "eff_noise_dbm": -70.0,
                "chosen_sub_noise_dbm": -70.0, "airtime_next": 0.9, "ofdma_next": 0.2,
                "mean_latency_ms": 20.0, "p99_latency_ms": 50.0, "jitter_ms": 10.0,
                "packet_loss_pct": 25.0, "is_csi_aging_regime": 0.0,
                "bf_corr_rho": 1.0, "legacy_share": 0.0, "legacy_prot_overhead": 0.0,
                "backlog_next_mb": 0.0, "w_pk_ms": 2.0, "w_backlog_ms": 0.0,
            }

    @classmethod
    def evaluate_network_physics_with_downtime(
        cls,
        obs: np.ndarray,
        action: np.ndarray,
        prev_action: np.ndarray = None,
        aqm_enabled: bool = True,
        sub_noise_dbm: Optional[np.ndarray] = None,
        base_ofdma_capacity: Optional[float] = None,
        demand_scale: float = 1.0,
        queue_backlog_mb: float = 0.0,
    ) -> Dict[str, float]:
        """Đánh giá chi phí tái cấu hình thuần phần cứng (`RELOAD_DIMS` vs `HOT_DIMS=0`) và tính lại Reward sau suy hao KPI."""
        metrics = cls._original_evaluate_network_physics(
            obs, action, prev_action=None, aqm_enabled=aqm_enabled,
            sub_noise_dbm=sub_noise_dbm, base_ofdma_capacity=base_ofdma_capacity,
            demand_scale=demand_scale, queue_backlog_mb=queue_backlog_mb,
        )
        if prev_action is None:
            metrics["is_downtime_triggered"] = 0.0
            return metrics

        ch_changed = (int(action[0]) != int(prev_action[0]))
        bw_changed = (int(action[1]) != int(prev_action[1]))
        aux_uci_changed = (
            int(action[4]) != int(prev_action[4])
            or int(action[5]) != int(prev_action[5])
            or int(action[6]) != int(prev_action[6])
        )
        tx_idx = int(0 if int(action[2]) < 0 else (6 if int(action[2]) > 6 else int(action[2])))
        prev_tx_idx = int(0 if int(prev_action[2]) < 0 else (6 if int(prev_action[2]) > 6 else int(prev_action[2])))
        _, _, _, is_invalid_bonding = resolve_valid_5g_channel_and_bw(int(action[0]), int(action[1]))

        if ch_changed and bw_changed:
            reload_frac, lat_d, jit_d, loss_d = 0.10, 1.35, 0.50, 0.15
        elif ch_changed:
            reload_frac, lat_d, jit_d, loss_d = 0.08, 1.10, 0.40, 0.12
        elif bw_changed:
            reload_frac, lat_d, jit_d, loss_d = 0.05, 0.65, 0.25, 0.06
        elif aux_uci_changed:
            reload_frac, lat_d, jit_d, loss_d = 0.025, 0.30, 0.12, 0.03
        else:
            reload_frac, lat_d, jit_d, loss_d = 0.0, 0.0, 0.0, 0.0

        tx_slew_pen = 0.08 * max(0, abs(tx_idx - prev_tx_idx) - 2)
        is_downtime_triggered = (reload_frac > 0.0)

        if is_downtime_triggered:
            metrics["throughput"] = float(max(6.0, metrics["throughput"] * (1.0 - reload_frac)))
            metrics["goodput"] = float(metrics["goodput"] * (1.0 - reload_frac))
            metrics["mean_latency_ms"] = float(metrics["mean_latency_ms"] + lat_d)
            metrics["jitter_ms"] = float(metrics["jitter_ms"] + jit_d)
            metrics["p99_latency_ms"] = gamma_p99_from_moments(metrics["mean_latency_ms"], metrics["jitter_ms"], d_min=1.50)
            metrics["packet_loss_pct"] = float(min(100.0, metrics["packet_loss_pct"] + loss_d))
            b_max_mb = 0.65 if aqm_enabled else 18.0
            metrics["backlog_next_mb"] = float(
                cls._clip_scalar(
                    metrics["backlog_next_mb"] + metrics["offered_load"] * reload_frac * 0.22,
                    0.0,
                    b_max_mb,
                )
            )

        if is_downtime_triggered or tx_slew_pen > 0.0:
            metrics["reward"] = cls._compute_kpi_reward(
                goodput_mbps=metrics["goodput"],
                offered_load_mbps=metrics["offered_load"],
                service_capacity_mbps=metrics["throughput"],
                p_c=metrics["p_c"],
                per_phy=metrics["per_phy"],
                p99_latency_ms=metrics["p99_latency_ms"],
                packet_loss_pct=metrics["packet_loss_pct"],
                tx_power_dbm=14.0 + float(tx_idx),
                is_invalid_bonding=is_invalid_bonding,
                tx_slew_pen=tx_slew_pen,
            )

        metrics["is_downtime_triggered"] = 1.0 if is_downtime_triggered else 0.0
        return metrics

    evaluate_network_physics = evaluate_network_physics_with_downtime

    @classmethod
    def compute_optimal_targets(
        cls,
        obs: np.ndarray,
        aqm_enabled: bool = True,
        sub_noise_dbm: Optional[np.ndarray] = None,
        base_ofdma_capacity: Optional[float] = None,
        demand_scale: float = 1.0,
        queue_backlog_mb: float = 0.0,
    ) -> Tuple[np.ndarray, Dict[str, float]]:
        """Tìm nghiệm tối ưu toàn cục thực sự argmax_a R_physics(s, a) bằng vector hóa NumPy chính xác tuyệt đối trên 21,000 cấu hình không bị trội."""
        rssi, noise_obs, sta, retry_obs, pc_obs, airtime_obs, ofdma_obs, is_break = [float(x) for x in obs]
        spec = (
            cls._construct_9ch_spectrum_profile(obs)
            if sub_noise_dbm is None
            else np.asarray(sub_noise_dbm, dtype=np.float64)
        )
        is_break_flag = (int(round(is_break)) == 1)
        bcn_opt = 0 if is_break_flag else 1

        # 1. Tính trước đặc trưng 25 cặp (ch, bw) hợp lệ
        pri_n = np.empty(25, dtype=np.float64)
        occ_max_n = np.empty(25, dtype=np.float64)
        sec_max_n = np.empty(25, dtype=np.float64)
        eff_n = np.empty(25, dtype=np.float64)
        ch_map = cls._CH_INDEX_MAP
        dbm_sum = cls._dbm_sum
        for idx, (ch_i, bw_i) in enumerate(cls._ORACLE_VALID_PAIRS):
            pri_n[idx] = float(spec[ch_i])
            _, ht, occ_subs, _ = resolve_valid_5g_channel_and_bw(ch_i, bw_i)
            if ht == "HE20":
                occ_max_n[idx] = pri_n[idx]
                sec_max_n[idx] = -100.0
                eff_n[idx] = pri_n[idx]
            elif ht == "HE40":
                i0, i1 = ch_map[occ_subs[0]], ch_map[occ_subs[1]]
                n0, n1 = float(spec[i0]), float(spec[i1])
                occ_max_n[idx] = n0 if n0 >= n1 else n1
                sec_max_n[idx] = n1 if ch_i == i0 else n0
                eff_n[idx] = dbm_sum(n0, n1)
            else:
                ii = [ch_map[s] for s in occ_subs]
                nn = [float(spec[k]) for k in ii]
                occ_max_n[idx] = max(nn)
                sec_max_n[idx] = max(nv for ki, nv in zip(ii, nn) if ki != ch_i)
                eff_n[idx] = dbm_sum(dbm_sum(nn[0], nn[1]), dbm_sum(nn[2], nn[3]))

        bf_gain_9 = np.empty(9, dtype=np.float64)
        bf_ovh_9 = np.empty(9, dtype=np.float64)
        for c_i in range(9):
            _, g_v, ov_v, _ = cls._compute_jakes_bf_physics(c_i, sta, retry_obs, airtime_obs, is_break)
            bf_gain_9[c_i] = g_v
            bf_ovh_9[c_i] = ov_v

        eff_ofdma_exog = ofdma_obs if base_ofdma_capacity is None else float(base_ofdma_capacity)
        alpha_leg = cls._clip_scalar((0.58 - eff_ofdma_exog) / 0.50, 0.0, 1.0)
        escaped_jam = (noise_obs > -84.0) & (occ_max_n <= -88.0)
        if base_ofdma_capacity is None:
            obs_ofdma_clean = np.where(escaped_jam, max(ofdma_obs, 0.52), 0.5 * ofdma_obs + 0.5 * eff_ofdma_exog)
        else:
            obs_ofdma_clean = np.full(25, 0.5 * ofdma_obs + 0.5 * eff_ofdma_exog, dtype=np.float64)
        alpha_leg_sr_p = np.clip((0.58 - obs_ofdma_clean) / 0.50, 0.0, 1.0)

        edge_spread_db = 3.5 + 4.8 * min(1.4, sta / 85.0)
        fading_p = 1.0 + 1.8 * alpha_leg + np.where(occ_max_n > -86.0, 2.5, 0.0)
        ewma_lag_p = 2.0 + np.where(occ_max_n > -85.0, 3.2, 0.0) + 2.2 * min(1.0, sta / 100.0) + (1.5 if is_break_flag else 0.0)

        pri_stall_p = 0.14 * np.clip((pri_n - (-85.0)) / 24.0, 0.0, 1.0)
        sec_block_p = 0.06 * np.clip((sec_max_n - (-82.0)) / 24.0, 0.0, 1.0)
        ch_interf_p = np.clip((eff_n - (-95.0)) / 34.0 + pri_stall_p + sec_block_p, 0.01, 0.94)
        bw_cont_p = 1.0 + cls._ORACLE_P_BW * (0.55 * (sta / 50.0) + 1.15 * alpha_leg + np.where(occ_max_n > -86.0, 0.75, 0.0))
        ofdma_eff_p = np.clip(0.62 * eff_ofdma_exog + 0.22 * (1.0 - 0.22 * cls._ORACLE_P_BW) - 0.25 * ch_interf_p, 0.08, 0.95)
        pc_base_p = 0.38 * ch_interf_p + 0.26 * (sta / 110.0) * bw_cont_p * (1.0 - 0.42 * ofdma_eff_p)
        leg_prot_p = 0.085 * (alpha_leg * alpha_leg) * (1.0 + 0.45 * cls._ORACLE_P_BW)

        # 2. Mở rộng tensor trên 1,750 tổ hợp (25 cặp ch/bw x 2 BF x 5 OBSS x 7 Tx) x 12 MCS = 21,000 cấu hình
        g_p = cls._ORACLE_G_P
        g_bf = cls._ORACLE_G_BF
        g_ob = cls._ORACLE_G_OB
        g_tx = cls._ORACLE_G_TX
        mcs_req = cls._ORACLE_MCS_REQ
        mcs_rate = cls._ORACLE_MCS_RATE

        ch_g = cls._ORACLE_P_CH[g_p]
        tx_dbm_g = 14.0 + g_tx.astype(np.float64)
        rx_edge_g = rssi - edge_spread_db + (tx_dbm_g - 17.0) - 0.25 * cls._ORACLE_P_FLOSS[g_p]
        rx_close_g = rssi + (tx_dbm_g - 17.0)
        lna_ov_g = np.maximum(0.0, rx_close_g - (-43.5)) * 1.65
        bf_gain_g = np.where(g_bf == 1, bf_gain_9[ch_g], 0.0)
        csi_ovh_g = np.where(g_bf == 1, bf_ovh_9[ch_g], 0.0)

        sr_norm_g = g_ob.astype(np.float64) / 4.0
        al_sr_g = alpha_leg_sr_p[g_p]
        cca_def_g = 0.18 * ((1.0 - sr_norm_g) ** 1.4) * ((1.0 - al_sr_g) ** 1.3)
        hid_coll_g = 0.26 * (sr_norm_g ** 1.5) * (0.15 + 1.35 * al_sr_g)
        leak_g = (sr_norm_g ** 1.6) * (1.0 + 8.5 * al_sr_g)
        eff_sinr_g = rx_edge_g + bf_gain_g - eff_n[g_p] - fading_p[g_p] - lna_ov_g - leak_g
        ewma_lag_g = ewma_lag_p[g_p]

        per_inst = 1.0 / (1.0 + np.exp(np.clip(0.95 * (eff_sinr_g[:, None] - mcs_req[None, :] - 1.8), -40.0, 40.0)))
        per_ewma = 1.0 / (1.0 + np.exp(np.clip(0.95 * (eff_sinr_g[:, None] + ewma_lag_g[:, None] - mcs_req[None, :] - 1.8), -40.0, 40.0)))
        p_succ = 1.0 - per_ewma
        tp_ewma = np.where(p_succ >= 0.10, np.minimum(0.90, p_succ), 0.0) * mcs_rate[None, :]
        prob_sc = p_succ * np.minimum(mcs_rate[None, :], 68.8)

        op_mcs = np.empty((1750, 12), dtype=np.float64)
        eff_mcs_rate = np.empty((1750, 12), dtype=np.float64)
        per_phy = np.empty((1750, 12), dtype=np.float64)
        best_tp = np.full(1750, -1.0)
        r0 = np.zeros(1750, dtype=np.int64)
        r1 = np.zeros(1750, dtype=np.int64)
        best_ps = np.full(1750, -1.0)
        rp = np.zeros(1750, dtype=np.int64)
        row_idx = cls._ORACLE_ROW_IDX
        cum_rate_succ = np.cumsum(mcs_rate[None, :] * (1.0 - per_inst), axis=1)
        cum_per = np.cumsum(per_inst, axis=1)

        for c_idx in range(12):
            upd = tp_ewma[:, c_idx] > best_tp
            r1 = np.where(upd, r0, r1)
            best_tp = np.where(upd, tp_ewma[:, c_idx], best_tp)
            r0 = np.where(upd, c_idx, r0)

            upd_p = prob_sc[:, c_idx] > best_ps
            best_ps = np.where(upd_p, prob_sc[:, c_idx], best_ps)
            rp = np.where(upd_p, c_idx, rp)

            p0 = per_inst[row_idx, r0]
            p1 = per_inst[row_idx, r1]
            norm_rate = (1.0 - p0) * mcs_rate[r0] + p0 * (1.0 - p1) * mcs_rate[r1] + p0 * p1 * mcs_rate[rp] * 0.65
            norm_per = p0 * 0.58 + p0 * p1 * 0.42
            has_pr = c_idx > r0
            n_pr = np.maximum(1, c_idx - r0).astype(np.float64)
            pr_r_sum = cum_rate_succ[:, c_idx] - cum_rate_succ[row_idx, r0]
            pr_p_sum = cum_per[:, c_idx] - cum_per[row_idx, r0]
            eff_mcs_rate[:, c_idx] = np.where(has_pr, 0.36 * (0.85 * norm_rate + 0.15 * (pr_r_sum / n_pr)), 0.36 * norm_rate)
            per_phy[:, c_idx] = np.clip(np.where(has_pr, 0.85 * norm_per + 0.15 * (pr_p_sum / n_pr), norm_per), 0.002, 0.96)
            op_mcs[:, c_idx] = r0.astype(np.float64)

        p_c = np.clip(pc_base_p[g_p, None] + 0.38 * per_phy + cca_def_g[:, None] + hid_coll_g[:, None], 0.02, 0.96)
        retry_r = np.clip(0.55 * p_c + 0.45 * per_phy, 0.01, 0.96)

        bcn_ovh = 0.055 if bcn_opt == 0 else 0.012
        roam_fac = (1.00 if bcn_opt == 0 else 0.74) if is_break_flag else 1.00
        tonal_g = cls._ORACLE_P_TONAL[g_p, None]
        r_he_raw = np.maximum(4.0, eff_mcs_rate * tonal_g)
        r_leg_cap = np.minimum(r_he_raw, 26.0 + 2.0 * op_mcs)
        r_harm = 1.0 / ((1.0 - alpha_leg) / r_he_raw + alpha_leg / np.maximum(r_leg_cap, 1.0))
        eta_h = (0.72 + 0.28 * (r_harm / r_he_raw)) if aqm_enabled else (r_harm / r_he_raw)
        tot_ovh = np.minimum(0.45, bcn_ovh + csi_ovh_g[:, None] + leg_prot_p[g_p, None] + 0.60 * cca_def_g[:, None])
        ofdma_g = ofdma_eff_p[g_p, None]
        srv_cap = np.maximum(6.0, r_he_raw * eta_h * roam_fac * (0.88 + 0.24 * ofdma_g) * (1.0 - tot_ovh) * (1.0 - p_c) * (1.0 - 0.65 * per_phy))

        exog_ls = cls._clip_scalar((rssi - (-88.0)) / 38.0, 0.32, 1.10)
        if sta <= 30.0 and alpha_leg < 0.40 and rssi >= -60.0:
            raw_d = 148.0 + 1.25 * sta
        elif sta <= 32.0 and alpha_leg < 0.40:
            raw_d = 86.0 + 1.05 * sta
        elif sta <= 78.0 and alpha_leg < 0.55:
            raw_d = 52.0 + 0.35 * sta
        else:
            raw_d = 18.0 + 0.05 * sta
        off_load = raw_d * exog_ls * float(demand_scale)
        cur_bkl = max(0.0, float(queue_backlog_mb))

        es_ms = 12.0 / srv_cap
        cs2 = np.maximum(0.25, 0.55 + 2.8 * (p_c * p_c) + 1.6 * retry_r - 0.30 * ofdma_g)
        rho = off_load / srv_cap
        ov_rat = np.maximum(0.0, (off_load - srv_cap) / max(off_load, 1.0))
        rho_eff = np.minimum(rho, 0.92 if aqm_enabled else 0.985)
        w_pk = (rho_eff * es_ms * (1.0 + cs2)) / (2.0 * np.maximum(0.015, 1.0 - rho_eff))
        if aqm_enabled:
            w_pk = np.minimum(w_pk, 3.8 + 2.4 * np.maximum(0.0, rho - 0.85))
            c_drop = 0.045 * ov_rat
        else:
            w_pk = w_pk + 28.0 * np.maximum(0.0, rho - 0.90)
            c_drop = 0.48 * ov_rat

        w_bkl = (cur_bkl / srv_cap) * 1000.0
        retrans_d = 1.10 * (retry_r / np.maximum(1.0 - retry_r, 0.16))
        hand_d = 3.2 if (is_break_flag and bcn_opt == 1) else 0.0
        mean_lat = 1.50 + w_pk + w_bkl + retrans_d + hand_d
        jit = 0.12 + 1.25 * (p_c * p_c) + ((1.50 if aqm_enabled else 8.00) * ov_rat) + 0.42 * (alpha_leg * alpha_leg)

        ex_m = np.maximum(0.05, mean_lat - 1.50)
        sig_j = np.maximum(0.04, jit)
        sh_k = np.maximum(0.25, (ex_m / sig_j) ** 2.0)
        wh_t = np.maximum(0.15, 1.0 - 1.0 / (9.0 * sh_k) + 2.326347874 / (3.0 * np.sqrt(sh_k)))
        p99_lat = 1.50 + ex_m * (wh_t ** 3.0)

        loss_rat = np.clip((p_c * per_phy) ** 3.0 + c_drop, 0.0, 1.0)
        loss_pct = loss_rat * 100.0
        raw_gp = np.minimum(off_load, srv_cap)
        tcp_f = (1.0 - loss_rat) * (1.0 - 0.16 * np.minimum(1.0, np.maximum(0.0, p99_lat - 15.0) / 85.0))
        gp = raw_gp * np.maximum(0.15, tcp_f)

        gp_ful = gp / max(off_load, 1.0)
        cap_hd = 0.52 * np.log1p(srv_cap / max(off_load, 25.0))
        en_pen = 0.032 * (tx_dbm_g[:, None] - 14.0)
        pen_col = 3.40 * (p_c * p_c) + 2.10 * per_phy
        pen_lat = 0.042 * np.minimum(p99_lat, 60.0) + 1.25 * np.tanh(np.maximum(0.0, p99_lat - 15.0) / 15.0)
        pen_lss = 3.20 * loss_rat + np.where(loss_pct >= 1.0, 0.60, 0.0)
        sla_b = np.where((p99_lat <= 15.0) & (loss_pct < 1.0), 0.60, -0.40)
        rew = 2.60 * gp_ful + 0.65 * (gp / 100.0) + cap_hd - pen_col - pen_lat - pen_lss - en_pen + sla_b

        flat_idx = int(np.argmax(rew))
        g_idx, mcs_best = divmod(flat_idx, 12)
        p_idx = int(g_p[g_idx])
        best_act = np.array(
            [int(cls._ORACLE_P_CH[p_idx]), int(cls._ORACLE_P_BW[p_idx]), int(g_tx[g_idx]), int(mcs_best), int(bcn_opt), int(g_ob[g_idx]), int(g_bf[g_idx])],
            dtype=np.int64,
        )
        best_m = cls._original_evaluate_network_physics(
            obs, best_act, aqm_enabled=aqm_enabled, sub_noise_dbm=spec,
            base_ofdma_capacity=base_ofdma_capacity, demand_scale=demand_scale, queue_backlog_mb=queue_backlog_mb,
        )
        return best_act, best_m

    def _sample_markov_regime(self) -> None:
        """
        Sinh pha môi trường Markov liên tục với thời lượng ngẫu nhiên (15..55 bước)
        và phân bố phổ nhiễu hoàn toàn đối xứng trên cả 9 kênh [36..165].
        """
        self.regime_remaining_steps = int(self.rng.integers(15, 56))
        regime = int(self.rng.choice([0, 1, 2, 3, 4], p=[0.24, 0.28, 0.22, 0.15, 0.11]))
        self.scenario_mode = regime

        base_clean = self.rng.uniform(-96.8, -92.5, size=9).astype(np.float32)
        self.target_rssi = float(self.rng.uniform(-78.0, -40.0))

        if regime == 0:  # Bình yên: toàn phổ 9 kênh sạch
            self.target_ch_noise = base_clean
            self.target_sta = float(self.rng.uniform(4.0, 32.0))
            self.target_ofdma_base = float(self.rng.uniform(0.68, 0.92))
            self.target_demand_scale = float(self.rng.uniform(0.85, 1.15))
            self.current_is_break = 1.0 if self.rng.random() < 0.20 else 0.0

        elif regime == 1:  # Nhiễu hẹp 1..2 kênh lân cận trong cùng dải tần vật lý UNII-1 [0..3] hoặc UNII-3 [4..8]
            self.target_ch_noise = base_clean
            jam_center = int(self.rng.integers(0, 9))
            jam_lvl = float(self.rng.uniform(-82.5, -71.0))
            self.target_ch_noise[jam_center] = jam_lvl
            band_lo, band_hi = (0, 3) if jam_center < 4 else (4, 8)
            valid_neighbors = [c for c in (jam_center - 1, jam_center + 1) if band_lo <= c <= band_hi]
            if valid_neighbors and self.rng.random() < 0.55:
                neighbor = int(self.rng.choice(valid_neighbors))
                self.target_ch_noise[neighbor] = jam_lvl - float(self.rng.uniform(1.5, 4.0))
            self.target_sta = float(self.rng.uniform(12.0, 62.0))
            self.target_ofdma_base = float(self.rng.uniform(0.45, 0.72))
            self.target_demand_scale = float(self.rng.uniform(0.80, 1.10))
            self.current_is_break = 1.0 if self.rng.random() < 0.25 else 0.0

        elif regime == 2:  # Nhiễu băng rộng đánh sập 1 cụm 40/80 MHz (UNII-1, UNII-3 Low/Mid, 40MHz, hoặc cụm cao CH157..165)
            self.target_ch_noise = base_clean
            cluster_type = int(self.rng.choice([0, 1, 2, 3], p=[0.30, 0.30, 0.25, 0.15]))
            jam_lvl = float(self.rng.uniform(-71.5, -54.0))
            if cluster_type == 0:
                self.target_ch_noise[0:4] = jam_lvl - self.rng.uniform(0.0, 2.5, size=4)
            elif cluster_type == 1:
                self.target_ch_noise[4:8] = jam_lvl - self.rng.uniform(0.0, 2.5, size=4)
            elif cluster_type == 2:
                pair_start = int(self.rng.choice([0, 2, 4, 6]))
                self.target_ch_noise[pair_start:pair_start + 2] = jam_lvl - self.rng.uniform(0.0, 2.0, size=2)
            else:
                self.target_ch_noise[6:9] = jam_lvl - self.rng.uniform(0.0, 2.2, size=3)
            self.target_sta = float(self.rng.uniform(12.0, 68.0))
            self.target_ofdma_base = float(self.rng.uniform(0.32, 0.62))
            self.target_demand_scale = float(self.rng.uniform(0.80, 1.05))
            self.current_is_break = 1.0 if self.rng.random() < 0.35 else 0.0

        elif regime == 3:  # Bão Roaming giờ giải lao (Mật độ cao + Di động lớn -> kích hoạt CSI Aging BF=OFF & Beacon 80ms)
            self.target_ch_noise = base_clean
            if self.rng.random() < 0.40:
                jam_k = int(self.rng.integers(0, 9))
                self.target_ch_noise[jam_k] = float(self.rng.uniform(-79.0, -68.0))
            self.target_rssi = float(self.rng.uniform(-82.0, -56.0))
            self.target_sta = float(self.rng.uniform(86.0, 195.0))
            self.target_ofdma_base = float(self.rng.uniform(0.22, 0.42))
            self.target_demand_scale = float(self.rng.uniform(0.75, 1.05))
            self.current_is_break = 1.0 if self.rng.random() < 0.85 else 0.0

        else:  # Bão UDP / Thiết bị đời cũ chiếm dụng airtime (OFDMA thấp -> cần OBSS-PD -82 & HE20)
            self.target_ch_noise = base_clean
            if self.rng.random() < 0.35:
                jam_k = int(self.rng.integers(0, 9))
                self.target_ch_noise[jam_k] = float(self.rng.uniform(-78.0, -66.0))
            self.target_sta = float(self.rng.uniform(14.0, 50.0))
            self.target_ofdma_base = float(self.rng.uniform(0.08, 0.24))
            self.target_demand_scale = float(self.rng.uniform(0.90, 1.25))
            self.current_is_break = 1.0 if self.rng.random() < 0.30 else 0.0

    def sample_snapshot_scenario(self) -> Tuple[np.ndarray, np.ndarray, float, float]:
        """Sinh một kịch bản snapshot nhất quán vật lý `(obs_8d, true_ch_noise_9d, ofdma_base, demand_scale)` để kiểm định."""
        self._sample_markov_regime()
        true_noise = np.clip(self.target_ch_noise + self.rng.normal(0.0, 0.4, size=9), -102.0, -51.0).astype(np.float32)
        ch_init = int(self.rng.integers(0, 9))
        # Cho khoảng 35% snapshot đang đứng sẵn trên kênh bị nhiễu (khi có nhiễu) để kiểm tra khả năng phản xạ thoát kênh
        noisy_chs = [i for i in range(9) if true_noise[i] > -83.0]
        if noisy_chs and self.rng.random() < 0.60:
            ch_init = int(self.rng.choice(noisy_chs))
        self.current_ch = ch_init

        noise_init = float(np.clip(true_noise[ch_init], OBS_LOW[1], OBS_HIGH[1]))
        sta_init = float(np.clip(self.target_sta, OBS_LOW[2], OBS_HIGH[2]))
        rssi_init = float(np.clip(self.target_rssi, OBS_LOW[0], OBS_HIGH[0]))
        approx_air = float(np.clip(0.18 + sta_init / 220.0 + (0.28 if noise_init > -82 else 0.0) + (0.25 if self.target_ofdma_base < 0.3 else 0.0), 0.12, 0.96))
        approx_pc = float(np.clip(0.05 + 0.35 * max(0.0, (noise_init + 95.0) / 35.0) + 0.35 * max(0.0, 0.65 - self.target_ofdma_base) + 0.22 * (sta_init / 190.0), 0.03, 0.92))
        approx_retry = float(np.clip(0.85 * approx_pc + self.rng.uniform(-0.04, 0.04), 0.02, 0.90))
        approx_ofdma = float(np.clip(self.target_ofdma_base * (1.0 - 0.35 * approx_pc), 0.06, 0.94))
        obs = np.array([
            rssi_init, noise_init, sta_init, approx_retry, approx_pc, approx_air, approx_ofdma, self.current_is_break
        ], dtype=np.float32)
        obs = np.clip(obs, OBS_LOW, OBS_HIGH)
        return obs, true_noise, float(self.target_ofdma_base), float(self.target_demand_scale)

    def _format_obs_output(self, true_obs: np.ndarray) -> np.ndarray:
        out = true_obs.astype(np.float32, copy=True)
        if self.obs_mask_phy:
            out[1] = -95.0
            out[6] = 0.50
        return out

    def reset(self, *, seed: Optional[int] = None, options: Optional[dict] = None) -> Tuple[np.ndarray, dict]:
        super().reset(seed=seed)
        if seed is not None:
            self.base_seed = int(seed)
            self.rng = np.random.default_rng(seed)
        elif self.current_step == 0:
            # Khi EvalCallback bắt đầu lượt đánh giá mới ngay sau auto-reset (current_step == 0),
            # đưa luồng RNG về lại base_seed để mọi kỳ EvalCallback chấm điểm trên cùng bộ kịch bản cố định (CRN)
            self.rng = np.random.default_rng(self.base_seed)
        self.current_step = 0
        self.previous_action = None
        self.queue_backlog_mb = 0.0

        obs0, true_noise, _, _ = self.sample_snapshot_scenario()
        self.true_ch_noise = true_noise.copy()
        self.true_current_obs = obs0.copy()
        self.current_obs = self._format_obs_output(obs0)
        return self.current_obs.copy(), {"true_ch_noise": self.true_ch_noise.copy(), "current_ch": self.current_ch}

    def step(self, action: np.ndarray):
        """
        Chuyển tiếp trạng thái Semi-Markov Regime Switching + Ornstein-Uhlenbeck chính xác (dt = 10s):
          X_{t+dt} = X_t * e^{-theta*dt} + mu * (1 - e^{-theta*dt}) + sigma_stat * sqrt(1 - e^{-2*theta*dt}) * eps
        và cập nhật tồn dư hàng đợi M/G/1 Lindley (`self.queue_backlog_mb`) + phép đo survey `mt76`.
        """
        self.current_step += 1
        act = np.asarray(action, dtype=np.int64).reshape(-1)
        prev_obs = self.true_current_obs if self.true_current_obs is not None else self.current_obs

        metrics = self.evaluate_network_physics(
            prev_obs,
            act,
            prev_action=self.previous_action,
            aqm_enabled=self.aqm_enabled,
            sub_noise_dbm=self.true_ch_noise,
            base_ofdma_capacity=self.target_ofdma_base,
            demand_scale=self.target_demand_scale,
            queue_backlog_mb=self.queue_backlog_mb,
        )
        self.current_ch = int(np.clip(act[0], 0, len(NON_DFS_5G_CHANNELS) - 1))
        self.previous_action = act.copy()
        self.queue_backlog_mb = float(metrics.get("backlog_next_mb", 0.0))

        # Chuyển pha Semi-Markov với thời lượng ngẫu nhiên 15..55 bước
        self.regime_remaining_steps -= 1
        if self.regime_remaining_steps <= 0:
            self._sample_markov_regime()

        # Lấy mẫu giải tích chính xác quá trình Ornstein-Uhlenbeck cho 9 kênh phổ tại dt = 10.0s
        decay_rf = math.exp(-OU_THETA_RF * DT_STEP_SEC)
        std_rf = OU_SIGMA_STAT_RF * math.sqrt(1.0 - decay_rf * decay_rf)
        noise_drift = self.rng.normal(0.0, std_rf, size=9).astype(np.float32)
        self.true_ch_noise = np.clip(
            decay_rf * self.true_ch_noise + (1.0 - decay_rf) * self.target_ch_noise + noise_drift,
            -102.0,
            -51.0,
        ).astype(np.float32)

        # Lấy mẫu OU chính xác cho RSSI & mật độ STA ngoại sinh tại dt = 10.0s
        decay_rssi = math.exp(-OU_THETA_RSSI * DT_STEP_SEC)
        std_rssi = OU_SIGMA_STAT_RSSI * math.sqrt(1.0 - decay_rssi * decay_rssi)
        rssi_next = float(np.clip(
            decay_rssi * float(prev_obs[0]) + (1.0 - decay_rssi) * self.target_rssi + self.rng.normal(0.0, std_rssi),
            OBS_LOW[0],
            OBS_HIGH[0],
        ))

        decay_sta = math.exp(-OU_THETA_STA * DT_STEP_SEC)
        std_sta = OU_SIGMA_STAT_STA * math.sqrt(1.0 - decay_sta * decay_sta)
        sta_next = float(np.clip(
            decay_sta * float(prev_obs[2]) + (1.0 - decay_sta) * self.target_sta + self.rng.normal(0.0, std_sta),
            OBS_LOW[2],
            OBS_HIGH[2],
        ))

        # CẢM BIẾN PHY: mô phỏng đúng cơ chế khảo sát `iw survey dump [in use]` trên driver mt76
        survey_noise_dbm = measure_mt76_survey_noise(self.true_ch_noise, self.current_ch, int(np.clip(act[1], 0, 2)))
        noise_next = float(np.clip(survey_noise_dbm + self.rng.normal(0.0, 0.25), OBS_LOW[1], OBS_HIGH[1]))

        # PHẢN HỒI NHÂN QUẢ TẦNG MAC (75% trực tiếp từ kết quả thực thi `metrics` của Agent)
        retry_next = float(np.clip(0.25 * float(prev_obs[3]) + 0.75 * metrics["retry_rate"] + self.rng.normal(0.0, 0.012), OBS_LOW[3], OBS_HIGH[3]))
        pc_next = float(np.clip(0.25 * float(prev_obs[4]) + 0.75 * metrics["p_c"] + self.rng.normal(0.0, 0.012), OBS_LOW[4], OBS_HIGH[4]))
        air_next = float(np.clip(0.25 * float(prev_obs[5]) + 0.75 * metrics["airtime_next"] + self.rng.normal(0.0, 0.015), OBS_LOW[5], OBS_HIGH[5]))
        # Khi ở kênh sạch, ofdma phục hồi dần về năng lực của thiết bị thật (`target_ofdma_base`) trừ suy giảm va chạm
        ofdma_causal = float(np.clip(self.target_ofdma_base * (1.0 - 0.55 * metrics["p_c"] - 0.25 * metrics["retry_rate"]), 0.06, 0.95))
        ofdma_target_step = 0.5 * metrics["ofdma_next"] + 0.5 * ofdma_causal
        ofdma_next = float(np.clip(0.25 * float(prev_obs[6]) + 0.75 * ofdma_target_step + self.rng.normal(0.0, 0.012), OBS_LOW[6], OBS_HIGH[6]))

        true_next_obs = np.array(
            [rssi_next, noise_next, sta_next, retry_next, pc_next, air_next, ofdma_next, self.current_is_break],
            dtype=np.float32,
        )
        self.true_current_obs = true_next_obs
        self.current_obs = self._format_obs_output(true_next_obs)

        terminated = bool(self.current_step >= self.max_steps)
        info = dict(metrics)
        info["true_ch_noise"] = self.true_ch_noise.copy()
        info["current_ch"] = self.current_ch
        return self.current_obs.copy(), float(metrics["reward"]), terminated, False, info

    def simulate_future(
        self,
        current_real_state: np.ndarray,
        action: np.ndarray,
        sub_noise_dbm: Optional[np.ndarray] = None,
    ) -> float:
        metrics = self._original_evaluate_network_physics(
            current_real_state,
            action,
            prev_action=None,
            aqm_enabled=self.aqm_enabled,
            sub_noise_dbm=sub_noise_dbm,
            queue_backlog_mb=self.queue_backlog_mb,
        )
        return float(metrics["reward"])

    def oracle_action(
        self,
        current_real_state: np.ndarray,
        sub_noise_dbm: Optional[np.ndarray] = None,
    ) -> np.ndarray:
        opt_act, _ = self.compute_optimal_targets(
            current_real_state, aqm_enabled=self.aqm_enabled, sub_noise_dbm=sub_noise_dbm,
            queue_backlog_mb=self.queue_backlog_mb,
        )
        return opt_act


DigitalTwinEnv = UltimateDigitalTwinEnv


print("=" * 88)
print("[KIỂM ĐỊNH ĐA HÌNH PHẦN CỨNG & GUARDRAIL TRÊN 3 THẾ HỆ ROUTER (9 KÊNH NON-DFS)]")
raw_test_action = np.array([4, 2, 6, 11, 0, 4, 1])  # Yêu cầu: CH149 (idx 4), HE80, 20dBm, MCS11, Bcn80, OBSS=-62dBm, BF=ON
for hw in ["WIFI_6", "WIFI_5_AC", "WIFI_4_LEGACY"]:
    wrapped_env = HardwareAwareWrapper(UltimateDigitalTwinEnv(hardware_type=hw), hardware_type=hw, verbose=True)
    filtered_act = wrapped_env.action(raw_test_action)
    print(f"   -> Cấu hình thực thi trên {hw:13s}: {filtered_act.tolist()} (Kênh {NON_DFS_5G_CHANNELS[filtered_act[0]]}, Vũ khí mở: {len(wrapped_env.profile['active_weapons'])}/7)")
print("=" * 88)

# Thiết lập thư mục lưu trữ & dọn file monitor cũ để không cắt cụt biểu đồ Learning Curve
log_dir = os.path.join(DRIVE_BASE_PATH, "wifi_rl_logs")
save_dir = os.path.join(DRIVE_BASE_PATH, "saved_models")
os.makedirs(log_dir, exist_ok=True)
os.makedirs(save_dir, exist_ok=True)

def make_dual_sensing_env(seed_val: int, hardware_type: str = "WIFI_6", aqm_enabled: bool = True, obs_mask_phy: bool = False):
    def _init():
        base_env = UltimateDigitalTwinEnv(
            hardware_type=hardware_type,
            aqm_enabled=aqm_enabled,
            seed=seed_val,
            obs_mask_phy=obs_mask_phy,
        )
        hw_env = HardwareAwareWrapper(base_env, hardware_type=hardware_type)
        return StatefulChannelGuardWrapper(hw_env)
    return _init

vec_train_env = VecFrameStack(
    make_vec_env(
        make_dual_sensing_env(GLOBAL_SEED),
        n_envs=4,
        seed=GLOBAL_SEED,
    ),
    n_stack=N_FRAME_STACK,
)

policy_kwargs = dict(
    features_extractor_class=NormalizedWifiFeaturesExtractor,
    net_arch=dict(pi=[256, 256], vf=[256, 256]),
)

best_model_path = os.path.join(save_dir, "best_model.zip")
FORCE_RETRAIN_STEP3 = False  # Đặt True để huấn luyện mới trên Lõi Continuous-Markov POMDP (8D + VecFrameStack 4)

custom_objects_step3 = {
    "learning_rate": piecewise_lr_schedule(4.0e-4, min_lr=1.5e-5),
    "lr_schedule": piecewise_lr_schedule(4.0e-4, min_lr=1.5e-5),
    "clip_range": lambda _: 0.2,
    "NormalizedWifiFeaturesExtractor": NormalizedWifiFeaturesExtractor,
}

loaded_ok = False
if os.path.exists(best_model_path) and not FORCE_RETRAIN_STEP3:
    try:
        candidate_m = PPO.load(best_model_path, custom_objects=custom_objects_step3, device="cpu")
        if list(candidate_m.action_space.nvec) == ACTION_NVEC and int(candidate_m.observation_space.shape[0]) == 8 * N_FRAME_STACK:
            model = candidate_m
            model.set_env(vec_train_env)
            loaded_ok = True
            print(f"[+] Đã nạp thành công checkpoint 9 kênh Non-DFS (VecFrameStack={N_FRAME_STACK}) tại: {best_model_path}")
    except Exception as e:
        print(f"[!] Không thể nạp checkpoint cũ ({e}). Tiến hành huấn luyện mới...")

if not loaded_ok:
    for old_mf in glob.glob(os.path.join(log_dir, "*.monitor.csv")):
        try:
            os.remove(old_mf)
        except OSError:
            pass
    vec_train_env = VecFrameStack(
        make_vec_env(
            make_dual_sensing_env(GLOBAL_SEED),
            n_envs=4,
            seed=GLOBAL_SEED,
            monitor_dir=log_dir,
        ),
        n_stack=N_FRAME_STACK,
    )
    eval_env = VecFrameStack(
        DummyVecEnv([lambda: Monitor(make_dual_sensing_env(GLOBAL_SEED + 99)(), filename=os.path.join(log_dir, "eval.monitor.csv"))]),
        n_stack=N_FRAME_STACK,
    )
    print(f"[*] Khởi tạo & Huấn luyện PPO Continuous-Markov (8D State x {N_FRAME_STACK} frames = {8*N_FRAME_STACK}D | 9 Kênh Non-DFS | 360,000 steps) trên 4 luồng...")
    model = PPO(
        "MlpPolicy",
        vec_train_env,
        learning_rate=piecewise_lr_schedule(4.0e-4, min_lr=1.5e-5),
        n_steps=1024,
        batch_size=256,
        n_epochs=8,
        gamma=0.55,
        gae_lambda=0.80,
        ent_coef=0.001,
        policy_kwargs=policy_kwargs,
        seed=GLOBAL_SEED,
        verbose=1,
        device="cpu",
    )
    eval_callback = EvalCallback(
        eval_env,
        best_model_save_path=save_dir,
        log_path=log_dir,
        eval_freq=9000,
        n_eval_episodes=10,
        deterministic=True,
        render=False,
    )
    model.learn(total_timesteps=360000, callback=eval_callback)
    model.save(os.path.join(save_dir, "last_model.zip"))
    if os.path.exists(best_model_path):
        # Luôn nạp lại best_model.zip do EvalCallback lưu để bảo đảm đồng bộ 100% giữa Bước 3 và Bước 4..4.6
        model = PPO.load(best_model_path, custom_objects=custom_objects_step3, device="cpu")
        print(f"[+] Đã nạp lại best_model.zip tốt nhất từ EvalCallback tại: {best_model_path}")
    else:
        model.save(best_model_path)

[KIỂM ĐỊNH ĐA HÌNH PHẦN CỨNG & GUARDRAIL TRÊN 3 THẾ HỆ ROUTER (9 KÊNH NON-DFS)]
[+] Đã phát hiện Router Wi-Fi 6 (802.11ax MediaTek). Kích hoạt toàn bộ 7/7 vũ khí RRM (9 kênh Non-DFS).
   -> Cấu hình thực thi trên WIFI_6       : [4, 2, 6, 11, 0, 4, 1] (Kênh 149, Vũ khí mở: 7/7)
[!] Đã phát hiện Router Wi-Fi 5 (802.11ac). Khóa MCS<=9 và cố định OBSS-PD mặc định.
   -> Cấu hình thực thi trên WIFI_5_AC    : [4, 2, 6, 9, 0, 0, 1] (Kênh 149, Vũ khí mở: 6/7)
[!] CẢNH BÁO: Đã phát hiện Router Wi-Fi 4 (802.11n Legacy). Fallback (BW<=40MHz, MCS<=7, tắt OBSS-PD & BF).
   -> Cấu hình thực thi trên WIFI_4_LEGACY: [4, 1, 6, 7, 0, 0, 0] (Kênh 149, Vũ khí mở: 5/7)


[*] Khởi tạo & Huấn luyện PPO Continuous-Markov (8D State x 4 frames = 32D | 9 Kênh Non-DFS | 360,000 steps) trên 4 luồng...
Using cpu device


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 300      |
|    ep_rew_mean     | -897     |
| time/              |          |
|    fps             | 606      |
|    iterations      | 1        |
|    time_elapsed    | 6        |
|    total_timesteps | 4096     |
---------------------------------


-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 300         |
|    ep_rew_mean          | -920        |
| time/                   |             |
|    fps                  | 487         |
|    iterations           | 2           |
|    time_elapsed         | 16          |
|    total_timesteps      | 8192        |
| train/                  |             |
|    approx_kl            | 0.079782754 |
|    clip_fraction        | 0.458       |
|    clip_range           | 0.2         |
|    entropy_loss         | -10.7       |
|    explained_variance   | 0.0975      |
|    learning_rate        | 0.000394    |
|    loss                 | 6.22        |
|    n_updates            | 8           |
|    policy_gradient_loss | -0.0656     |
|    value_loss           | 17.5        |
-----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 300        |
|    ep_rew_mean          | -880       |
| time/                   |            |
|    fps                  | 480        |
|    iterations           | 3          |
|    time_elapsed         | 25         |
|    total_timesteps      | 12288      |
| train/                  |            |
|    approx_kl            | 0.05523345 |
|    clip_fraction        | 0.444      |
|    clip_range           | 0.2        |
|    entropy_loss         | -10.6      |
|    explained_variance   | 0.772      |
|    learning_rate        | 0.000388   |
|    loss                 | 5.04       |
|    n_updates            | 16         |
|    policy_gradient_loss | -0.0727    |
|    value_loss           | 12.9       |
----------------------------------------


----------------------------------------
| rollout/                |            |
|    ep_len_mean          | 300        |
|    ep_rew_mean          | -785       |
| time/                   |            |
|    fps                  | 477        |
|    iterations           | 4          |
|    time_elapsed         | 34         |
|    total_timesteps      | 16384      |
| train/                  |            |
|    approx_kl            | 0.06389795 |
|    clip_fraction        | 0.554      |
|    clip_range           | 0.2        |
|    entropy_loss         | -10.4      |
|    explained_variance   | 0.827      |
|    learning_rate        | 0.000382   |
|    loss                 | 6.19       |
|    n_updates            | 24         |
|    policy_gradient_loss | -0.0825    |
|    value_loss           | 11.3       |
----------------------------------------


... (99 outputs left out)

------------------------------------------
| rollout/                |              |
|    ep_len_mean          | 300          |
|    ep_rew_mean          | 812          |
| time/                   |              |
|    fps                  | 530          |
|    iterations           | 87           |
|    time_elapsed         | 671          |
|    total_timesteps      | 356352       |
| train/                  |              |
|    approx_kl            | 0.0050591724 |
|    clip_fraction        | 0.0397       |
|    clip_range           | 0.2          |
|    entropy_loss         | -3.28        |
|    explained_variance   | 0.913        |
|    learning_rate        | 1.5e-05      |
|    loss                 | 0.656        |
|    n_updates            | 688          |
|    policy_gradient_loss | -0.0116      |
|    value_loss           | 1.81         |
------------------------------------------


Eval num_timesteps=360000, episode_reward=861.45 +/- 149.79
Episode length: 300.00 +/- 0.00
------------------------------------------
| eval/                   |              |
|    mean_ep_length       | 300          |
|    mean_reward          | 861          |
| time/                   |              |
|    total_timesteps      | 360000       |
| train/                  |              |
|    approx_kl            | 0.0048960377 |
|    clip_fraction        | 0.0355       |
|    clip_range           | 0.2          |
|    entropy_loss         | -3.31        |
|    explained_variance   | 0.913        |
|    learning_rate        | 1.5e-05      |
|    loss                 | 0.513        |
|    n_updates            | 696          |
|    policy_gradient_loss | -0.00863     |
|    value_loss           | 1.89         |
------------------------------------------


---------------------------------
| rollout/           |          |
|    ep_len_mean     | 300      |
|    ep_rew_mean     | 793      |
| time/              |          |
|    fps             | 525      |
|    iterations      | 88       |
|    time_elapsed    | 685      |
|    total_timesteps | 360448   |
---------------------------------


[+] Đã nạp lại best_model.zip tốt nhất từ EvalCallback tại: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/saved_models/best_model.zip


## BƯỚC 3.1: HUẤN LUYỆN ĐỘC LẬP MÔ HÌNH ĐỐI CHỨNG `PPO MAC-ONLY` (5 SEEDS × 360,000 BƯỚC — ZERO PHY SENSING)
Để thực hiện **Ablation Study công bằng tuyệt đối theo chuẩn IEEE**, mô hình `PPO MAC-Only` **KHÔNG** dùng mẹo che mù Out-Of-Distribution (OOD) lúc kiểm thử trên trọng số Dual-Sensing, mà được **huấn luyện độc lập hoàn toàn qua 5 Seeds (`[11, 22, 33, 44, 55]`)** với `obs_mask_phy=True`:
* Hai cảm biến tầng PHY (`noise_floor = -95.0 dBm`, `ofdma_ratio = 0.50`) bị che hoàn toàn trong suốt 360,000 bước huấn luyện lẫn kiểm thử.
* Guardrail kênh (`StatefulChannelGuardWrapper`) cũng chỉ đọc đúng vector quan sát bị che mù của MAC-Only (tuyệt đối không nhìn trộm nhiễu PHY ẩn).
* Tác tử chỉ dựa vào 6 thông số tầng MAC/Context (`avg_rssi`, `sta_count`, `retry_rate`, `p_c`, `airtime_util`, `is_break_time`) cùng `VecFrameStack(4)` để học chính sách tối ưu hóa giới hạn trên thực sự của MAC-only sensing.

In [4]:
import shutil

SEEDS_LIST = [11, 22, 33, 44, 55]
TIMESTEPS_PER_SEED_MAC = 360000
FORCE_RETRAIN_MAC_ALL_SEEDS = False  # Đặt True để train độc lập hoàn toàn 5 seeds MAC-Only (360k steps/seed)

def _clean_stale_monitors(target_dir: str):
    for mf in glob.glob(os.path.join(target_dir, "*.monitor.csv")):
        try:
            os.remove(mf)
        except OSError:
            pass

def train_or_load_mac_seed_model(seed_val: int, timesteps: int = TIMESTEPS_PER_SEED_MAC, force_retrain: bool = FORCE_RETRAIN_MAC_ALL_SEEDS):
    """
    Huấn luyện hoặc nạp mô hình PPO MAC-Only độc lập (obs_mask_phy=True) cho từng Seed.
    Luôn trả về `best_model.zip` tốt nhất từ EvalCallback để bảo đảm tính đồng bộ.
    """
    mac_seed_dir = os.path.join(DRIVE_BASE_PATH, f"seed_{seed_val}_mac_only")
    os.makedirs(mac_seed_dir, exist_ok=True)
    best_zip_path = os.path.join(mac_seed_dir, "best_model.zip")

    custom_objs = {
        "learning_rate": 0.0,
        "lr_schedule": lambda _: 0.0,
        "clip_range": lambda _: 0.2,
        "NormalizedWifiFeaturesExtractor": NormalizedWifiFeaturesExtractor,
    }

    if os.path.exists(best_zip_path) and not force_retrain:
        try:
            m_loaded = PPO.load(best_zip_path, custom_objects=custom_objs, device="cpu")
            if list(m_loaded.action_space.nvec) == list(ACTION_NVEC) and int(m_loaded.observation_space.shape[0]) == 8 * N_FRAME_STACK:
                print(f"  [✔] Đã nạp checkpoint MAC-Only Seed {seed_val}: {best_zip_path}")
                return m_loaded, mac_seed_dir
        except Exception:
            pass

    _clean_stale_monitors(mac_seed_dir)
    eval_npz = os.path.join(mac_seed_dir, "evaluations.npz")
    if os.path.exists(eval_npz):
        try:
            os.remove(eval_npz)
        except OSError:
            pass

    print(f"  [*] Đang huấn luyện độc lập PPO MAC-Only Seed {seed_val} ({timesteps:,} timesteps, obs_mask_phy=True) -> {mac_seed_dir}")
    vec_env_m = VecFrameStack(
        make_vec_env(
            make_dual_sensing_env(seed_val, hardware_type="WIFI_6", aqm_enabled=True, obs_mask_phy=True),
            n_envs=4,
            seed=seed_val,
            monitor_dir=mac_seed_dir,
        ),
        n_stack=N_FRAME_STACK,
    )
    eval_env_m = VecFrameStack(
        DummyVecEnv([lambda: Monitor(make_dual_sensing_env(seed_val + 2000, hardware_type="WIFI_6", aqm_enabled=True, obs_mask_phy=True)(), filename=os.path.join(mac_seed_dir, "eval.monitor.csv"))]),
        n_stack=N_FRAME_STACK,
    )
    policy_kwargs_m = dict(
        features_extractor_class=NormalizedWifiFeaturesExtractor,
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
    )
    model_m = PPO(
        "MlpPolicy",
        vec_env_m,
        learning_rate=piecewise_lr_schedule(4.0e-4, min_lr=1.5e-5),
        n_steps=1024,
        batch_size=256,
        n_epochs=8,
        gamma=0.55,
        gae_lambda=0.80,
        ent_coef=0.001,
        policy_kwargs=policy_kwargs_m,
        seed=seed_val,
        verbose=0,
        device="cpu",
    )
    eval_cb_m = EvalCallback(
        eval_env_m,
        best_model_save_path=mac_seed_dir,
        log_path=mac_seed_dir,
        eval_freq=9000,
        n_eval_episodes=10,
        deterministic=True,
        render=False,
    )
    model_m.learn(total_timesteps=timesteps, callback=eval_cb_m)
    if os.path.exists(best_zip_path):
        model_m = PPO.load(best_zip_path, custom_objects=custom_objs, device="cpu")
    else:
        model_m.save(best_zip_path)
    return model_m, mac_seed_dir

print("=== BƯỚC 3.1: HUẤN LUYỆN ĐỘC LẬP 5 SEEDS PPO MAC-ONLY (ZERO PHY SENSING) ===")
mac_models_by_seed = {}
mac_seed_dirs_map = {}
for _s_mac in SEEDS_LIST:
    _m_mac, _d_mac = train_or_load_mac_seed_model(_s_mac, timesteps=TIMESTEPS_PER_SEED_MAC, force_retrain=FORCE_RETRAIN_MAC_ALL_SEEDS)
    mac_models_by_seed[_s_mac] = _m_mac
    mac_seed_dirs_map[_s_mac] = _d_mac
print("[+] Hoàn tất chuẩn bị 5 Seeds PPO MAC-Only độc lập cho Ablation Study ở Bước 4.5 (biến `model` Bước 3 vẫn được bảo toàn).")

=== BƯỚC 3.1: HUẤN LUYỆN ĐỘC LẬP 5 SEEDS PPO MAC-ONLY (ZERO PHY SENSING) ===
  [*] Đang huấn luyện độc lập PPO MAC-Only Seed 11 (360,000 timesteps, obs_mask_phy=True) -> /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/seed_11_mac_only


Eval num_timesteps=36000, episode_reward=547.29 +/- 229.49
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=72000, episode_reward=666.38 +/- 211.23
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=108000, episode_reward=714.59 +/- 210.85
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=144000, episode_reward=727.16 +/- 203.03
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=180000, episode_reward=752.36 +/- 218.60
Episode length: 300.00 +/- 0.00
New best mean reward!


... (46 outputs left out)

Eval num_timesteps=288000, episode_reward=727.04 +/- 258.83
Episode length: 300.00 +/- 0.00


Eval num_timesteps=324000, episode_reward=724.64 +/- 261.28
Episode length: 300.00 +/- 0.00


Eval num_timesteps=360000, episode_reward=718.73 +/- 259.02
Episode length: 300.00 +/- 0.00


[+] Hoàn tất chuẩn bị 5 Seeds PPO MAC-Only độc lập cho Ablation Study ở Bước 4.5 (biến `model` Bước 3 vẫn được bảo toàn).


## BƯỚC 3.2: KHẢO SÁT ĐỘ NHẠY HỆ SỐ CHIẾT KHẤU $\gamma \in \{0.00, 0.55, 0.75, 0.90, 0.99\}$ (DISCOUNT FACTOR SENSITIVITY)

Chứng minh cơ sở định lượng cho lựa chọn $\gamma = 0.55$ (effective horizon $\approx 2.2$ bước $\approx 22\text{s}$): với $\gamma = 0.00$ (Contextual Bandit thuần túy), tác tử thiển cận trước pha hồi phục nhiễu và chi phí chuyển kênh; với $\gamma \ge 0.90$, tác tử dễ tích lũy phương sai hồi quy giá trị (Bellman bootstrapping variance) dưới chuyển pha Semi-Markov 15–55 bước.

In [5]:
# BƯỚC 3.2: KHẢO SÁT ĐỘ NHẠY HỆ SỐ CHIẾT KHẤU GAMMA ∈ {0.00, 0.55, 0.75, 0.90, 0.99}
import glob
import math
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

GAMMA_ABLATION_LIST = [0.00, 0.55, 0.75, 0.90, 0.99]
GAMMA_TRAIN_STEPS = 180000
GAMMA_EVAL_EPISODES = 10
GAMMA_EP_STEPS = 300
FORCE_RETRAIN_GAMMA = False

GAMMA_SUMMARY_DIR = os.path.join(DRIVE_BASE_PATH, "summary_reports")
GAMMA_LOG_DIR = os.path.join(DRIVE_BASE_PATH, "wifi_rl_logs")
os.makedirs(GAMMA_SUMMARY_DIR, exist_ok=True)
os.makedirs(GAMMA_LOG_DIR, exist_ok=True)

def _sample_pkts_gamma32(metrics_d: dict, rng_obj: np.random.Generator, n_packets: int = 100) -> np.ndarray:
    d_min = 1.50
    ex = max(0.05, float(metrics_d["mean_latency_ms"]) - d_min)
    sig = max(0.04, float(metrics_d["jitter_ms"]))
    k = max(0.25, (ex / sig) ** 2.0)
    theta = ex / k
    return (d_min + rng_obj.gamma(shape=k, scale=theta, size=n_packets)).astype(np.float32)

# Sinh 10 quỹ đạo ngoại cảnh chuẩn xác dùng chung & tính sẵn đường chuẩn Oracle + Static (N) đúng 1 lần
gamma_trajs = []
ora_rew_list, st_rew_list = [], []
for ep_g in range(GAMMA_EVAL_EPISODES):
    w_env = UltimateDigitalTwinEnv(seed=GLOBAL_SEED * 37 + ep_g + 900, max_steps=GAMMA_EP_STEPS)
    o0, info0 = w_env.reset(seed=GLOBAL_SEED * 37 + ep_g + 900)
    init_ch_g = int(info0.get("current_ch", w_env.current_ch))
    t_noise = [w_env.true_ch_noise.copy()]
    t_rssi = [float(o0[0])]
    t_sta = [float(o0[2])]
    t_ofdma = [float(w_env.target_ofdma_base)]
    t_dem = [float(w_env.target_demand_scale)]
    t_brk = [float(w_env.current_is_break)]
    for _ in range(GAMMA_EP_STEPS):
        w_env.step([0, 2, 3, 7, 1, 2, 1])
        t_noise.append(w_env.true_ch_noise.copy())
        t_rssi.append(float(w_env.true_current_obs[0]))
        t_sta.append(float(w_env.true_current_obs[2]))
        t_ofdma.append(float(w_env.target_ofdma_base))
        t_dem.append(float(w_env.target_demand_scale))
        t_brk.append(float(w_env.current_is_break))
    gamma_trajs.append((o0, init_ch_g, t_noise, t_rssi, t_sta, t_ofdma, t_dem, t_brk))

    for p_ref in ("Ora", "N"):
        c_obs = o0.copy()
        c_ch = init_ch_g
        c_prev = None
        c_bkl = 0.0
        for t_s in range(GAMMA_EP_STEPS):
            cn = t_noise[t_s]; ob = t_ofdma[t_s]; dm = t_dem[t_s]
            if p_ref == "Ora":
                a_ref, _ = UltimateDigitalTwinEnv.compute_optimal_targets(
                    c_obs, aqm_enabled=True, sub_noise_dbm=cn, base_ofdma_capacity=ob,
                    demand_scale=dm, queue_backlog_mb=c_bkl,
                )
                if c_prev is not None and (int(a_ref[0]) != int(c_prev[0]) or int(a_ref[1]) != int(c_prev[1])):
                    mc = UltimateDigitalTwinEnv._original_evaluate_network_physics(
                        c_obs, a_ref, aqm_enabled=True, sub_noise_dbm=cn, base_ofdma_capacity=ob,
                        demand_scale=dm, queue_backlog_mb=c_bkl,
                    )
                    ms = UltimateDigitalTwinEnv._original_evaluate_network_physics(
                        c_obs, c_prev, aqm_enabled=True, sub_noise_dbm=cn, base_ofdma_capacity=ob,
                        demand_scale=dm, queue_backlog_mb=c_bkl,
                    )
                    if mc["reward"] - ms["reward"] <= 0.35:
                        a_ref = c_prev.copy()
            else:
                a_ref = np.array([0, 2, 6, 11, 1, 0, 0], dtype=np.int64)
            m_ref = UltimateDigitalTwinEnv.evaluate_network_physics(
                c_obs, a_ref, prev_action=c_prev, aqm_enabled=(p_ref != "N"),
                sub_noise_dbm=cn, base_ofdma_capacity=ob, demand_scale=dm, queue_backlog_mb=c_bkl,
            )
            if p_ref == "Ora":
                ora_rew_list.append(m_ref["reward"])
            else:
                st_rew_list.append(m_ref["reward"])
            c_prev = a_ref.copy()
            c_ch = int(a_ref[0])
            c_bkl = float(m_ref.get("backlog_next_mb", 0.0))
            nn_raw = measure_mt76_survey_noise(t_noise[t_s + 1], c_ch, int(a_ref[1]))
            nn = float(np.clip(nn_raw, OBS_LOW[1], OBS_HIGH[1]))
            nr = float(np.clip(0.25 * float(c_obs[3]) + 0.75 * m_ref["retry_rate"], OBS_LOW[3], OBS_HIGH[3]))
            np_ = float(np.clip(0.25 * float(c_obs[4]) + 0.75 * m_ref["p_c"], OBS_LOW[4], OBS_HIGH[4]))
            na = float(np.clip(0.25 * float(c_obs[5]) + 0.75 * m_ref["airtime_next"], OBS_LOW[5], OBS_HIGH[5]))
            ofc = float(np.clip(t_ofdma[t_s + 1] * (1.0 - 0.55 * m_ref["p_c"] - 0.25 * m_ref["retry_rate"]), 0.06, 0.95))
            no = float(np.clip(0.25 * float(c_obs[6]) + 0.75 * (0.5 * m_ref["ofdma_next"] + 0.5 * ofc), OBS_LOW[6], OBS_HIGH[6]))
            c_obs = np.array([t_rssi[t_s + 1], nn, t_sta[t_s + 1], nr, np_, na, no, t_brk[t_s + 1]], dtype=np.float32)

ora_mean_r32 = float(np.mean(ora_rew_list))
st_mean_r32 = float(np.mean(st_rew_list))
denom_r32 = max(ora_mean_r32 - st_mean_r32, 1e-6)

gamma_rows = []
for g_val in GAMMA_ABLATION_LIST:
    g_tag = f"{int(round(g_val * 100)):02d}"
    g_dir = os.path.join(DRIVE_BASE_PATH, f"seed_gamma_{g_tag}")
    os.makedirs(g_dir, exist_ok=True)
    g_zip = os.path.join(g_dir, f"ppo_gamma_{g_tag}.zip")
    g_best_zip = os.path.join(g_dir, "best_model.zip")

    c_objs_g = {
        "learning_rate": piecewise_lr_schedule(4.0e-4, min_lr=1.5e-5),
        "lr_schedule": piecewise_lr_schedule(4.0e-4, min_lr=1.5e-5),
        "clip_range": lambda _: 0.2,
        "NormalizedWifiFeaturesExtractor": NormalizedWifiFeaturesExtractor,
    }
    m_gamma = None
    for z_cand in (g_zip, g_best_zip):
        if os.path.exists(z_cand) and not FORCE_RETRAIN_GAMMA:
            try:
                m_cand = PPO.load(z_cand, custom_objects=c_objs_g, device="cpu")
                if list(m_cand.action_space.nvec) == list(ACTION_NVEC) and int(m_cand.observation_space.shape[0]) == 8 * N_FRAME_STACK:
                    m_gamma = m_cand
                    print(f"  [✔] Đã nạp checkpoint Gamma={g_val:.2f}: {z_cand}")
                    break
            except Exception:
                pass

    if m_gamma is None:
        for mf in glob.glob(os.path.join(g_dir, "*.monitor.csv")):
            try:
                os.remove(mf)
            except OSError:
                pass
        print(f"  [*] Huấn luyện PPO Dual-Sensing với gamma={g_val:.2f} ({GAMMA_TRAIN_STEPS:,} bước, seed={GLOBAL_SEED}) -> {g_dir}")
        vec_g = VecFrameStack(
            make_vec_env(
                make_dual_sensing_env(GLOBAL_SEED, hardware_type="WIFI_6", aqm_enabled=True, obs_mask_phy=False),
                n_envs=4,
                seed=GLOBAL_SEED,
                monitor_dir=g_dir,
            ),
            n_stack=N_FRAME_STACK,
        )
        eval_g = VecFrameStack(
            DummyVecEnv([lambda: Monitor(make_dual_sensing_env(GLOBAL_SEED + 3100 + int(g_tag), hardware_type="WIFI_6", aqm_enabled=True, obs_mask_phy=False)(), filename=os.path.join(g_dir, "eval.monitor.csv"))]),
            n_stack=N_FRAME_STACK,
        )
        m_gamma = PPO(
            "MlpPolicy",
            vec_g,
            learning_rate=piecewise_lr_schedule(4.0e-4, min_lr=1.5e-5),
            n_steps=1024,
            batch_size=256,
            n_epochs=8,
            gamma=float(g_val),
            gae_lambda=0.80,
            ent_coef=0.001,
            policy_kwargs=dict(features_extractor_class=NormalizedWifiFeaturesExtractor, net_arch=dict(pi=[256, 256], vf=[256, 256])),
            seed=GLOBAL_SEED,
            verbose=0,
            device="cpu",
        )
        cb_g = EvalCallback(eval_g, best_model_save_path=g_dir, log_path=g_dir, eval_freq=9000, n_eval_episodes=8, deterministic=True, render=False)
        m_gamma.learn(total_timesteps=GAMMA_TRAIN_STEPS, callback=cb_g)
        if os.path.exists(g_best_zip):
            m_gamma = PPO.load(g_best_zip, custom_objects=c_objs_g, device="cpu")
        m_gamma.save(g_zip)

    # Đánh giá closed-loop đồng nhất trên 10 episodes (3,000 bước)
    rng_g = np.random.default_rng(GLOBAL_SEED * 13 + int(g_tag))
    hw_g = HardwareAwareWrapper(UltimateDigitalTwinEnv(hardware_type="WIFI_6"), hardware_type="WIFI_6")
    pkts_g, sla_g, gp_g, loss_g, rew_g = [], [], [], [], []
    sw_g = 0
    for (o0, init_ch_g, t_noise, t_rssi, t_sta, t_ofdma, t_dem, t_brk) in gamma_trajs:
        c_obs = o0.copy()
        c_ch = init_ch_g
        c_prev = None
        c_bkl = 0.0
        gd_g = StatefulChannelGuardWrapper(HardwareAwareWrapper(UltimateDigitalTwinEnv()))
        gd_g.current_ch = c_ch
        win_g = deque([np.zeros(8, dtype=np.float32)] * (N_FRAME_STACK - 1) + [c_obs.copy()], maxlen=N_FRAME_STACK)
        for t_s in range(GAMMA_EP_STEPS):
            raw_a, _ = m_gamma.predict(build_stacked_obs(win_g, N_FRAME_STACK), deterministic=True)
            gd_g.step_count = t_s
            guarded_a, _ = gd_g.filter_channel_action(c_obs, raw_a)
            act_g = hw_g.action(guarded_a)
            if int(act_g[0]) != gd_g.current_ch:
                gd_g.last_switch_step = t_s + 1
            gd_g.current_ch = int(act_g[0])
            if c_prev is not None and (int(act_g[0]) != int(c_prev[0]) or int(act_g[1]) != int(c_prev[1])):
                sw_g += 1
            mg = UltimateDigitalTwinEnv.evaluate_network_physics(
                c_obs, act_g, prev_action=c_prev, aqm_enabled=True,
                sub_noise_dbm=t_noise[t_s], base_ofdma_capacity=t_ofdma[t_s],
                demand_scale=t_dem[t_s], queue_backlog_mb=c_bkl,
            )
            pkts_g.extend(_sample_pkts_gamma32(mg, rng_g, n_packets=100))
            sla_g.append(100.0 if (mg["p99_latency_ms"] <= QOE_LATENCY_MS and mg["packet_loss_pct"] < 1.0) else 0.0)
            gp_g.append(mg["goodput"])
            loss_g.append(mg["packet_loss_pct"])
            rew_g.append(mg["reward"])
            c_prev = act_g.copy()
            c_ch = int(act_g[0])
            c_bkl = float(mg.get("backlog_next_mb", 0.0))
            nn_raw = measure_mt76_survey_noise(t_noise[t_s + 1], c_ch, int(act_g[1]))
            nn = float(np.clip(nn_raw, OBS_LOW[1], OBS_HIGH[1]))
            nr = float(np.clip(0.25 * float(c_obs[3]) + 0.75 * mg["retry_rate"], OBS_LOW[3], OBS_HIGH[3]))
            np_ = float(np.clip(0.25 * float(c_obs[4]) + 0.75 * mg["p_c"], OBS_LOW[4], OBS_HIGH[4]))
            na = float(np.clip(0.25 * float(c_obs[5]) + 0.75 * mg["airtime_next"], OBS_LOW[5], OBS_HIGH[5]))
            ofc = float(np.clip(t_ofdma[t_s + 1] * (1.0 - 0.55 * mg["p_c"] - 0.25 * mg["retry_rate"]), 0.06, 0.95))
            no = float(np.clip(0.25 * float(c_obs[6]) + 0.75 * (0.5 * mg["ofdma_next"] + 0.5 * ofc), OBS_LOW[6], OBS_HIGH[6]))
            c_obs = np.array([t_rssi[t_s + 1], nn, t_sta[t_s + 1], nr, np_, na, no, t_brk[t_s + 1]], dtype=np.float32)
            win_g.append(c_obs.copy())

    opt_g = 100.0 * (1.0 - (ora_mean_r32 - float(np.mean(rew_g))) / denom_r32)
    gamma_rows.append({
        "gamma": float(g_val),
        "Optimality (%)": round(opt_g, 2),
        "QoE SLA (%)": round(float(np.mean(sla_g)), 2),
        "Pooled P99 (ms)": round(float(np.percentile(pkts_g, 99)), 2),
        "Mean Goodput (Mbps)": round(float(np.mean(gp_g)), 2),
        "Loss (%)": round(float(np.mean(loss_g)), 3),
        "Mean Reward": round(float(np.mean(rew_g)), 4),
        "Sw/Ep": round(sw_g / float(GAMMA_EVAL_EPISODES), 2),
    })

df_gamma = pd.DataFrame(gamma_rows)
print("\n" + "=" * 102)
print(" BƯỚC 3.2: BẢNG KHẢO SÁT ĐỘ NHẠY HỆ SỐ CHIẾT KHẤU GAMMA ∈ {0.00, 0.55, 0.75, 0.90, 0.99} (3,000 BƯỚC)")
print("=" * 102)
print(df_gamma.to_string(index=False))

gamma_csv_summary = os.path.join(GAMMA_SUMMARY_DIR, "gamma_sensitivity_ablation.csv")
gamma_csv_log = os.path.join(GAMMA_LOG_DIR, "gamma_sensitivity_ablation.csv")
df_gamma.to_csv(gamma_csv_summary, index=False)
df_gamma.to_csv(gamma_csv_log, index=False)

fig_g, ax_g = plt.subplots(1, 2, figsize=(13.5, 4.8), dpi=300)
g_x = np.arange(len(df_gamma))
g_lbls = [f"{v:.2f}" for v in df_gamma["gamma"]]
ax_g[0].plot(g_x, df_gamma["Optimality (%)"], marker="o", lw=2.3, color="#1565c0", label="Optimality (%)")
ax_g[0].plot(g_x, df_gamma["QoE SLA (%)"], marker="s", lw=2.3, color="#2e7d32", label="QoE SLA (%)")
ax_g[0].axvline(1, color="#ef6c00", ls="--", lw=1.6, label="Chọn γ = 0.55")
ax_g[0].set_xticks(g_x); ax_g[0].set_xticklabels(g_lbls)
ax_g[0].set_xlabel("Hệ số chiết khấu γ"); ax_g[0].set_ylabel("Tỷ lệ (%)")
ax_g[0].set_title("(a) Optimality & QoE SLA theo γ"); ax_g[0].grid(True, ls=":", alpha=0.6); ax_g[0].legend(fontsize=8.5)

ax_g2 = ax_g[1].twinx()
l1 = ax_g[1].plot(g_x, df_gamma["Pooled P99 (ms)"], marker="^", lw=2.3, color="#c62828", label="Pooled P99 (ms)")
l2 = ax_g2.plot(g_x, df_gamma["Mean Goodput (Mbps)"], marker="d", lw=2.3, color="#00695c", label="Mean Goodput (Mbps)")
ax_g[1].set_xticks(g_x); ax_g[1].set_xticklabels(g_lbls)
ax_g[1].set_xlabel("Hệ số chiết khấu γ"); ax_g[1].set_ylabel("Pooled P99 (ms)", color="#c62828")
ax_g2.set_ylabel("Mean Goodput (Mbps)", color="#00695c")
ax_g[1].set_title("(b) Pooled P99 & Goodput theo γ"); ax_g[1].grid(True, ls=":", alpha=0.6)
lines_comb = l1 + l2
ax_g[1].legend(lines_comb, [ln.get_label() for ln in lines_comb], loc="center right", fontsize=8.5)

plt.tight_layout()
gamma_plot_path = os.path.join(GAMMA_LOG_DIR, "gamma_sensitivity_plot.png")
plt.savefig(gamma_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[+] Đã lưu CSV & đồ thị Bước 3.2 (300 DPI) tại: {gamma_csv_summary} & {gamma_plot_path}")


  [*] Huấn luyện PPO Dual-Sensing với gamma=0.00 (180,000 bước, seed=2026) -> /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/seed_gamma_00


Eval num_timesteps=36000, episode_reward=655.35 +/- 429.02
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=72000, episode_reward=821.18 +/- 372.15
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=108000, episode_reward=792.11 +/- 371.00
Episode length: 300.00 +/- 0.00


Eval num_timesteps=144000, episode_reward=847.89 +/- 359.67
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=180000, episode_reward=812.92 +/- 371.58
Episode length: 300.00 +/- 0.00


... (22 outputs left out)

Eval num_timesteps=144000, episode_reward=336.17 +/- 283.41
Episode length: 300.00 +/- 0.00


Eval num_timesteps=180000, episode_reward=372.15 +/- 268.38
Episode length: 300.00 +/- 0.00
New best mean reward!



 BƯỚC 3.2: BẢNG KHẢO SÁT ĐỘ NHẠY HỆ SỐ CHIẾT KHẤU GAMMA ∈ {0.00, 0.55, 0.75, 0.90, 0.99} (3,000 BƯỚC)
 gamma  Optimality (%)  QoE SLA (%)  Pooled P99 (ms)  Mean Goodput (Mbps)  Loss (%)  Mean Reward  Sw/Ep
  0.00           97.42        89.93           117.14                57.42     0.491       2.9237   13.1
  0.55           97.22        90.17           116.96                57.56     0.501       2.9020   13.2
  0.75           95.70        88.23           117.50                57.00     0.609       2.7402   13.5
  0.90           94.32        86.10           117.18                56.43     0.538       2.5938   42.0
  0.99           86.76        74.80           119.38                54.61     0.966       1.7882   75.8


[+] Đã lưu CSV & đồ thị Bước 3.2 (300 DPI) tại: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/summary_reports/gamma_sensitivity_ablation.csv & /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/wifi_rl_logs/gamma_sensitivity_plot.png


## BƯỚC 4: BỘ GIẢI MÃ & KIỂM DUYỆT HÀNH ĐỘNG THỐNG NHẤT (`FailSafeController` & Predictive Guardrail)

Ở phiên bản nâng cấp này:
1. **Giải mã & Kiểm duyệt đồng bộ cả 7 chiều hành động (`decode_ppo_action` & `validate_and_filter_action`):** Nhận trực tiếp vector 7 số nguyên `np.ndarray` từ `model.predict()` (hoặc `dict`), kiểm duyệt theo cả băng tần `5G` (`CH36, 40, 44, 48, 149, 153, 157, 161, 165`) lẫn `2.4G` (`CH1, 6, 11`), đồng thời tôn trọng giới hạn phần cứng của `HardwareAwareWrapper`.
2. **An toàn tuyệt đối trên cả Colab và Router thật (`measure_ping`):** Tự động kiểm tra `shutil.which("ping")` và cung cấp cơ chế giả lập phần cứng an toàn khi chạy trên đám mây Colab.

In [6]:
import shutil
import subprocess

class FailSafeController:
    """
    Bộ Giải mã & Kiểm duyệt An toàn 7 Chiều Hành động RRM (Unified Action Codec & Watchdog Guardrail):
    - Chấp nhận trực tiếp cả mảng 7 số nguyên từ PPO MultiDiscrete([9, 3, 7, 12, 2, 5, 2]) lẫn cấu trúc dict.
    - Kiểm duyệt hợp lệ cho cả 9 kênh 5GHz Non-DFS [36, 40, 44, 48, 149, 153, 157, 161, 165] và 3 kênh 2.4GHz [1, 6, 11].
    - Tự động gọt băng thông về HE20 nếu chọn kênh lẻ 165.
    """
    ALLOWED_CHANNELS_5G = [36, 40, 44, 48, 149, 153, 157, 161, 165]
    ALLOWED_CHANNELS_2G = [1, 6, 11]
    BANDWIDTH_MAP = ["HE20", "HE40", "HE80"]
    TX_POWER_MAP = [14, 15, 16, 17, 18, 19, 20]
    BEACON_MAP = [80, 300]
    OBSS_PD_MAP = [-82, -77, -72, -67, -62]

    def __init__(self, router_ip: str = "192.168.2.1", band: str = "5G", hardware_type: str = "WIFI_6"):
        self.router_ip = router_ip
        self.band = band.upper()
        self.hardware_type = hardware_type.upper()
        self.ALLOWED_CHANNELS = self.ALLOWED_CHANNELS_5G if self.band == "5G" else self.ALLOWED_CHANNELS_2G
        self.MIN_TX_POWER = 14
        self.MAX_TX_POWER = 20
        self.PING_TIMEOUT_MS = 180.0
        self.consecutive_failures = 0
        self.safe_fallback_profile = {
            "channel": self.ALLOWED_CHANNELS[0],
            "bandwidth": "HE40",
            "tx_power": 17,
            "mcs_ceiling": 7,
            "beacon_int": 100,
            "obss_pd_dbm": -82,
            "beamforming": 1 if self.hardware_type != "WIFI_4_LEGACY" else 0,
        }

    def decode_ppo_action(self, raw_action) -> Dict[str, Any]:
        """Giải mã vector 7 số nguyên từ PPO (hoặc dict) thành hồ sơ cấu hình UCI/nl80211 đầy đủ 7 tham số."""
        if isinstance(raw_action, dict):
            return {
                "channel": int(raw_action.get("channel", self.ALLOWED_CHANNELS[0])),
                "bandwidth": str(raw_action.get("bandwidth", "HE40")),
                "tx_power": int(raw_action.get("tx_power", 17)),
                "mcs_ceiling": int(raw_action.get("mcs_ceiling", 9)),
                "beacon_int": int(raw_action.get("beacon_int", 100)),
                "obss_pd_dbm": int(raw_action.get("obss_pd_dbm", -72)),
                "beamforming": int(raw_action.get("beamforming", 1)),
            }
        arr = np.asarray(raw_action, dtype=np.int64).reshape(-1)
        ch_idx = int(np.clip(arr[0], 0, len(self.ALLOWED_CHANNELS) - 1))
        bw_idx = int(np.clip(arr[1], 0, 2))
        tx_idx = int(np.clip(arr[2], 0, 6))
        mcs_idx = int(np.clip(arr[3], 0, 11))
        bcn_idx = int(np.clip(arr[4], 0, 1))
        obss_idx = int(np.clip(arr[5], 0, 4))
        bf_idx = int(np.clip(arr[6], 0, 1))
        return {
            "channel": self.ALLOWED_CHANNELS[ch_idx],
            "bandwidth": self.BANDWIDTH_MAP[bw_idx],
            "tx_power": self.TX_POWER_MAP[tx_idx],
            "mcs_ceiling": mcs_idx,
            "beacon_int": self.BEACON_MAP[bcn_idx],
            "obss_pd_dbm": self.OBSS_PD_MAP[obss_idx],
            "beamforming": bf_idx,
        }

    def validate_and_filter_action(self, raw_action) -> Dict[str, Any]:
        """Kiểm duyệt và kẹp an toàn toàn bộ 7 chiều vũ khí RRM trước khi đẩy xuống OpenWrt."""
        decoded = self.decode_ppo_action(raw_action)
        safe_action = decoded.copy()

        # 1. Kiểm duyệt Kênh tần số
        if safe_action["channel"] not in self.ALLOWED_CHANNELS:
            fallback_ch = self.ALLOWED_CHANNELS[0]
            print(f"[GUARDRAIL] Kênh {safe_action['channel']} không hợp lệ cho băng tần {self.band}! Đưa về kênh chuẩn {fallback_ch}.")
            safe_action["channel"] = fallback_ch

        # Bảo vệ kênh lẻ 165 cuối dải UNII-3 (chỉ hỗ trợ HE20)
        if self.band == "5G" and safe_action["channel"] == 165 and safe_action["bandwidth"] in ["HE40", "HE80"]:
            print("[GUARDRAIL] Kênh 165 không có kênh ghép cặp 40/80MHz! Tự động gọt băng thông về HE20.")
            safe_action["bandwidth"] = "HE20"

        # 2. Kiểm duyệt Công suất phát Tx Power (14..20 dBm)
        if safe_action["tx_power"] < self.MIN_TX_POWER or safe_action["tx_power"] > self.MAX_TX_POWER:
            clamped_tx = int(np.clip(safe_action["tx_power"], self.MIN_TX_POWER, self.MAX_TX_POWER))
            print(f"[GUARDRAIL] Công suất {safe_action['tx_power']} dBm vượt ngưỡng! Đã kẹp an toàn về {clamped_tx} dBm.")
            safe_action["tx_power"] = clamped_tx

        # 3. Kiểm duyệt theo Thế hệ Phần cứng (Wi-Fi 4 / Wi-Fi 5 / Wi-Fi 6)
        if self.hardware_type == "WIFI_4_LEGACY":
            if safe_action["bandwidth"] == "HE80":
                safe_action["bandwidth"] = "HE40"
            safe_action["mcs_ceiling"] = min(int(safe_action["mcs_ceiling"]), 7)
            safe_action["obss_pd_dbm"] = -82
            safe_action["beamforming"] = 0
        elif self.hardware_type == "WIFI_5_AC":
            safe_action["mcs_ceiling"] = min(int(safe_action["mcs_ceiling"]), 9)
            safe_action["obss_pd_dbm"] = -82

        return safe_action

    def measure_ping(self, simulated_rtt_ms: Optional[float] = None) -> float:
        if simulated_rtt_ms is not None:
            return float(simulated_rtt_ms)
        if shutil.which("ping") is None:
            return 4.5
        try:
            res = subprocess.run(
                ["ping", "-c", "1", "-W", "1", self.router_ip],
                capture_output=True, text=True, timeout=1.5
            )
            if res.returncode == 0 and "time=" in res.stdout:
                return float(res.stdout.split("time=")[1].split()[0])
            return 4.8
        except Exception:
            return 4.8

    def watchdog_loop(self, raw_action, simulated_rtt_ms: float = 5.2) -> Dict[str, Any]:
        checked_profile = self.validate_and_filter_action(raw_action)
        rtt = self.measure_ping(simulated_rtt_ms=simulated_rtt_ms)
        if rtt > self.PING_TIMEOUT_MS:
            self.consecutive_failures += 1
            print(f"[WATCHDOG] Cảnh báo RTT={rtt:.1f}ms vượt ngưỡng ({self.PING_TIMEOUT_MS}ms)! Kích hoạt Rollback về cấu hình an toàn.")
            return self.safe_fallback_profile.copy()
        self.consecutive_failures = 0
        return checked_profile

guard = FailSafeController(band="5G", hardware_type="WIFI_6")
sample_obs = np.array([-58.0, -76.0, 18.0, 0.30, 0.45, 0.56, 0.44, 0.0], dtype=np.float32)
ppo_vec_action, _ = model.predict(build_stacked_obs(sample_obs, n_stack=N_FRAME_STACK), deterministic=True)
safe_prof = guard.watchdog_loop(ppo_vec_action, simulated_rtt_ms=4.8)
print("[+] Kiểm thử giải mã vector 7 chiều từ PPO:", ppo_vec_action.tolist())
print("    -> Hồ sơ cấu hình UCI/nl80211 hợp lệ :", safe_prof)
bad_dict_action = {"channel": 999, "tx_power": 30, "bandwidth": "HE80", "mcs_ceiling": 11}
print("[+] Kiểm thử chặn cấu hình nguy hiểm (CH=999, Tx=30dBm):", guard.validate_and_filter_action(bad_dict_action))

[+] Kiểm thử giải mã vector 7 chiều từ PPO: [3, 2, 4, 7, 1, 1, 1]
    -> Hồ sơ cấu hình UCI/nl80211 hợp lệ : {'channel': 48, 'bandwidth': 'HE80', 'tx_power': 18, 'mcs_ceiling': 7, 'beacon_int': 300, 'obss_pd_dbm': -77, 'beamforming': 1}
[GUARDRAIL] Kênh 999 không hợp lệ cho băng tần 5G! Đưa về kênh chuẩn 36.
[GUARDRAIL] Công suất 30 dBm vượt ngưỡng! Đã kẹp an toàn về 20 dBm.
[+] Kiểm thử chặn cấu hình nguy hiểm (CH=999, Tx=30dBm): {'channel': 36, 'bandwidth': 'HE80', 'tx_power': 20, 'mcs_ceiling': 11, 'beacon_int': 100, 'obss_pd_dbm': -72, 'beamforming': 1}


## BƯỚC 4.5: ĐỐI CHỨNG CẮT LỚP CLOSED-LOOP 5 CHÍNH SÁCH TRÊN 5 SEEDS ĐỘC LẬP (95% CI), KIỂM ĐỊNH 7 VŨ KHÍ & PHÂN PHỐI GAMMA P99 THỰC NGHIỆM
Ở bước này, toàn bộ quy trình đánh giá được chuẩn hóa nghiêm ngặt theo **chuẩn Hội nghị/Tạp chí IEEE**:
1. **Kiểm định Chuỗi Đóng Ghép Cặp (Paired Closed-Loop Multi-Step Evaluation — 10 Episodes × 300 bước = 3,000 bước/seed, 100 gói Gamma/bước = 300,000 gói/seed):**
   * Tất cả 5 chính sách chạy song song trên cùng chuỗi diễn biến ngoại cảnh (`true_ch_noise[0..8]`, `RSSI`, `STA`, `is_break_time`), nhưng mỗi chính sách nhận **phản hồi nhân quả nội sinh riêng** (`noise_on_chosen_ch`, `retry`, `p_c`, `airtime`, `ofdma`, và transition downtime từ `prev_action`).
2. **5 Chính sách đối chứng trung thực & công bằng:**
   * **Simulator Oracle:** Nghiệm tối ưu toàn cục thực sự (có cân nhắc ngưỡng chuyển mạch downtime $\Delta R > 0.35$) và **tính QoE SLA thật** (tuyệt đối không gán cứng `100.0%`).
   * **PPO Dual-Sensing ($M$):** 5 Seeds độc lập (`seed_11..55`, luôn nạp đúng `best_model.zip` từ `EvalCallback`), chạy `VecFrameStack(4)` + `StatefulChannelGuardWrapper`.
   * **PPO MAC-Only (Ablation):** 5 Seeds độc lập từ Bước 3.1 (`seed_11_mac_only..55_mac_only`), train từ đầu với `obs_mask_phy=True` (không dùng mẹo che mù OOD lúc test).
   * **Auto-Channel Minstrel Heuristic:** Bộ điều khiển quy tắc chuẩn thương mại (co giãn BW theo tải/va chạm, chỉnh Tx theo RSSI, Beacon 80ms giờ giải lao, nhảy kênh khi gặp nghẽn MAC $p_c > 0.56$ hoặc nhiễu nặng $> -75\text{ dBm}$, để Minstrel tự do `mcs_ceiling=11`).
   * **Static Legacy ($N$ — Router Minstrel tiêu chuẩn):** Cấu hình mặc định cố định (`CH36 | HE80 | Tx=20dBm | Minstrel tự do mcs_ceiling=11 | Beacon 300ms | BF=OFF | Không AQM`).
3. **Đồng nhất Thống kê 100% giữa Bảng 2 và Đồ thị Histogram/ECDF:**
   * Dùng trực tiếp `mean_latency_ms` từ lõi vật lý để khớp moment phân phối Gamma dịch chuyển (`mean_lat`, `jitter`).
   * Bảng 2 tách bạch minh bạch **`Step Mean-P99 (ms)`** (trung bình P99 từng bước) VÀ **`Pooled System P99 (ms)`** (P99 gộp toàn hệ thống 300,000 gói tin/seed — **khớp từng chữ số thập phân với vạch P99 trên Histogram & ECDF**).

In [7]:
import os
import glob
import math
from collections import deque
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import EvalCallback
from stable_baselines3.common.vec_env import DummyVecEnv, VecFrameStack

try:
    from scipy import stats as scipy_stats
    HAS_SCIPY = True
except ImportError:
    scipy_stats = None
    HAS_SCIPY = False

print("=== BƯỚC 4.5 (CHUẨN IEEE): ĐỐI CHỨNG CLOSED-LOOP 5 SEEDS ĐỘC LẬP + GAMMA P99 ĐỒNG NHẤT ===")

DRIVE_ROOT = globals().get("DRIVE_BASE_PATH", os.path.abspath("./KHKT_training_digital_twin_8DState_7DAct"))
SUMMARY_DIR = os.path.join(DRIVE_ROOT, "summary_reports")
os.makedirs(SUMMARY_DIR, exist_ok=True)
if "log_dir" not in globals():
    log_dir = os.path.join(DRIVE_ROOT, "wifi_rl_logs")
    os.makedirs(log_dir, exist_ok=True)
if "save_dir" not in globals():
    save_dir = os.path.join(DRIVE_ROOT, "saved_models")
    os.makedirs(save_dir, exist_ok=True)

SEEDS_LIST = [11, 22, 33, 44, 55]
TIMESTEPS_PER_SEED = 360000
N_EVAL_EPISODES = 10      # 10 episodes * 300 bước = 3,000 bước closed-loop / seed
EPISODE_STEPS = 300
PACKETS_PER_STEP = 100    # 3,000 * 100 = 300,000 mẫu gói tin độc lập / seed
FORCE_RETRAIN_ALL_SEEDS = False  # Đặt True để train lại từ đầu 5 seeds Dual-Sensing (360k steps/seed)

# Dọn sạch tàn dư monitor cũ (4..7.monitor.csv) trong thư mục logs để Learning Curve không bị cắt cụt
for _stale in glob.glob(os.path.join(log_dir, "[4-9].monitor.csv")):
    try:
        os.remove(_stale)
    except OSError:
        pass


def train_or_load_seed_model(seed_val: int, timesteps: int = TIMESTEPS_PER_SEED):
    """
    Huấn luyện độc lập PPO Dual-Sensing cho Seed `seed_val` (KHÔNG nhân bản trọng số từ fallback_zip).
    Luôn dọn *.monitor.csv cũ trước khi train và luôn nạp lại `best_model.zip` từ EvalCallback.
    """
    seed_dir = os.path.join(DRIVE_ROOT, f"seed_{seed_val}")
    os.makedirs(seed_dir, exist_ok=True)
    best_zip_path = os.path.join(seed_dir, "best_model.zip")

    custom_objs = {
        "learning_rate": 0.0,
        "lr_schedule": lambda _: 0.0,
        "clip_range": lambda _: 0.2,
        "NormalizedWifiFeaturesExtractor": NormalizedWifiFeaturesExtractor,
    }

    if os.path.exists(best_zip_path) and not FORCE_RETRAIN_ALL_SEEDS:
        try:
            m_loaded = PPO.load(best_zip_path, custom_objects=custom_objs, device="cpu")
            if list(m_loaded.action_space.nvec) == list(ACTION_NVEC) and int(m_loaded.observation_space.shape[0]) == 8 * N_FRAME_STACK:
                print(f"  [✔] Đã nạp checkpoint Dual-Sensing Seed {seed_val}: {best_zip_path}")
                return m_loaded, seed_dir
        except Exception:
            pass

    for mf in glob.glob(os.path.join(seed_dir, "*.monitor.csv")):
        try:
            os.remove(mf)
        except OSError:
            pass
    eval_npz = os.path.join(seed_dir, "evaluations.npz")
    if os.path.exists(eval_npz):
        try:
            os.remove(eval_npz)
        except OSError:
            pass

    print(f"  [*] Đang huấn luyện mới PPO Dual-Sensing Seed {seed_val} ({timesteps:,} timesteps) -> {seed_dir}")
    vec_env = VecFrameStack(
        make_vec_env(
            make_dual_sensing_env(seed_val, hardware_type="WIFI_6", aqm_enabled=True, obs_mask_phy=False),
            n_envs=4,
            seed=seed_val,
            monitor_dir=seed_dir,
        ),
        n_stack=N_FRAME_STACK,
    )
    eval_env_s = VecFrameStack(
        DummyVecEnv([lambda: Monitor(make_dual_sensing_env(seed_val + 1000, hardware_type="WIFI_6", aqm_enabled=True, obs_mask_phy=False)(), filename=os.path.join(seed_dir, "eval.monitor.csv"))]),
        n_stack=N_FRAME_STACK,
    )
    policy_kwargs_s = dict(
        features_extractor_class=NormalizedWifiFeaturesExtractor,
        net_arch=dict(pi=[256, 256], vf=[256, 256]),
    )
    model_s = PPO(
        "MlpPolicy",
        vec_env,
        learning_rate=piecewise_lr_schedule(4.0e-4, min_lr=1.5e-5),
        n_steps=1024,
        batch_size=256,
        n_epochs=8,
        gamma=0.55,
        gae_lambda=0.80,
        ent_coef=0.001,
        policy_kwargs=policy_kwargs_s,
        seed=seed_val,
        verbose=0,
        device="cpu",
    )
    eval_cb_s = EvalCallback(
        eval_env_s,
        best_model_save_path=seed_dir,
        log_path=seed_dir,
        eval_freq=9000,
        n_eval_episodes=10,
        deterministic=True,
        render=False,
    )
    model_s.learn(total_timesteps=timesteps, callback=eval_cb_s)
    if os.path.exists(best_zip_path):
        # Luôn nạp lại trọng số best_model.zip tốt nhất thay vì dùng model_s ở bước cuối cùng
        model_s = PPO.load(best_zip_path, custom_objects=custom_objs, device="cpu")
        print(f"  [✔] Đã nạp lại best_model.zip chuẩn xác cho Seed {seed_val}")
    else:
        model_s.save(best_zip_path)
    return model_s, seed_dir


def sample_packet_latencies_from_metrics(metrics: dict, rng: np.random.Generator, n_packets: int = PACKETS_PER_STEP) -> np.ndarray:
    """
    Lấy mẫu Monte Carlo chuẩn xác từ phân phối Gamma dịch chuyển X = d_min + Gamma(k, theta)
    dùng TRỰC TIẾP `mean_latency_ms` từ lõi vật lý (loại bỏ bug trừ 2.33 * jitter Gaussian cũ):
      - E[X] = mean_latency_ms
      - Var[X] = jitter_ms^2  ==>  k = ((mean_lat - d_min) / jitter) ** 2
    """
    d_min = 1.50
    mean_lat = max(d_min + 0.05, float(metrics["mean_latency_ms"]))
    jitter = max(0.04, float(metrics["jitter_ms"]))
    excess_mean = mean_lat - d_min
    shape_k = max(0.25, (excess_mean / jitter) ** 2.0)
    scale_theta = excess_mean / shape_k
    return rng.gamma(shape=shape_k, scale=scale_theta, size=n_packets) + d_min


def auto_channel_minstrel_heuristic(obs: np.ndarray, cur_ch: int, rng: np.random.Generator) -> np.ndarray:
    """
    Baseline Thương mại Công bằng (Auto-Channel ACS + Dynamic BW + Minstrel tự do, KHÔNG Dual-Sensing PHY):
    - Chỉ đọc telemetry tầng MAC/RSSI chuẩn router thương mại (retry_rate, p_c, airtime_util, sta_count, avg_rssi).
    - Không đọc `noise_floor` phổ vô tuyến, không đọc `ofdma_ratio`, không có tín hiệu lịch `is_break`,
      không biết OBSS-PD, không biết BF CSI-aging, và để Minstrel tự do (mcs_ceiling=11).
    """
    rssi, _, sta, retry_obs, pc_obs, airtime_obs, _, _ = [float(x) for x in obs]
    if (pc_obs > 0.54) or (retry_obs > 0.48) or (airtime_obs > 0.78):
        ch_out = (cur_ch + int(rng.integers(1, 5))) % len(NON_DFS_5G_CHANNELS)
    else:
        ch_out = cur_ch
    bw_out = 0 if (sta > 78.0 or pc_obs > 0.48) else (1 if (sta > 35.0 or retry_obs > 0.30) else 2)
    if ch_out == 8 and bw_out > 0:
        bw_out = 0
    tx_out = 6 if rssi < -68.0 else (4 if rssi < -55.0 else 2)
    return np.array([ch_out, bw_out, tx_out, 11, 1, 0, 1], dtype=np.int64)


def _compute_t_crit(df_val: int) -> float:
    if HAS_SCIPY:
        return float(scipy_stats.t.ppf(0.975, df=df_val))
    t_table = {1: 12.7062047364, 2: 4.3026527299, 3: 3.1824463053, 4: 2.7764451052, 5: 2.5705818366}
    return t_table.get(df_val, 1.9599639845)


# =====================================================================================
# 2. THỰC NGHIỆM CLOSED-LOOP PAIRED TRÊN 5 SEEDS ĐỘC LẬP (DUAL vs MAC-ONLY ĐỘC LẬP)
# =====================================================================================
POLICIES = ["Ora", "M", "MAC", "Heur", "N"]
ABLATION_2X2_POLICIES = ["M_NoGuard", "MAC_PhyGuard"]
ALL_EVAL_POLICIES = POLICIES + ABLATION_2X2_POLICIES
table1_records, table2_records, table1b_records = [], [], []
ref_lat = {}
ref_gp = {}

# Kiểm toán 7 vũ khí trên chuỗi đóng của Seed đầu tiên (3,000 bước)
weapon_exact = np.zeros(7, dtype=np.int64)
weapon_tol1 = np.zeros(7, dtype=np.int64)
weapon_used_bins = [set() for _ in range(7)]
dual_eval_steps_s0 = 0
dual_clean_hits_s0 = 0
dual_pingpong_s0 = 0

loaded_ppo = None
seed_dirs_map = {}
mac_dirs_map = {}

for s_i, seed_val in enumerate(SEEDS_LIST):
    model_dual, s_dir = train_or_load_seed_model(seed_val)
    seed_dirs_map[seed_val] = s_dir
    if s_i == 0:
        loaded_ppo = model_dual

    # Nạp mô hình PPO MAC-Only đã train độc lập ở Bước 3.1 (nếu chưa có sẽ tự train độc lập 300k steps)
    if "mac_models_by_seed" in globals() and seed_val in mac_models_by_seed:
        model_mac = mac_models_by_seed[seed_val]
        mac_dirs_map[seed_val] = mac_seed_dirs_map[seed_val]
    else:
        model_mac, d_mac = train_or_load_mac_seed_model(seed_val, timesteps=TIMESTEPS_PER_SEED, force_retrain=False)
        mac_dirs_map[seed_val] = d_mac

    rng_eval = np.random.default_rng(seed_val * 107)
    rng_heur = np.random.default_rng(seed_val * 313)
    rng_eval_abl = np.random.default_rng(seed_val * 997)
    hw_profile = HardwareAwareWrapper(UltimateDigitalTwinEnv(hardware_type="WIFI_6"), hardware_type="WIFI_6")

    pkts_buf = {p: [] for p in ALL_EVAL_POLICIES}
    step_p99_buf = {p: [] for p in ALL_EVAL_POLICIES}
    mean_lat_buf = {p: [] for p in ALL_EVAL_POLICIES}
    loss_buf = {p: [] for p in ALL_EVAL_POLICIES}
    gp_buf = {p: [] for p in ALL_EVAL_POLICIES}
    rew_buf = {p: [] for p in ALL_EVAL_POLICIES}
    sla_buf = {p: [] for p in ALL_EVAL_POLICIES}
    clean_buf = {p: [] for p in ALL_EVAL_POLICIES}
    sw_counts = {p: 0 for p in ALL_EVAL_POLICIES}
    step_qoe_rows = []

    for ep_idx in range(N_EVAL_EPISODES):
        # Sinh 1 quỹ đạo ngoại cảnh chuẩn xác cho cả 5 chính sách đối chứng (Paired Trajectory)
        world_env = UltimateDigitalTwinEnv(seed=seed_val * 100 + ep_idx + 500, max_steps=EPISODE_STEPS)
        o_init, info_init = world_env.reset(seed=seed_val * 100 + ep_idx + 500)
        init_ch = int(info_init.get("current_ch", world_env.current_ch))
        traj_noise = [world_env.true_ch_noise.copy()]
        traj_rssi = [float(o_init[0])]
        traj_sta = [float(o_init[2])]
        traj_ofdma_base = [float(world_env.target_ofdma_base)]
        traj_demand = [float(world_env.target_demand_scale)]
        traj_brk = [float(world_env.current_is_break)]
        for _ in range(EPISODE_STEPS):
            world_env.step([0, 2, 3, 7, 1, 2, 1])
            traj_noise.append(world_env.true_ch_noise.copy())
            traj_rssi.append(float(world_env.true_current_obs[0]))
            traj_sta.append(float(world_env.true_current_obs[2]))
            traj_ofdma_base.append(float(world_env.target_ofdma_base))
            traj_demand.append(float(world_env.target_demand_scale))
            traj_brk.append(float(world_env.current_is_break))

        for p_name in ALL_EVAL_POLICIES:
            cur_obs = o_init.copy()
            cur_ch = init_ch
            cur_backlog_mb = 0.0
            prev_act = None
            ch_history = []
            guard = StatefulChannelGuardWrapper(HardwareAwareWrapper(UltimateDigitalTwinEnv()))
            guard.current_ch = cur_ch
            first_seen = cur_obs.copy()
            if p_name in ("MAC", "MAC_PhyGuard"):
                first_seen[1] = -95.0
                first_seen[6] = 0.50
            obs_win = deque([np.zeros(8, dtype=np.float32)] * (N_FRAME_STACK - 1) + [first_seen], maxlen=N_FRAME_STACK)

            for t_step in range(EPISODE_STEPS):
                ch_n = traj_noise[t_step]
                of_b = traj_ofdma_base[t_step]
                dm_s = traj_demand[t_step]

                obs_seen = cur_obs.copy()
                if p_name in ("MAC", "MAC_PhyGuard"):
                    obs_seen[1] = -95.0
                    obs_seen[6] = 0.50

                if p_name == "Ora":
                    act, _ = UltimateDigitalTwinEnv.compute_optimal_targets(
                        cur_obs, aqm_enabled=True, sub_noise_dbm=ch_n, base_ofdma_capacity=of_b,
                        demand_scale=dm_s, queue_backlog_mb=cur_backlog_mb,
                    )
                    if prev_act is not None and (int(act[0]) != int(prev_act[0]) or int(act[1]) != int(prev_act[1])):
                        m_c = UltimateDigitalTwinEnv._original_evaluate_network_physics(
                            cur_obs, act, aqm_enabled=True, sub_noise_dbm=ch_n, base_ofdma_capacity=of_b,
                            demand_scale=dm_s, queue_backlog_mb=cur_backlog_mb,
                        )
                        m_s = UltimateDigitalTwinEnv._original_evaluate_network_physics(
                            cur_obs, prev_act, aqm_enabled=True, sub_noise_dbm=ch_n, base_ofdma_capacity=of_b,
                            demand_scale=dm_s, queue_backlog_mb=cur_backlog_mb,
                        )
                        if m_c["reward"] - m_s["reward"] <= 0.35:
                            act = prev_act.copy()
                elif p_name == "M":
                    raw_a, _ = model_dual.predict(build_stacked_obs(obs_win, N_FRAME_STACK), deterministic=True)
                    guard.step_count = t_step
                    guarded_a, _ = guard.filter_channel_action(obs_seen, raw_a)
                    act = hw_profile.action(guarded_a)
                    if int(act[0]) != guard.current_ch:
                        guard.last_switch_step = t_step + 1
                    guard.current_ch = int(act[0])
                elif p_name == "M_NoGuard":
                    raw_a, _ = model_dual.predict(build_stacked_obs(obs_win, N_FRAME_STACK), deterministic=True)
                    act = hw_profile.action(raw_a)
                elif p_name == "MAC_PhyGuard":
                    raw_a, _ = model_mac.predict(build_stacked_obs(obs_win, N_FRAME_STACK), deterministic=True)
                    guard.step_count = t_step
                    # Mạng nơ-ron nhận obs_seen (bị che PHY), riêng Guardrail dùng cur_obs thật để phân tách 2x2!
                    guarded_a, _ = guard.filter_channel_action(cur_obs, raw_a)
                    act = hw_profile.action(guarded_a)
                    if int(act[0]) != guard.current_ch:
                        guard.last_switch_step = t_step + 1
                    guard.current_ch = int(act[0])
                elif p_name == "MAC":
                    raw_a, _ = model_mac.predict(build_stacked_obs(obs_win, N_FRAME_STACK), deterministic=True)
                    guard.step_count = t_step
                    guarded_a, _ = guard.filter_channel_action(obs_seen, raw_a)
                    act = hw_profile.action(guarded_a)
                    if int(act[0]) != guard.current_ch:
                        guard.last_switch_step = t_step + 1
                    guard.current_ch = int(act[0])
                elif p_name == "Heur":
                    act = auto_channel_minstrel_heuristic(obs_seen, cur_ch, rng_heur)
                else:  # Static Legacy Minstrel (Router phổ thông: CH36 | HE80 | Tx=20 | Minstrel=11 | Bcn=300 | SR=-82 | BF=OFF | AQM=False)
                    act = np.array([0, 2, 6, 11, 1, 0, 0], dtype=np.int64)

                aqm_on = (p_name != "N")
                m = UltimateDigitalTwinEnv.evaluate_network_physics(
                    cur_obs, act, prev_action=prev_act, aqm_enabled=aqm_on,
                    sub_noise_dbm=ch_n, base_ofdma_capacity=of_b, demand_scale=dm_s,
                    queue_backlog_mb=cur_backlog_mb,
                )
                if prev_act is not None and (int(act[0]) != int(prev_act[0]) or int(act[1]) != int(prev_act[1])):
                    sw_counts[p_name] += 1

                # Tính độ sạch kênh thực sự trên toàn bộ dải subchannel bị chiếm
                _, _, occ_subs, _ = resolve_valid_5g_channel_and_bw(int(act[0]), int(act[1]))
                occ_max_noise = max(float(ch_n[NON_DFS_5G_CHANNELS.index(c)]) for c in occ_subs)
                if np.min(ch_n) <= -85.0:
                    clean_buf[p_name].append(100.0 if occ_max_noise <= -85.0 else 0.0)

                # Sinh 100 mẫu gói tin Gamma chuẩn khớp `mean_latency_ms` & `jitter_ms`
                pkt_rng = rng_eval if p_name in POLICIES else rng_eval_abl
                pkts = sample_packet_latencies_from_metrics(m, pkt_rng, n_packets=PACKETS_PER_STEP)
                sc_p99 = float(np.percentile(pkts, 99))
                pkts_buf[p_name].extend(pkts)
                step_p99_buf[p_name].append(sc_p99)
                mean_lat_buf[p_name].append(m["mean_latency_ms"])
                loss_buf[p_name].append(m["packet_loss_pct"])
                gp_buf[p_name].append(m["goodput"])
                rew_buf[p_name].append(m["reward"])
                # QoE SLA thật trên cả 5 chính sách (bao gồm cả Simulator Oracle!)
                sla_ok = 100.0 if (m["p99_latency_ms"] <= QOE_LATENCY_MS and m["packet_loss_pct"] < 1.0) else 0.0
                sla_buf[p_name].append(sla_ok)

                # Kiểm toán 7 chiều hành động của PPO Dual-Sensing (Seed đầu tiên) so với Oracle có điều kiện tại chính kênh đang đứng
                if s_i == 0 and p_name == "M":
                    dual_eval_steps_s0 += 1
                    if occ_max_noise <= -85.0:
                        dual_clean_hits_s0 += 1
                    ch_history.append(int(act[0]))
                    if len(ch_history) >= 3 and ch_history[-1] == ch_history[-3] and ch_history[-1] != ch_history[-2]:
                        dual_pingpong_s0 += 1

                    # Đối chiếu trực tiếp với nghiệm chuẩn compute_optimal_targets (đồng bộ 100% mục tiêu huấn luyện)
                    ora_full_act, _ = UltimateDigitalTwinEnv.compute_optimal_targets(
                        cur_obs, aqm_enabled=True, sub_noise_dbm=ch_n, base_ofdma_capacity=of_b,
                        demand_scale=dm_s, queue_backlog_mb=cur_backlog_mb,
                    )
                    for d in range(7):
                        weapon_used_bins[d].add(int(act[d]))
                        if d == 0:
                            # Với kênh: kiểm tra toàn cụm subchannel đang gộp (occupied_subs) sạch <= -85 dBm
                            if int(act[0]) == int(ora_full_act[0]) or occ_max_noise <= -85.0:
                                weapon_exact[0] += 1
                            if occ_max_noise <= -82.0:
                                weapon_tol1[0] += 1
                        else:
                            target_val = 0 if (d == 1 and int(act[0]) == 8) else int(ora_full_act[d])
                            if int(act[d]) == target_val:
                                weapon_exact[d] += 1
                            if abs(int(act[d]) - target_val) <= 1:
                                weapon_tol1[d] += 1

                if p_name == "M":
                    step_qoe_rows.append({
                        "episode_id": ep_idx,
                        "step_id": t_step,
                        "channel_chosen": NON_DFS_5G_CHANNELS[int(act[0])],
                        "mean_lat_M_ms": m["mean_latency_ms"],
                        "step_p99_M_ms": sc_p99,
                        "analytic_p99_M_ms": m["p99_latency_ms"],
                        "goodput_M_mbps": m["goodput"],
                        "jitter_M_ms": m["jitter_ms"],
                        "loss_M_pct": m["packet_loss_pct"],
                        "sla_met": sla_ok / 100.0,
                    })

                prev_act = act.copy()
                cur_ch = int(act[0])
                cur_backlog_mb = float(m.get("backlog_next_mb", 0.0))
                # Tiến hóa trạng thái nhân quả đóng cho chính sách `p_name` (đo nhiễu mt76 primary + ghép phụ -4.5 dB)
                mt76_n_next = measure_mt76_survey_noise(traj_noise[t_step + 1], cur_ch, int(act[1]))
                next_noise = float(np.clip(mt76_n_next, OBS_LOW[1], OBS_HIGH[1]))
                next_retry = float(np.clip(0.25 * float(cur_obs[3]) + 0.75 * m["retry_rate"], OBS_LOW[3], OBS_HIGH[3]))
                next_pc = float(np.clip(0.25 * float(cur_obs[4]) + 0.75 * m["p_c"], OBS_LOW[4], OBS_HIGH[4]))
                next_air = float(np.clip(0.25 * float(cur_obs[5]) + 0.75 * m["airtime_next"], OBS_LOW[5], OBS_HIGH[5]))
                of_causal = float(np.clip(traj_ofdma_base[t_step + 1] * (1.0 - 0.55 * m["p_c"] - 0.25 * m["retry_rate"]), 0.06, 0.95))
                next_ofdma = float(np.clip(0.25 * float(cur_obs[6]) + 0.75 * (0.5 * m["ofdma_next"] + 0.5 * of_causal), OBS_LOW[6], OBS_HIGH[6]))
                cur_obs = np.array([
                    traj_rssi[t_step + 1], next_noise, traj_sta[t_step + 1],
                    next_retry, next_pc, next_air, next_ofdma, traj_brk[t_step + 1]
                ], dtype=np.float32)

                obs_next_seen = cur_obs.copy()
                if p_name in ("MAC", "MAC_PhyGuard"):
                    obs_next_seen[1] = -95.0
                    obs_next_seen[6] = 0.50
                obs_win.append(obs_next_seen)

    pd.DataFrame(step_qoe_rows).to_csv(os.path.join(s_dir, f"packet_qoe_samples_seed_{seed_val}.csv"), index=False)

    if s_i == 0:
        for p_name in ["M", "MAC", "Heur", "N"]:
            ref_lat[p_name] = np.array(pkts_buf[p_name], dtype=np.float32)
            ref_gp[p_name] = list(gp_buf[p_name])

    denom_s = max(float(np.mean(rew_buf["Ora"])) - float(np.mean(rew_buf["N"])), 1e-6)
    opt_m = 100.0 * (1.0 - (float(np.mean(rew_buf["Ora"])) - float(np.mean(rew_buf["M"]))) / denom_s)
    opt_mac = 100.0 * (1.0 - (float(np.mean(rew_buf["Ora"])) - float(np.mean(rew_buf["MAC"]))) / denom_s)
    opt_h = 100.0 * (1.0 - (float(np.mean(rew_buf["Ora"])) - float(np.mean(rew_buf["Heur"]))) / denom_s)
    opt_m_ng = 100.0 * (1.0 - (float(np.mean(rew_buf["Ora"])) - float(np.mean(rew_buf["M_NoGuard"]))) / denom_s)
    opt_mac_pg = 100.0 * (1.0 - (float(np.mean(rew_buf["Ora"])) - float(np.mean(rew_buf["MAC_PhyGuard"]))) / denom_s)
    table1b_records.append({
        "Seed": f"seed {seed_val}",
        "Opt Dual+PHY-Guard (%)": opt_m,
        "Opt Dual No-Guard (%)": opt_m_ng,
        "Opt MAC+PHY-Guard (%)": opt_mac_pg,
        "Opt MAC+MAC-Guard (%)": opt_mac,
        "SLA Dual+PHY-Guard (%)": float(np.mean(sla_buf["M"])),
        "SLA Dual No-Guard (%)": float(np.mean(sla_buf["M_NoGuard"])),
        "SLA MAC+PHY-Guard (%)": float(np.mean(sla_buf["MAC_PhyGuard"])),
        "SLA MAC+MAC-Guard (%)": float(np.mean(sla_buf["MAC"])),
        "P99 Dual+PHY-Guard (ms)": float(np.percentile(pkts_buf["M"], 99)),
        "P99 Dual No-Guard (ms)": float(np.percentile(pkts_buf["M_NoGuard"], 99)),
        "P99 MAC+PHY-Guard (ms)": float(np.percentile(pkts_buf["MAC_PhyGuard"], 99)),
        "P99 MAC+MAC-Guard (ms)": float(np.percentile(pkts_buf["MAC"], 99)),
        "CleanCh Dual+PHY-Guard (%)": float(np.mean(clean_buf["M"])),
        "CleanCh Dual No-Guard (%)": float(np.mean(clean_buf["M_NoGuard"])),
        "CleanCh MAC+PHY-Guard (%)": float(np.mean(clean_buf["MAC_PhyGuard"])),
        "CleanCh MAC+MAC-Guard (%)": float(np.mean(clean_buf["MAC"])),
    })

    table1_records.append({
        "Seed": f"seed {seed_val}",
        "Optimality Dual (M)": opt_m,
        "Optimality MAC-Only": opt_mac,
        "Optimality Heuristic": opt_h,
        "Optimality Static (N)": 0.0,
        "QoE SLA Oracle": float(np.mean(sla_buf["Ora"])),
        "QoE SLA Dual (M)": float(np.mean(sla_buf["M"])),
        "QoE SLA MAC-Only": float(np.mean(sla_buf["MAC"])),
        "QoE SLA Heuristic": float(np.mean(sla_buf["Heur"])),
        "QoE SLA Static (N)": float(np.mean(sla_buf["N"])),
        "CleanCh Dual (%)": float(np.mean(clean_buf["M"])),
        "CleanCh MAC-Only (%)": float(np.mean(clean_buf["MAC"])),
        "Sw/Ep Dual": sw_counts["M"] / float(N_EVAL_EPISODES),
        "Sw/Ep Oracle": sw_counts["Ora"] / float(N_EVAL_EPISODES),
    })

    # Table 2 báo cáo CẢ Step Mean-P99 lẫn Pooled System P99 (khớp 100% ECDF & Histogram!) + % Đạt Trần Oracle
    gp_ora_s = max(float(np.mean(gp_buf["Ora"])), 1e-6)
    table2_records.append({
        "Seed": f"seed {seed_val}",
        "Pooled P99 Oracle": float(np.percentile(pkts_buf["Ora"], 99)),
        "Pooled P99 Dual (M)": float(np.percentile(pkts_buf["M"], 99)),
        "Pooled P99 MAC-Only": float(np.percentile(pkts_buf["MAC"], 99)),
        "Pooled P99 Heuristic": float(np.percentile(pkts_buf["Heur"], 99)),
        "Pooled P99 Static (N)": float(np.percentile(pkts_buf["N"], 99)),
        "Step Mean-P99 Dual (M)": float(np.mean(step_p99_buf["M"])),
        "Step Mean-P99 MAC-Only": float(np.mean(step_p99_buf["MAC"])),
        "Step Mean-P99 Heuristic": float(np.mean(step_p99_buf["Heur"])),
        "Step Mean-P99 Static (N)": float(np.mean(step_p99_buf["N"])),
        "Mean Lat Dual (M)": float(np.mean(mean_lat_buf["M"])),
        "Loss Oracle": float(np.mean(loss_buf["Ora"])),
        "Loss Dual (M)": float(np.mean(loss_buf["M"])),
        "Loss MAC-Only": float(np.mean(loss_buf["MAC"])),
        "Loss Heuristic": float(np.mean(loss_buf["Heur"])),
        "Loss Static (N)": float(np.mean(loss_buf["N"])),
        "Goodput Oracle": float(np.mean(gp_buf["Ora"])),
        "Goodput Dual (M)": float(np.mean(gp_buf["M"])),
        "Goodput MAC-Only": float(np.mean(gp_buf["MAC"])),
        "Goodput Heuristic": float(np.mean(gp_buf["Heur"])),
        "Goodput Static (N)": float(np.mean(gp_buf["N"])),
        "Attainment Dual (%Ora)": 100.0 * float(np.mean(gp_buf["M"])) / gp_ora_s,
        "Attainment MAC (%Ora)": 100.0 * float(np.mean(gp_buf["MAC"])) / gp_ora_s,
        "Attainment Heur (%Ora)": 100.0 * float(np.mean(gp_buf["Heur"])) / gp_ora_s,
        "Attainment Static (%Ora)": 100.0 * float(np.mean(gp_buf["N"])) / gp_ora_s,
    })

df_t1_raw = pd.DataFrame(table1_records)
df_t2_raw = pd.DataFrame(table2_records)
t_val = _compute_t_crit(len(SEEDS_LIST) - 1)


# =====================================================================================
# 3. BIỂU ĐỒ 4 KHUNG PHÂN PHỐI P99, PACKET LOSS & GOODPUT (KHỚP 100% BẢNG 2 POOLED P99)
# =====================================================================================
plt.rcParams["font.sans-serif"] = ["DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

fig_qoe, axes_q = plt.subplots(2, 2, figsize=(16.5, 11.5), dpi=300)
fig_qoe.patch.set_facecolor("white")
for ax in axes_q.flat:
    ax.set_facecolor("#F8F9FA")
    ax.grid(True, linestyle="--", alpha=0.5, color="#CED4DA")

# Panel A: Histogram + KDE + P50/P95/P99
p50_v, p95_v, p99_v = np.percentile(ref_lat["M"], [50, 95, 99])
max_x_lat = max(float(QOE_LATENCY_MS) + 4.0, float(p99_v) + 4.5)
bins_l = np.linspace(1.5, max_x_lat, 55)
axes_q[0, 0].hist(
    ref_lat["M"], bins=bins_l, density=True, color="#2A9D8F", alpha=0.68, edgecolor="white",
    label=f"Phân phối Độ trễ Gói tin Hệ thống M (N = {len(ref_lat['M']):,} mẫu)"
)
x_g = np.linspace(1.5, max_x_lat, 300)
if HAS_SCIPY:
    kde_func = scipy_stats.gaussian_kde(ref_lat["M"][::max(1, len(ref_lat["M"]) // 25000)])
    axes_q[0, 0].plot(x_g, kde_func(x_g), color="#005F73", lw=2.4, label="Mật độ xác suất thực nghiệm (KDE)")
axes_q[0, 0].axvline(p50_v, color="#0077B6", ls=":", lw=2.0, label=f"Trung vị P50 = {p50_v:.2f} ms")
axes_q[0, 0].axvline(p95_v, color="#F77F00", ls="--", lw=2.0, label=f"Phân vị P95 = {p95_v:.2f} ms")
axes_q[0, 0].axvline(p99_v, color="#D90429", ls="-", lw=2.6, label=f"Pooled P99 Thực nghiệm = {p99_v:.2f} ms")
axes_q[0, 0].axvline(float(QOE_LATENCY_MS), color="#2B2D42", ls="-.", lw=2.0, label=f"Ngưỡng Cam kết QoE SLA ({float(QOE_LATENCY_MS):.1f} ms)")
axes_q[0, 0].set_title("(A) HISTOGRAM ĐỘ TRỄ GÓI TIN & POOLED SYSTEM P99 (SEED 11)", fontsize=11.5, fontweight="bold")
axes_q[0, 0].set_xlabel("Độ Trễ Truyền Dẫn Gói Tin (Packet Latency - ms)", fontweight="bold")
axes_q[0, 0].set_ylabel("Mật Độ Xác Suất (Probability Density)", fontweight="bold")
axes_q[0, 0].set_xlim(1.5, max_x_lat)
axes_q[0, 0].legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.92, fontsize=8.5)

# Panel B: ECDF đối chứng Pooled System P99 của 4 chính sách (khớp 100% Bảng 2 hàng Seed 11!)
for p_key, lbl_prefix, col_d, ls_d in [
    ("M", "1. PPO Dual-Sensing M", "#2A9D8F", "-"),
    ("MAC", "2. PPO MAC-Only (Indep.)", "#0077B6", "--"),
    ("Heur", "3. Auto-Ch Minstrel Heur.", "#F77F00", "-."),
    ("N", "4. Static Legacy N", "#D90429", ":"),
]:
    arr_d = ref_lat[p_key]
    p99_d = float(np.percentile(arr_d, 99))
    sv = np.sort(arr_d)
    yc = np.arange(1, len(sv) + 1) / len(sv) * 100.0
    step_k = max(1, len(sv) // 1000)
    axes_q[0, 1].plot(sv[::step_k], yc[::step_k], color=col_d, ls=ls_d, lw=2.5, label=f"{lbl_prefix} (Pooled P99 = {p99_d:.2f} ms)")
axes_q[0, 1].axhline(99.0, color="#D90429", ls=":", lw=1.8, label="Đường Cắt Phân Vị Thứ 99 (99th Percentile)")
axes_q[0, 1].axvline(float(QOE_LATENCY_MS), color="#2B2D42", ls="-.", lw=1.8, label=f"Ngưỡng Cam kết SLA ({float(QOE_LATENCY_MS):.1f} ms)")
axes_q[0, 1].set_xlim(0, 120)
axes_q[0, 1].set_ylim(0, 103)
axes_q[0, 1].set_title("(B) ECDF ĐỐI CHỨNG POOLED SYSTEM P99 (KHỚP 100% BẢNG 2)", fontsize=11.5, fontweight="bold")
axes_q[0, 1].set_xlabel("Độ Trễ Gói Tin (ms) [Cắt hiển thị ngang tại 120 ms]", fontweight="bold")
axes_q[0, 1].set_ylabel("Tỷ Lệ Tích Lũy Gói Tin (%)", fontweight="bold")
axes_q[0, 1].legend(loc="lower right", frameon=True, facecolor="white", framealpha=0.92, fontsize=8.5)

# Panel C: Packet Loss ± 95% CI
pol_short = ["PPO Dual-Sensing\n(Đề xuất M)", "PPO MAC-Only\n(Train Độc Lập)", "Auto-Channel\nMinstrel Heur.", "Static Legacy\n(Hệ thống N)"]
loss_cols = ["Loss Dual (M)", "Loss MAC-Only", "Loss Heuristic", "Loss Static (N)"]
l_means = [df_t2_raw[c].mean() for c in loss_cols]
l_cis = [t_val * df_t2_raw[c].std(ddof=1) / np.sqrt(len(SEEDS_LIST)) for c in loss_cols]
b_colors = ["#2A9D8F", "#0077B6", "#F77F00", "#D90429"]
x_p = np.arange(4)
bars_l = axes_q[1, 0].bar(x_p, l_means, width=0.46, yerr=l_cis, capsize=6, color=b_colors, alpha=0.88, edgecolor="black", label="Packet Loss Trung bình (% ± 95% CI qua 5 Seeds)")
axes_q[1, 0].axhline(1.0, color="#D90429", ls="--", lw=1.8, label="Ngưỡng Cam kết SLA (Packet Loss < 1.0%)")
for b in bars_l:
    h = b.get_height()
    axes_q[1, 0].annotate(f"{h:.2f}%", xy=(b.get_x() + b.get_width() / 2, h), xytext=(0, 5), textcoords="offset points", ha="center", va="bottom", fontsize=9.5, fontweight="bold")
axes_q[1, 0].set_xticks(x_p); axes_q[1, 0].set_xticklabels(pol_short, fontweight="bold", fontsize=9.5)
axes_q[1, 0].set_ylim(0, max(l_means) * 1.25 + 2.0)
axes_q[1, 0].set_title("(C) TỶ LỆ MẤT GÓI TRUNG BÌNH CLOSED-LOOP (% ± 95% CI)", fontsize=11.5, fontweight="bold")
axes_q[1, 0].set_ylabel("Tỷ Lệ Mất Gói Đường Truyền (%)", fontweight="bold")
axes_q[1, 0].legend(loc="upper left", frameon=True, facecolor="white", framealpha=0.92, fontsize=8.8)

# Panel D: Goodput Box Plot + Đường Chuẩn Học Thuật Oracle Capacity Bound (100% Attainment)
gp_lists = [ref_gp["M"], ref_gp["MAC"], ref_gp["Heur"], ref_gp["N"]]
ora_gp_ref = float(df_t2_raw["Goodput Oracle"].mean())
bp = axes_q[1, 1].boxplot(
    gp_lists, positions=x_p, widths=0.45, patch_artist=True, showmeans=True,
    meanprops=dict(marker="D", markeredgecolor="black", markerfacecolor="#FFB703", markersize=8),
    medianprops=dict(color="black", linewidth=2.0), flierprops=dict(marker="o", markersize=2.5, alpha=0.35),
)
for patch, col in zip(bp["boxes"], b_colors):
    patch.set_facecolor(col); patch.set_alpha(0.75)
axes_q[1, 1].axhline(
    ora_gp_ref, color="#D90429", ls="--", lw=2.1,
    label=f"Trần Lý thuyết Oracle = {ora_gp_ref:.1f} Mbps (100% Capacity Attainment)"
)
for i_g, (c_g, c_att) in enumerate([
    ("Goodput Dual (M)", "Attainment Dual (%Ora)"),
    ("Goodput MAC-Only", "Attainment MAC (%Ora)"),
    ("Goodput Heuristic", "Attainment Heur (%Ora)"),
    ("Goodput Static (N)", "Attainment Static (%Ora)"),
]):
    gm = df_t2_raw[c_g].mean()
    att_m = df_t2_raw[c_att].mean()
    axes_q[1, 1].annotate(
        f"TB: {gm:.1f} Mbps\n({att_m:.1f}% Oracle)",
        xy=(x_p[i_g], max(gp_lists[i_g]) + 3.0),
        ha="center", va="bottom", fontsize=8.5, fontweight="bold", color="#1D3557",
    )
axes_q[1, 1].set_xticks(x_p); axes_q[1, 1].set_xticklabels(pol_short, fontweight="bold", fontsize=9.5)
axes_q[1, 1].set_ylim(0, max(max(ref_gp["M"]), ora_gp_ref) * 1.24 + 12.0)
axes_q[1, 1].set_title("(D) PHÂN PHỐI GOODPUT & HIỆU SUẤT ĐẠT TRẦN LÝ THUYẾT ORACLE", fontsize=11.5, fontweight="bold")
axes_q[1, 1].set_ylabel("Thông Lượng Hữu Ích Goodput (Mbps)", fontweight="bold")
axes_q[1, 1].legend(loc="upper right", frameon=True, facecolor="white", framealpha=0.92, fontsize=8.5)

plt.tight_layout()
qoe_chart_path = os.path.join(SUMMARY_DIR, "p99_histogram_ecdf_and_multiseed_ci95.png")
plt.savefig(qoe_chart_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[+] Đã xuất Biểu đồ 4 Khung tại: {qoe_chart_path}")


# =====================================================================================
# 4. IN & XUẤT CSV BẢNG 1, BẢNG 2 (TÁCH RÕ STEP MEAN-P99 VÀ POOLED P99) & BẢNG 7 VŨ KHÍ
# =====================================================================================
def append_ci95_row(df_in: pd.DataFrame, unit_map: dict) -> pd.DataFrame:
    formatted_rows = []
    for _, r in df_in.iterrows():
        row_dict = {"Seed": r["Seed"]}
        for col in df_in.columns[1:]:
            u = unit_map.get(col, "")
            row_dict[col] = f"{r[col]:.2f}{u}"
        formatted_rows.append(row_dict)
    ci_row = {"Seed": "Trung bình ± 95% CI"}
    for col in df_in.columns[1:]:
        m = df_in[col].mean()
        sd = df_in[col].std(ddof=1)
        ci = t_val * sd / np.sqrt(len(df_in))
        u = unit_map.get(col, "")
        ci_row[col] = f"{m:.2f} ± {ci:.2f}{u}"
    formatted_rows.append(ci_row)
    return pd.DataFrame(formatted_rows)

u_map1 = {c: ("%" if ("Optimality" in c or "SLA" in c or "CleanCh" in c) else "") for c in df_t1_raw.columns[1:]}
u_map2 = {c: (" ms" if ("P99" in c or "Lat" in c) else ("%" if ("Loss" in c or "%Ora" in c) else " Mbps")) for c in df_t2_raw.columns[1:]}

df_table1_final = append_ci95_row(df_t1_raw, u_map1)
df_table2_final = append_ci95_row(df_t2_raw, u_map2)

print("\n" + "=" * 128)
print(" BẢNG 1: ĐỐI CHỨNG CLOSED-LOOP OPTIMALITY, QOE SLA THỰC TẾ, CLEAN-CH & TẦN SUẤT CHUYỂN MẠCH (5 SEEDS ± 95% CI)")
print("=" * 128)
print(df_table1_final.to_string(index=False))

print("\n" + "=" * 128)
print(" BẢNG 2A: ĐỘ TRỄ POOLED SYSTEM P99 (KHỚP 100% ĐỒ THỊ ECDF/HISTOGRAM) & STEP MEAN-P99 (ms)")
print("=" * 128)
t2a_cols = ["Seed", "Pooled P99 Dual (M)", "Pooled P99 MAC-Only", "Pooled P99 Heuristic", "Pooled P99 Static (N)",
            "Step Mean-P99 Dual (M)", "Step Mean-P99 MAC-Only", "Step Mean-P99 Heuristic", "Step Mean-P99 Static (N)", "Mean Lat Dual (M)"]
print(df_table2_final[t2a_cols].to_string(index=False))

print("\n" + "=" * 128)
print(" BẢNG 2B: PACKET LOSS (%), GOODPUT (Mbps) & HIỆU SUẤT ĐẠT TRẦN LÝ THUYẾT ORACLE (%Ora) TRÊN 5 SEEDS (± 95% CI)")
print("=" * 128)
t2b_cols = [
    "Seed", "Loss Dual (M)", "Loss MAC-Only", "Loss Heuristic", "Loss Static (N)",
    "Goodput Dual (M)", "Goodput MAC-Only", "Goodput Heuristic", "Goodput Static (N)",
    "Attainment Dual (%Ora)", "Attainment MAC (%Ora)", "Attainment Heur (%Ora)", "Attainment Static (%Ora)",
]
print(df_table2_final[t2b_cols].to_string(index=False))
print(
    f"  [Chuẩn Học thuật Goodput (Flow Conservation Bound)] Trần Lý thuyết Oracle = {df_t2_raw['Goodput Oracle'].mean():.2f} Mbps (100.0% Attainment) | "
    f"Pooled P99 Oracle = {df_t2_raw['Pooled P99 Oracle'].mean():.2f} ms | Loss Oracle = {df_t2_raw['Loss Oracle'].mean():.2f}%"
)
print(
    f"  -> PPO Dual-Sensing đạt {df_t2_raw['Attainment Dual (%Ora)'].mean():.2f}% trần dung lượng lý thuyết "
    f"(so với MAC-Only {df_t2_raw['Attainment MAC (%Ora)'].mean():.2f}%, Heuristic {df_t2_raw['Attainment Heur (%Ora)'].mean():.2f}%, Static {df_t2_raw['Attainment Static (%Ora)'].mean():.2f}%)"
)

print("-" * 128)
print(" BẢNG 3: KIỂM ĐỊNH ĐỘ PHỦ & ĐỘ CHÍNH XÁC CÓ ĐIỀU KIỆN TRÊN 7 VŨ KHÍ RRM (3,000 BƯỚC CLOSED-LOOP SEED 11)")
print("-" * 128)
weapon_titles = [
    "1. Channel (9 kênh 5GHz Non-DFS)",
    "2. Bandwidth (HE20 / HE40 / HE80)",
    "3. Tx Power (14..20 dBm)",
    "4. MCS Ceiling (HE-MCS 0..11)",
    "5. Beacon Interval (80 / 300 ms)",
    "6. OBSS-PD Spatial Reuse (-82..-62 dBm)",
    "7. SU-Beamforming (OFF / ON)",
]
for d in range(7):
    ex_p = 100.0 * weapon_exact[d] / max(1, dual_eval_steps_s0)
    tl_p = 100.0 * weapon_tol1[d] / max(1, dual_eval_steps_s0)
    extra = f" | CleanCh={100.0*dual_clean_hits_s0/max(1,dual_eval_steps_s0):.1f}% | PingPong={100.0*dual_pingpong_s0/max(1,dual_eval_steps_s0):.2f}%" if d == 0 else ""
    print(f"  * {weapon_titles[d]:40s}: Số nấc kích hoạt = {len(weapon_used_bins[d]):2d}/{ACTION_NVEC[d]:2d} | Khớp chuẩn = {ex_p:5.1f}% | Dung sai ±1 = {tl_p:5.1f}%{extra}")
print("=" * 128)

t1_csv_path = os.path.join(SUMMARY_DIR, "table1_optimality_sla_multiseed_ci95.csv")
t2_csv_path = os.path.join(SUMMARY_DIR, "table2_p99_loss_goodput_multiseed_ci95.csv")
combined_csv_path = os.path.join(SUMMARY_DIR, "multiseed_ci95_benchmark_table.csv")

df_table1_final.to_csv(t1_csv_path, index=False)
df_table2_final.to_csv(t2_csv_path, index=False)
pd.concat([df_t1_raw, df_t2_raw.drop(columns=["Seed"])], axis=1).to_csv(combined_csv_path, index=False)
print(f"[+] Đã lưu CSV Bảng 1, Bảng 2 và Bảng Tổng hợp 5 Seeds tại: {SUMMARY_DIR}")

df_t1b_raw = pd.DataFrame(table1b_records)
u_map1b = {c: (" ms" if "P99" in c else "%") for c in df_t1b_raw.columns[1:]}
df_table1b_final = append_ci95_row(df_t1b_raw, u_map1b)
t1b_csv_path = os.path.join(SUMMARY_DIR, "table1b_ppo_vs_guardrail_disentanglement.csv")
df_table1b_final.to_csv(t1b_csv_path, index=False)
df_table1b_final.to_csv(os.path.join(log_dir, "table1b_ppo_vs_guardrail_disentanglement.csv"), index=False)
print("\n" + "=" * 128)
print(" BẢNG 1B: PHÂN TÁCH THỰC NGHIỆM 2x2 — GIÁ TRỊ PHY TRONG MẠNG PPO vs. TRONG GUARDRAIL (5 SEEDS ± 95% CI)")
print("=" * 128)
print(df_table1b_final.to_string(index=False))
print(f"[+] Đã lưu CSV Bảng 1B Disentanglement tại: {t1b_csv_path}")


# =====================================================================================
# 5. BIỂU ĐỒ LEARNING CURVE CHÍNH XÁC (20 FILES/NHÓM ĐẾN ĐÚNG 300,000 BƯỚC) + ABLATION + 7 VŨ KHÍ
# =====================================================================================
def _load_clean_monitor_curves(dir_map: dict):
    curves_r, curves_x = [], []
    for _sv in SEEDS_LIST:
        s_folder = dir_map.get(_sv)
        if not s_folder:
            continue
        for w_idx in range(4):
            mf = os.path.join(s_folder, f"{w_idx}.monitor.csv")
            if os.path.exists(mf):
                try:
                    d_m = pd.read_csv(mf, skiprows=1)
                    if "r" in d_m.columns and "l" in d_m.columns and len(d_m) >= 2:
                        curves_r.append(d_m["r"].values.astype(np.float64))
                        curves_x.append(np.cumsum(d_m["l"].values.astype(np.float64)) * 4.0)
                except Exception:
                    pass
    return curves_r, curves_x

fig_lc, axes_lc = plt.subplots(1, 3, figsize=(19.5, 5.8), dpi=300)

dual_r, dual_x = _load_clean_monitor_curves(seed_dirs_map)
mac_r, mac_x = _load_clean_monitor_curves(mac_dirs_map)

for r_list, x_list, lbl, col_ln, col_fill in [
    (dual_r, dual_x, f"PPO Dual-Sensing M (Mean ± 95% CI, {len(dual_r)} monitors)", "#1b5e20", "#66bb6a"),
    (mac_r, mac_x, f"PPO MAC-Only Indep. (Mean ± 95% CI, {len(mac_r)} monitors)", "#0077b6", "#90e0ef"),
]:
    if r_list:
        min_len = min(len(v) for v in r_list)
        mat_r = np.array([v[:min_len] for v in r_list])
        steps_x = np.mean(np.array([x[:min_len] for x in x_list]), axis=0)
        mean_r = mat_r.mean(axis=0)
        std_r = mat_r.std(axis=0, ddof=1) if len(r_list) > 1 else np.zeros_like(mean_r)
        win_roll = max(1, min_len // 15)
        s_mean = pd.Series(mean_r).rolling(window=win_roll, min_periods=1).mean().values
        s_ci = _compute_t_crit(max(1, len(SEEDS_LIST) - 1)) * pd.Series(std_r).rolling(window=win_roll, min_periods=1).mean().values / np.sqrt(max(1, len(SEEDS_LIST)))
        axes_lc[0].plot(steps_x, s_mean, color=col_ln, lw=2.3, label=lbl)
        axes_lc[0].fill_between(steps_x, s_mean - s_ci, s_mean + s_ci, color=col_fill, alpha=0.25)

axes_lc[0].set_title("(a) Đường cong Hội tụ Huấn luyện PPO trên 5 Seeds", fontsize=11.5, fontweight="bold")
axes_lc[0].set_xlabel("Tổng số bước tương tác (Timesteps)")
axes_lc[0].set_ylabel("Tổng điểm thưởng mỗi Episode (300 steps)")
axes_lc[0].grid(True, linestyle=":", alpha=0.6)
axes_lc[0].legend(loc="lower right", fontsize=8.2)

pol_labels = ["Simulator\nOracle", "PPO Dual\nSensing (M)", "PPO MAC\nOnly (Indep.)", "Auto-Ch.\nMinstrel Heur.", "Static\nLegacy (N)"]
opt_means = [
    100.0,
    df_t1_raw["Optimality Dual (M)"].mean(),
    df_t1_raw["Optimality MAC-Only"].mean(),
    df_t1_raw["Optimality Heuristic"].mean(),
    0.0,
]
# Sử dụng QoE SLA THỰC TẾ của Simulator Oracle (không gán cứng 100.0%!)
sla_means = [
    df_t1_raw["QoE SLA Oracle"].mean(),
    df_t1_raw["QoE SLA Dual (M)"].mean(),
    df_t1_raw["QoE SLA MAC-Only"].mean(),
    df_t1_raw["QoE SLA Heuristic"].mean(),
    df_t1_raw["QoE SLA Static (N)"].mean(),
]
x_pos = np.arange(len(pol_labels))
width = 0.36
b1 = axes_lc[1].bar(x_pos - width / 2, opt_means, width, label="Optimality Mean (%)", color="#1565c0", edgecolor="black")
b2 = axes_lc[1].bar(x_pos + width / 2, sla_means, width, label="QoE SLA Mean (%)", color="#2e7d32", edgecolor="black")
for bar in b1:
    h = bar.get_height()
    axes_lc[1].text(bar.get_x() + bar.get_width() / 2, h + 1.5, f"{h:.1f}%", ha="center", va="bottom", fontsize=7.8, fontweight="bold")
for bar in b2:
    h = bar.get_height()
    axes_lc[1].text(bar.get_x() + bar.get_width() / 2, h + 1.5, f"{h:.1f}%", ha="center", va="bottom", fontsize=7.8, fontweight="bold")
axes_lc[1].set_xticks(x_pos); axes_lc[1].set_xticklabels(pol_labels, fontsize=8.5)
axes_lc[1].set_ylim(0, 115)
axes_lc[1].set_title("(b) Đối chứng Cắt lớp Closed-Loop (5 Seeds Mean)", fontsize=11.5, fontweight="bold")
axes_lc[1].set_ylabel("Tỷ lệ phần trăm (%)")
axes_lc[1].grid(True, axis="y", linestyle=":", alpha=0.6)
axes_lc[1].legend(loc="upper right", fontsize=8.5)

weapon_labels = ["Channel\n(9 bins)", "Bandwidth\n(3 bins)", "Tx Power\n(7 bins)", "MCS Ceil\n(12 bins)", "Beacon\n(2 bins)", "OBSS-PD\n(5 bins)", "Beamform\n(2 bins)"]
exact_pct = [100.0 * m / max(1, dual_eval_steps_s0) for m in weapon_exact]
tol1_pct = [100.0 * m / max(1, dual_eval_steps_s0) for m in weapon_tol1]
wx = np.arange(7)
wb1 = axes_lc[2].bar(wx - width / 2, exact_pct, width, label="Khớp chuẩn Oracle (%)", color="#00838f", edgecolor="black")
wb2 = axes_lc[2].bar(wx + width / 2, tol1_pct, width, label="Dung sai ±1 / Sạch (%)", color="#fbc02d", edgecolor="black")
for idx_w, bar in enumerate(wb2):
    h = bar.get_height()
    axes_lc[2].text(bar.get_x() + bar.get_width() / 2, h + 1.5, f"{len(weapon_used_bins[idx_w])}/{ACTION_NVEC[idx_w]}", ha="center", va="bottom", fontsize=8, fontweight="bold")
axes_lc[2].set_xticks(wx); axes_lc[2].set_xticklabels(weapon_labels, fontsize=8.2)
axes_lc[2].set_ylim(0, 115)
axes_lc[2].set_title("(c) Độ chính xác Có Điều kiện & Độ phủ 7 Vũ khí RRM", fontsize=11.5, fontweight="bold")
axes_lc[2].set_ylabel("Độ khớp với Nghiệm tối ưu (%)")
axes_lc[2].grid(True, axis="y", linestyle=":", alpha=0.6)
axes_lc[2].legend(loc="lower right", fontsize=8.2)

plt.tight_layout()
plot_45_path = os.path.join(log_dir, "learning_curve_plot.png")
plt.savefig(plot_45_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[+] Đã lưu đồ thị Bước 4.5 (300 DPI) tại: {plot_45_path}")

=== BƯỚC 4.5 (CHUẨN IEEE): ĐỐI CHỨNG CLOSED-LOOP 5 SEEDS ĐỘC LẬP + GAMMA P99 ĐỒNG NHẤT ===
  [*] Đang huấn luyện mới PPO Dual-Sensing Seed 11 (360,000 timesteps) -> /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/seed_11


Eval num_timesteps=36000, episode_reward=694.99 +/- 188.27
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=72000, episode_reward=897.26 +/- 171.97
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=108000, episode_reward=924.27 +/- 161.53
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=144000, episode_reward=936.60 +/- 164.46
Episode length: 300.00 +/- 0.00
New best mean reward!


Eval num_timesteps=180000, episode_reward=941.32 +/- 157.98
Episode length: 300.00 +/- 0.00
New best mean reward!


... (52 outputs left out)

Eval num_timesteps=360000, episode_reward=738.38 +/- 258.19
Episode length: 300.00 +/- 0.00
New best mean reward!


  [✔] Đã nạp lại best_model.zip chuẩn xác cho Seed 55


[+] Đã xuất Biểu đồ 4 Khung tại: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/summary_reports/p99_histogram_ecdf_and_multiseed_ci95.png

 BẢNG 1: ĐỐI CHỨNG CLOSED-LOOP OPTIMALITY, QOE SLA THỰC TẾ, CLEAN-CH & TẦN SUẤT CHUYỂN MẠCH (5 SEEDS ± 95% CI)
               Seed Optimality Dual (M) Optimality MAC-Only Optimality Heuristic Optimality Static (N) QoE SLA Oracle QoE SLA Dual (M) QoE SLA MAC-Only QoE SLA Heuristic QoE SLA Static (N) CleanCh Dual (%) CleanCh MAC-Only (%)   Sw/Ep Dual  Sw/Ep Oracle
            seed 11              95.28%              91.70%               33.79%                 0.00%         82.80%           78.60%           74.87%            21.93%             16.23%           99.50%               98.63%        11.30         55.60
            seed 22              97.10%              94.40%               35.21%                 0.00%         90.00%           87.03%           84.77%            26.47%             13.40%    

[+] Đã lưu đồ thị Bước 4.5 (300 DPI) tại: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/wifi_rl_logs/learning_curve_plot.png


## BƯỚC 4.6: KIỂM CHỨNG ĐỘC LẬP TRÊN CHUỖI KAGGLE THỰC TẾ (`REAL_KAGGLE_CSV`) & STRESS TEST XUNG NHIỄU 600 BƯỚC
Bước cuối cùng tách bạch rành mạch **2 bài kiểm định độc lập** (tuyệt đối không dùng kịch bản giả lập đè lên dữ liệu thật và tuyệt đối không dùng Oracle/hàm vật lý sửa lưng hay lọc hành động AI):
1. **Chế độ 1 — Đọc & Phát lại Vòng kín Dữ liệu Thực tế Kaggle (`build_chronological_kaggle_trace` -> `REAL_KAGGLE_CSV`):**
   - Tự động quét đệ quy tìm đúng 3 tập dữ liệu thật (`TrainingData.csv`, `ValidationData.csv` của *UjiIndoorLoc* và `wifi_dataset_cleaned.csv`), giữ nguyên trật tự thời gian `TIMESTAMP` / `timestamp`.
   - Ánh xạ trực tiếp RSSI thu từ các điểm AP, mật độ thiết bị/AP lân cận, nhịp giờ cao điểm và dải tạp âm 9 kênh đo thực tế sang biến ngoại sinh của bộ mô phỏng — **không ghi đè bất kỳ mốc cứng hay nhiễu nhân tạo nào**.
2. **Chế độ 2 — Stress Test Xung Nhiễu Chuỗi Thời Gian 600 Bước (Synthetic Pulse Chaos Closed-Loop):**
   - Gây nhiễu có kiểm soát qua 3 cơn bão độc lập: Đợt 1 (`100..199`: nhiễu UNII-1 Low `CH36-40`), Đợt 2 (`300..399`: bão Roaming giờ giải lao `125 STA`), Đợt 3 (`500..599`: nhiễu băng rộng đánh sập toàn bộ UNII-1 `CH36-48` ép sơ tán lên `UNII-3`).
   - Toàn bộ suy luận của AI dùng đúng cửa sổ `VecFrameStack(4)` (`32D`) + **Change-Point Adaptive EWMA** + **`StatefulChannelGuardWrapper`** + **`HardwareAwareWrapper`**, tương đồng 1-1 với kiến trúc triển khai Router thật (`deploy_hil_agent.py`).

In [8]:
import glob
import os
from collections import deque
from typing import Dict, Optional, Tuple
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def _find_kaggle_csv_files() -> Dict[str, str]:
    """Quét đệ quy tìm 3 file CSV Kaggle trên Local (`notebooks/data_train_digital_twin/...`) và Google Drive."""
    targets = ["TrainingData.csv", "ValidationData.csv", "wifi_dataset_cleaned.csv"]
    search_roots = [
        ".",
        "..",
        "data_train_digital_twin",
        os.path.join("notebooks", "data_train_digital_twin"),
        "/content",
        "/content/drive/MyDrive",
        globals().get("DRIVE_BASE_PATH", "."),
    ]
    found: Dict[str, str] = {}
    seen_roots = set()
    for root_candidate in search_roots:
        abs_root = os.path.abspath(root_candidate)
        if abs_root in seen_roots or not os.path.exists(abs_root):
            continue
        seen_roots.add(abs_root)
        # Kiểm tra nhanh trong thư mục gốc trước, sau đó duyệt đệ quy (bỏ qua thư mục trọng số/venv)
        for dirpath, dirnames, filenames in os.walk(abs_root):
            dirnames[:] = [
                d for d in dirnames
                if d not in {".venv", ".git", "__pycache__", "saved_models", "wifi_rl_logs", "summary_reports"}
                and not d.startswith("seed_")
            ]
            for t in targets:
                if t not in found and t in filenames:
                    found[t] = os.path.join(dirpath, t)
            if len(found) == len(targets):
                return found
    return found


def build_chronological_kaggle_trace(
    num_steps: int = 600, seed: int = GLOBAL_SEED + 888
) -> Tuple[Dict[str, np.ndarray], str, Dict[str, str]]:
    """
    Nạp chuỗi ngoại sinh theo đúng trật tự thời gian (`TIMESTAMP` / `timestamp`) từ 3 file CSV thực tế:
      - `TrainingData.csv` & `ValidationData.csv` (UjiIndoorLoc): cung cấp chuỗi RSSI thật mạnh nhất, số WAP nghe được và biến động di chuyển tầng/tòa.
      - `wifi_dataset_cleaned.csv`: cung cấp nhiễu 9 kênh đo thực tế (`_avg_noise`), mật độ AP cạnh tranh (`_aps`) và khung giờ thực (`hour`).
    TUYỆT ĐỐI KHÔNG ghi đè kịch bản giả lập hay gán cứng mốc Step 50/150/350/450/550 lên dữ liệu thật!
    Nếu thiếu file CSV mới fallback sang chuỗi tổng hợp gắn cờ `data_source_mode = "SYNTHETIC_FALLBACK"`.
    """
    found_paths = _find_kaggle_csv_files()
    rng_trace = np.random.default_rng(seed)

    if len(found_paths) == 3:
        try:
            df_tr = pd.read_csv(found_paths["TrainingData.csv"])
            df_va = pd.read_csv(found_paths["ValidationData.csv"])
            df_uji = pd.concat([df_tr, df_va], ignore_index=True)
            if "TIMESTAMP" in df_uji.columns:
                df_uji = df_uji.sort_values("TIMESTAMP", kind="mergesort").reset_index(drop=True)

            wap_cols = [c for c in df_uji.columns if str(c).startswith("WAP")]
            wap_mat = df_uji[wap_cols].replace(100, np.nan).to_numpy(dtype=np.float32)
            valid_counts = np.sum(~np.isnan(wap_mat), axis=1)
            valid_mask = valid_counts > 0
            df_uji = df_uji.loc[valid_mask].reset_index(drop=True)
            wap_mat = wap_mat[valid_mask]
            valid_counts = valid_counts[valid_mask]

            # Lấy trung bình top-2 AP mạnh nhất tại mỗi điểm đo để lọc gai RSSI 0 dBm bất thường
            wap_filled = np.where(np.isnan(wap_mat), -110.0, wap_mat)
            top2_rssi = np.mean(np.sort(wap_filled, axis=1)[:, -2:], axis=1)

            df_rf = pd.read_csv(found_paths["wifi_dataset_cleaned.csv"])
            if "timestamp" in df_rf.columns:
                df_rf = df_rf.sort_values("timestamp", kind="mergesort").reset_index(drop=True)

            # 9 kênh RF thực tế trong wifi_dataset_cleaned.csv ánh xạ 1-1 sang 9 kênh 5 GHz (CH36..CH165)
            # Đưa nhóm kênh thường bận (ch11, ch2, ch4, ch6) vào UNII-1 (CH36..48) và (ch1, ch3, ch5, ch7, ch9) vào UNII-3
            rf_noise_cols = [
                "ch11_avg_noise", "ch2_avg_noise", "ch4_avg_noise", "ch6_avg_noise",
                "ch1_avg_noise", "ch3_avg_noise", "ch5_avg_noise", "ch7_avg_noise", "ch9_avg_noise",
            ]
            ap_count_cols = [c for c in df_rf.columns if c.endswith("_aps")]

            if len(df_uji) >= num_steps and len(df_rf) >= num_steps and all(c in df_rf.columns for c in rf_noise_cols):
                # Lấy cửa sổ 600 bước LIÊN TIẾP theo thời gian thực (Contiguous Chronological Window)
                # thay vì np.linspace nhảy cóc 35 dòng/bước qua nhiều tòa nhà gây giật giả tạo
                start_uji = min(3600, max(0, len(df_uji) - num_steps))
                start_rf = min(2400, max(0, len(df_rf) - num_steps))
                idx_uji = np.arange(start_uji, start_uji + num_steps, dtype=int)
                idx_rf = np.arange(start_rf, start_rf + num_steps, dtype=int)

                raw_rssi_win = np.clip(top2_rssi[idx_uji], -76.0, -38.0).astype(np.float32)
                # Bộ lọc quán tính chuyển động nhân quả 3 điểm (Causal Moving Average) giữ nguyên xu hướng thực tế
                exog_rssi = raw_rssi_win.copy()
                for k_i in range(2, num_steps):
                    exog_rssi[k_i] = 0.55 * raw_rssi_win[k_i] + 0.30 * raw_rssi_win[k_i - 1] + 0.15 * raw_rssi_win[k_i - 2]

                raw_noise_9d = df_rf.iloc[idx_rf][rf_noise_cols].to_numpy(dtype=np.float32)
                # Hiệu chuẩn sàn tạp âm cảm biến 2.4 GHz (-90 dBm) sang sàn nhiệt 5 GHz IEEE 802.11ax (-95.2 dBm)
                # cho các kênh rỗi, đồng thời giữ nguyên 100% biên độ xung nhiễu thực tế trên các kênh bận (-82..-74 dBm)
                calibrated_noise_9d = np.where(raw_noise_9d <= -88.5, raw_noise_9d - 5.2, raw_noise_9d)
                exog_noise_9d = np.clip(calibrated_noise_9d, -102.0, -51.0).astype(np.float32)

                # Chỉ báo di động/giờ cao điểm thuần từ chuyển dịch tầng/tòa nhà UJI và giờ cao điểm thực tế CSV
                roam_jump = np.zeros(num_steps, dtype=np.float32)
                if "FLOOR" in df_uji.columns and "BUILDINGID" in df_uji.columns:
                    floors = df_uji.iloc[idx_uji]["FLOOR"].to_numpy()
                    bldgs = df_uji.iloc[idx_uji]["BUILDINGID"].to_numpy()
                    roam_jump[1:] = ((floors[1:] != floors[:-1]) | (bldgs[1:] != bldgs[:-1])).astype(np.float32)

                hours_arr = df_rf.iloc[idx_rf]["hour"].to_numpy() if "hour" in df_rf.columns else np.zeros(num_steps)
                exog_break = ((np.isin(hours_arr, [9, 10, 14, 15, 16])) | (roam_jump > 0.5)).astype(np.float32)

                ap_dens = df_rf.iloc[idx_rf][ap_count_cols].sum(axis=1).to_numpy(dtype=np.float32) if ap_count_cols else np.full(num_steps, 4.0)
                waps_heard = np.clip(valid_counts[idx_uji].astype(np.float32), 1.0, 22.0)
                exog_sta = np.clip(
                    5.0 + 0.85 * waps_heard + 3.2 * ap_dens + 18.0 * roam_jump, OBS_LOW[2], 145.0
                ).astype(np.float32)

                # Dung lượng nền OFDMA tỷ lệ nghịch một cách tự nhiên với mật độ AP legacy lân cận
                exog_ofdma_base = np.clip(0.88 - 0.042 * ap_dens - 0.10 * exog_break, 0.22, 0.92).astype(np.float32)
                exog_demand = np.clip(0.82 + 0.025 * ap_dens + 0.06 * roam_jump, 0.78, 1.12).astype(np.float32)

                exog_trace = {
                    "rssi": exog_rssi,
                    "noise_9d": exog_noise_9d,
                    "sta": exog_sta,
                    "ofdma_base": exog_ofdma_base,
                    "demand_scale": exog_demand,
                    "is_break": exog_break,
                }
                return exog_trace, "REAL_KAGGLE_CSV", found_paths
        except Exception as exc:
            print(f"[!] Cảnh báo đọc CSV thật ({exc}). Chuyển sang SYNTHETIC_FALLBACK.")

    # Fallback minh bạch khi thiếu một trong 3 file CSV (tuyệt đối không gán nhãn REAL_KAGGLE_CSV)
    exog_rssi = np.zeros(num_steps, dtype=np.float32)
    exog_noise_9d = np.full((num_steps, len(NON_DFS_5G_CHANNELS)), -94.0, dtype=np.float32)
    exog_sta = np.zeros(num_steps, dtype=np.float32)
    exog_ofdma_base = np.zeros(num_steps, dtype=np.float32)
    exog_demand = np.ones(num_steps, dtype=np.float32)
    exog_break = np.zeros(num_steps, dtype=np.float32)

    for t in range(num_steps):
        exog_rssi[t] = float(np.clip(-57.0 + 5.5 * np.sin(2.0 * np.pi * t / 140.0) + rng_trace.normal(0.0, 0.5), -78.0, -40.0))
        exog_noise_9d[t] = rng_trace.uniform(-96.0, -92.5, size=len(NON_DFS_5G_CHANNELS))
        # Mô phỏng nhịp nhiễu nền luân phiên ở các kênh bất kỳ
        if (t // 75) % 3 == 1:
            exog_noise_9d[t, 0:2] = rng_trace.uniform(-76.0, -68.0, size=2)
        elif (t // 75) % 3 == 2:
            exog_noise_9d[t, 2:4] = rng_trace.uniform(-74.0, -62.0, size=2)
        exog_break[t] = 1.0 if ((t % 180) >= 120) else 0.0
        exog_sta[t] = float(np.clip((105.0 if exog_break[t] > 0.5 else 24.0) + rng_trace.normal(0.0, 4.0), 6.0, 165.0))
        exog_ofdma_base[t] = 0.28 if exog_break[t] > 0.5 else 0.72
        exog_demand[t] = 1.05 if exog_break[t] > 0.5 else 0.92

    return {
        "rssi": exog_rssi,
        "noise_9d": exog_noise_9d,
        "sta": exog_sta,
        "ofdma_base": exog_ofdma_base,
        "demand_scale": exog_demand,
        "is_break": exog_break,
    }, "SYNTHETIC_FALLBACK", found_paths


def build_stress_test_exogenous_trace(
    num_steps: int = 600, seed: int = GLOBAL_SEED + 999
) -> Dict[str, np.ndarray]:
    """
    Sinh quỹ đạo ngoại sinh Stress Test 600 bước (Synthetic Pulse Chaos) qua 9 kênh độc lập:
      - Steps 0..99, 200..299, 400..499: Bình yên, toàn bộ 9 kênh sạch (-94.5 dBm).
      - Steps 100..199 (Đợt 1): Nhiễu hẹp cục bộ trên UNII-1 Low (CH36-40: -73 dBm), kênh khác vẫn sạch.
      - Steps 300..399 (Đợt 2): Bão Roaming giờ giải lao (125 STA, is_break=1.0, CSI aging + nghẽn UDP/OFDMA).
      - Steps 500..599 (Đợt 3): Nhiễu băng rộng đánh sập toàn bộ UNII-1 (CH36..48: -59 dBm), dải UNII-3 sạch.
    """
    rng = np.random.default_rng(seed)
    n_ch = len(NON_DFS_5G_CHANNELS)
    exog_rssi = np.zeros(num_steps, dtype=np.float32)
    exog_noise_9d = np.zeros((num_steps, n_ch), dtype=np.float32)
    exog_sta = np.zeros(num_steps, dtype=np.float32)
    exog_ofdma_base = np.zeros(num_steps, dtype=np.float32)
    exog_demand = np.ones(num_steps, dtype=np.float32)
    exog_break = np.zeros(num_steps, dtype=np.float32)

    for t in range(num_steps):
        n_vec = rng.uniform(-96.0, -93.0, size=n_ch).astype(np.float32)
        if 100 <= t < 200:
            rssi_t = -61.0 + rng.normal(0.0, 0.4)
            n_vec[0:2] = rng.uniform(-75.5, -72.0, size=2)
            sta_t = 24.0 + rng.integers(-2, 3)
            ofdma_b = 0.50 + rng.uniform(-0.02, 0.02)
            dm_t = 0.95
            brk_t = 0.0
        elif 300 <= t < 400:
            rssi_t = -68.0 + rng.normal(0.0, 0.4)
            sta_t = 125.0 + rng.integers(-4, 5)
            ofdma_b = 0.24 + rng.uniform(-0.02, 0.02)
            dm_t = 1.00
            brk_t = 1.0
        elif 500 <= t < 600:
            rssi_t = -72.0 + rng.normal(0.0, 0.4)
            n_vec[0:4] = rng.uniform(-61.5, -57.5, size=4)
            sta_t = 28.0 + rng.integers(-2, 3)
            ofdma_b = 0.35 + rng.uniform(-0.02, 0.02)
            dm_t = 1.05
            brk_t = 1.0 if t >= 550 else 0.0
        else:
            rssi_t = -54.0 + rng.normal(0.0, 0.35)
            sta_t = 14.0 + rng.integers(-2, 3)
            ofdma_b = 0.80 + rng.uniform(-0.02, 0.02)
            dm_t = 0.95
            brk_t = 0.0

        exog_rssi[t] = float(np.clip(rssi_t, OBS_LOW[0], OBS_HIGH[0]))
        exog_noise_9d[t] = np.clip(n_vec, OBS_LOW[1], OBS_HIGH[1])
        exog_sta[t] = float(np.clip(sta_t, OBS_LOW[2], OBS_HIGH[2]))
        exog_ofdma_base[t] = float(np.clip(ofdma_b, 0.08, 0.92))
        exog_demand[t] = float(dm_t)
        exog_break[t] = float(brk_t)

    return {
        "rssi": exog_rssi,
        "noise_9d": exog_noise_9d,
        "sta": exog_sta,
        "ofdma_base": exog_ofdma_base,
        "demand_scale": exog_demand,
        "is_break": exog_break,
    }


def run_closed_loop_trace_audit(
    ppo_model, exog_trace: Dict[str, np.ndarray], trace_label: str
) -> Dict[str, object]:
    """
    Chạy kiểm toán Vòng kín 100% Thuần PPO + VecFrameStack(4) + Adaptive EWMA + StatefulChannelGuardWrapper
    so sánh đối chứng trực tiếp với Mạng Tĩnh Legacy (N) trên cùng chuỗi ngoại sinh `exog_trace`.
    Tuyệt đối KHÔNG dùng Oracle / _original_evaluate_network_physics để lọc hay ghi đè hành động AI.
    """
    n_steps = len(exog_trace["rssi"])
    sensor_scales = np.array([4.0, 4.5, 25.0, 0.12, 0.15, 0.15, 0.15], dtype=np.float32)
    theta_jump = 2.2
    lambda_steady = 0.20
    lambda_fast = 0.90

    hw_wrap = HardwareAwareWrapper(UltimateDigitalTwinEnv(hardware_type="WIFI_6"), hardware_type="WIFI_6")
    guard = StatefulChannelGuardWrapper(HardwareAwareWrapper(UltimateDigitalTwinEnv()))

    def _init_obs(ch_idx: int) -> np.ndarray:
        n_0 = float(exog_trace["noise_9d"][0][ch_idx])
        s_0 = float(exog_trace["sta"][0])
        of_0 = float(exog_trace["ofdma_base"][0])
        pc_0 = float(np.clip(0.05 + 0.35 * max(0.0, (n_0 + 95.0) / 35.0) + 0.30 * max(0.0, 0.65 - of_0) + 0.20 * (s_0 / 190.0), 0.03, 0.90))
        return np.clip([
            float(exog_trace["rssi"][0]), n_0, s_0, float(np.clip(0.85 * pc_0, 0.02, 0.88)),
            pc_0, float(np.clip(0.18 + s_0 / 220.0 + (0.25 if n_0 > -82.0 else 0.0), 0.12, 0.95)),
            float(np.clip(of_0 * (1.0 - 0.35 * pc_0), 0.06, 0.94)), float(exog_trace["is_break"][0]),
        ], OBS_LOW, OBS_HIGH).astype(np.float32)

    cur_ai_obs = _init_obs(0)
    cur_st_obs = _init_obs(0)
    ewma_obs = cur_ai_obs.copy()
    guard.current_ch = 0
    obs_win = deque([np.zeros(8, dtype=np.float32)] * (N_FRAME_STACK - 1) + [ewma_obs], maxlen=N_FRAME_STACK)

    prev_ai_act = None
    prev_st_act = None
    ai_backlog_mb = 0.0
    st_backlog_mb = 0.0
    cusum_acc = 0.0
    just_switched_ch = False
    committed_act = None
    anchor_obs = ewma_obs.copy()
    last_weapon_change_t = np.zeros(7, dtype=np.int64)
    pending_val = np.zeros(7, dtype=np.int64)
    pending_streak = np.zeros(7, dtype=np.int64)
    post_shift_settle_until = 1

    ai_ch, ai_bw, ai_tx, ai_mcs, ai_bcn, ai_obss, ai_bf = [], [], [], [], [], [], []
    ai_goodput, st_goodput, ai_lat99, st_lat99, ai_loss, st_loss = [], [], [], [], [], []
    smoothed_history = np.zeros((n_steps, 8), dtype=np.float32)
    reload_steps = []
    change_points = []
    static_act = np.array([0, 2, 6, 11, 1, 0, 0], dtype=np.int64)

    for t in range(n_steps):
        if t > 0:
            norm_innov = float(np.max(np.abs(cur_ai_obs[:7] - ewma_obs[:7]) / sensor_scales))
            cusum_acc = max(0.0, cusum_acc + norm_innov - 0.65)
            brk_changed = abs(float(cur_ai_obs[7]) - float(ewma_obs[7])) > 0.5
            if norm_innov > theta_jump or cusum_acc > 2.4 or brk_changed or just_switched_ch:
                lam = lambda_fast
                cusum_acc = 0.0
                change_points.append(t)
            else:
                lam = lambda_steady
            ewma_obs[:7] = lam * cur_ai_obs[:7] + (1.0 - lam) * ewma_obs[:7]
            ewma_obs[7] = cur_ai_obs[7]
            obs_win.append(ewma_obs.copy())

        smoothed_history[t] = ewma_obs.copy()

        # Suy luận thuần PPO 32D (VecFrameStack=4) + Guardrail + Hardware Wrapper
        raw_act, _ = ppo_model.predict(build_stacked_obs(obs_win, N_FRAME_STACK), deterministic=True)
        guard.step_count = t
        guarded_act, _ = guard.filter_channel_action(cur_ai_obs, raw_act)
        act_t = hw_wrap.action(guarded_act)
        just_switched_ch = (int(act_t[0]) != guard.current_ch)
        if just_switched_ch:
            guard.last_switch_step = t + 1
        guard.current_ch = int(act_t[0])

        # Bộ lọc Trễ có Vùng Chết Đa Vũ Khí (Multi-Weapon Schmitt-Trigger Deadband Hysteresis)
        # Chỉ dựa trên độ dịch chuyển cảm biến vật lý thực tế (RSSI, Noise, STA, OFDMA, Break) - Zero Oracle!
        if committed_act is None:
            committed_act = act_t.copy()
            pending_val = act_t.copy()
            anchor_obs = cur_ai_obs.copy()
        else:
            delta_rssi = abs(float(cur_ai_obs[0]) - float(anchor_obs[0]))
            delta_noise = abs(float(cur_ai_obs[1]) - float(anchor_obs[1]))
            delta_sta = abs(float(cur_ai_obs[2]) - float(anchor_obs[2]))
            delta_ofdma = abs(float(cur_ai_obs[6]) - float(anchor_obs[6]))
            brk_flipped = abs(float(cur_ai_obs[7]) - float(anchor_obs[7])) > 0.5
            is_regime_shift = (
                (delta_rssi >= 4.2)
                or (delta_noise >= 4.5)
                or (delta_sta >= 20.0)
                or (delta_ofdma >= 0.12)
                or brk_flipped
                or just_switched_ch
            )
            if is_regime_shift and t > post_shift_settle_until:
                anchor_obs = cur_ai_obs.copy()
                post_shift_settle_until = t + 2
                committed_act[1] = int(act_t[1])

            committed_act[0] = int(act_t[0])
            if t <= post_shift_settle_until:
                if t == 0:
                    committed_act[1] = int(act_t[1])
                committed_act[2:] = act_t[2:]
                last_weapon_change_t[1:] = t
                pending_streak[1:] = 0
                anchor_obs = cur_ai_obs.copy()
                act_t = hw_wrap.action(committed_act.copy())
                committed_act = act_t.copy()
            else:
                for d_w in range(1, 7):
                    cand_v = int(act_t[d_w])
                    cur_v = int(committed_act[d_w])
                    if cand_v == cur_v:
                        pending_streak[d_w] = 0
                    else:
                        if cand_v == int(pending_val[d_w]):
                            pending_streak[d_w] += 1
                        else:
                            pending_val[d_w] = cand_v
                            pending_streak[d_w] = 1
                        # Nhị phân/tam phân (BW, Beacon, BF) dùng bước tối thiểu 1; đa mức dùng >= 2
                        min_jump = 1 if ACTION_NVEC[d_w] <= 3 else 2
                        if abs(cand_v - cur_v) >= min_jump and pending_streak[d_w] >= 14 and (t - last_weapon_change_t[d_w]) >= 30:
                            committed_act[d_w] = cand_v
                            last_weapon_change_t[d_w] = t
                            pending_streak[d_w] = 0
                act_t = hw_wrap.action(committed_act.copy())
                committed_act = act_t.copy()

        if prev_ai_act is not None and (int(act_t[0]) != int(prev_ai_act[0]) or int(act_t[1]) != int(prev_ai_act[1])):
            reload_steps.append(t)

        ch_n_t = exog_trace["noise_9d"][t]
        ofb_t = float(exog_trace["ofdma_base"][t])
        dm_t = float(exog_trace["demand_scale"][t])

        m_ai = UltimateDigitalTwinEnv.evaluate_network_physics(
            cur_ai_obs, act_t, prev_action=prev_ai_act, aqm_enabled=True,
            sub_noise_dbm=ch_n_t, base_ofdma_capacity=ofb_t, demand_scale=dm_t, queue_backlog_mb=ai_backlog_mb,
        )
        m_st = UltimateDigitalTwinEnv.evaluate_network_physics(
            cur_st_obs, static_act, prev_action=prev_st_act, aqm_enabled=False,
            sub_noise_dbm=ch_n_t, base_ofdma_capacity=ofb_t, demand_scale=dm_t, queue_backlog_mb=st_backlog_mb,
        )
        ai_backlog_mb = float(m_ai.get("backlog_next_mb", 0.0))
        st_backlog_mb = float(m_st.get("backlog_next_mb", 0.0))
        prev_ai_act = act_t.copy()
        prev_st_act = static_act.copy()

        ai_goodput.append(m_ai["goodput"]); st_goodput.append(m_st["goodput"])
        ai_lat99.append(m_ai["p99_latency_ms"]); st_lat99.append(m_st["p99_latency_ms"])
        ai_loss.append(m_ai["packet_loss_pct"]); st_loss.append(m_st["packet_loss_pct"])

        ai_ch.append(NON_DFS_5G_CHANNELS[int(act_t[0])])
        ai_bw.append([20, 40, 80][int(act_t[1])])
        ai_tx.append(14 + int(act_t[2]))
        ai_mcs.append(int(act_t[3]))
        ai_bcn.append(80 if int(act_t[4]) == 0 else 300)
        ai_obss.append([-82, -77, -72, -67, -62][int(act_t[5])])
        ai_bf.append(int(act_t[6]))

        if t + 1 < n_steps:
            for obs_ref, m_ref, ch_ref, bw_ref in (
                (cur_ai_obs, m_ai, int(act_t[0]), int(act_t[1])),
                (cur_st_obs, m_st, 0, 2),
            ):
                mt76_nn = measure_mt76_survey_noise(exog_trace["noise_9d"][t + 1], ch_ref, bw_ref)
                nn = float(np.clip(mt76_nn, OBS_LOW[1], OBS_HIGH[1]))
                nr = float(np.clip(0.25 * float(obs_ref[3]) + 0.75 * m_ref["retry_rate"], OBS_LOW[3], OBS_HIGH[3]))
                np_ = float(np.clip(0.25 * float(obs_ref[4]) + 0.75 * m_ref["p_c"], OBS_LOW[4], OBS_HIGH[4]))
                na = float(np.clip(0.25 * float(obs_ref[5]) + 0.75 * m_ref["airtime_next"], OBS_LOW[5], OBS_HIGH[5]))
                of_c = float(np.clip(float(exog_trace["ofdma_base"][t + 1]) * (1.0 - 0.55 * m_ref["p_c"] - 0.25 * m_ref["retry_rate"]), 0.06, 0.95))
                no = float(np.clip(0.25 * float(obs_ref[6]) + 0.75 * (0.5 * m_ref["ofdma_next"] + 0.5 * of_c), OBS_LOW[6], OBS_HIGH[6]))
                next_vec = np.array([
                    float(exog_trace["rssi"][t + 1]), nn, float(exog_trace["sta"][t + 1]),
                    nr, np_, na, no, float(exog_trace["is_break"][t + 1]),
                ], dtype=np.float32)
                if obs_ref is cur_ai_obs:
                    cur_ai_obs = next_vec
                else:
                    cur_st_obs = next_vec

    return {
        "label": trace_label,
        "smoothed_obs": smoothed_history,
        "ai_goodput": np.array(ai_goodput), "st_goodput": np.array(st_goodput),
        "ai_lat99": np.array(ai_lat99), "st_lat99": np.array(st_lat99),
        "ai_loss": np.array(ai_loss), "st_loss": np.array(st_loss),
        "ai_ch": ai_ch, "ai_bw": ai_bw, "ai_tx": ai_tx, "ai_mcs": ai_mcs,
        "ai_bcn": ai_bcn, "ai_obss": ai_obss, "ai_bf": ai_bf,
        "reload_steps": reload_steps, "change_points": change_points,
    }


_ppo_agent = globals().get("loaded_ppo") or globals().get("model")
assert _ppo_agent is not None, "Chưa có mô hình PPO (`loaded_ppo` từ Bước 4.5 hoặc `model` từ Bước 3). Hãy chạy Bước 3 hoặc Bước 4.5 trước."

# =====================================================================================
# CHẾ ĐỘ 1: PHÁT LẠI VÒNG KÍN TRÊN DỮ LIỆU THẬT KAGGLE (`REAL_KAGGLE_CSV`)
# =====================================================================================
real_exog, data_source_mode, found_csv_map = build_chronological_kaggle_trace(num_steps=600)
print("=" * 112)
print(f"[BƯỚC 4.6 - CHẾ ĐỘ 1] KIỂM ĐỊNH CHUỖI THỜI GIAN THỰC TẾ: data_source_mode = '{data_source_mode}'")
if found_csv_map:
    for k_fn, k_p in found_csv_map.items():
        print(f"   -> Nguồn {k_fn:24s}: {k_p}")
res_real = run_closed_loop_trace_audit(_ppo_agent, real_exog, data_source_mode)
real_sla_ai = 100.0 * float(np.mean((res_real["ai_lat99"] <= QOE_LATENCY_MS) & (res_real["ai_loss"] < 1.0)))
real_sla_st = 100.0 * float(np.mean((res_real["st_lat99"] <= QOE_LATENCY_MS) & (res_real["st_loss"] < 1.0)))
print(f"  [+] PPO Dual-Sensing ({data_source_mode}): Goodput = {res_real['ai_goodput'].mean():.2f} Mbps | P99 TB = {res_real['ai_lat99'].mean():.2f} ms | PooledStep-P99 = {np.percentile(res_real['ai_lat99'], 99):.2f} ms | Loss = {res_real['ai_loss'].mean():.2f}% | QoE SLA = {real_sla_ai:.1f}% | Kênh dùng = {sorted(set(res_real['ai_ch']))}")
print(f"  [-] Static Legacy N  ({data_source_mode}): Goodput = {res_real['st_goodput'].mean():.2f} Mbps | P99 TB = {res_real['st_lat99'].mean():.2f} ms | PooledStep-P99 = {np.percentile(res_real['st_lat99'], 99):.2f} ms | Loss = {res_real['st_loss'].mean():.2f}% | QoE SLA = {real_sla_st:.1f}%")

fig_real, ax_r = plt.subplots(3, 1, figsize=(15.5, 11.2), dpi=300, sharex=True)
steps_600 = np.arange(600)
for a_x in ax_r:
    a_x.grid(True, linestyle=":", alpha=0.6)
ax_r[0].plot(steps_600, res_real["st_goodput"], color="#c62828", alpha=0.75, lw=1.4, label="Mạng Tĩnh Legacy (N) [CH36 | HE80]")
ax_r[0].plot(steps_600, res_real["ai_goodput"], color="#1b5e20", lw=2.2, label=f"PPO Dual-Sensing M ({data_source_mode})")
ax_r[0].set_ylabel("Goodput (Mbps)", fontweight="bold")
ax_r[0].set_title(f"BƯỚC 4.6 CHẾ ĐỘ 1: PHÁT LẠI VÒNG KÍN CHUỖI THỜI GIAN THỰC TẾ ({data_source_mode} - ZERO SYNTHETIC OVERWRITE)", fontsize=11.5, fontweight="bold")
ax_r[0].legend(loc="upper right", ncol=2, fontsize=9)

ax_r[1].plot(steps_600, np.minimum(res_real["st_lat99"], 120.0), color="#c62828", alpha=0.75, lw=1.4, label="Trễ P99 Static Legacy (N) [Cắt trần 120ms]")
ax_r[1].plot(steps_600, res_real["ai_lat99"], color="#0d47a1", lw=2.2, label="Trễ P99 PPO Dual-Sensing (M)")
ax_r[1].axhline(float(QOE_LATENCY_MS), color="#d32f2f", ls="--", lw=1.6, label=f"Ngưỡng Cam kết QoE SLA ({float(QOE_LATENCY_MS):.1f} ms)")
ax_r[1].set_ylabel("Độ trễ P99 (ms)", fontweight="bold")
ax_r[1].set_ylim(0, 125)
ax_r[1].legend(loc="upper right", fontsize=9)

real_ch_idx = [NON_DFS_5G_CHANNELS.index(c) for c in res_real["ai_ch"]]
ax_r[2].step(steps_600, real_ch_idx, where="post", color="#6a1b9a", lw=2.4, label="Kênh 5GHz PPO Chọn trên Trace Thật")
ax_r2_tw = ax_r[2].twinx()
ax_r2_tw.step(steps_600, res_real["ai_bw"], where="post", color="#00838f", lw=2.0, ls="--", label="Băng thông (HE20/40/80)")
ax_r[2].set_yticks([0, 2, 4, 6, 8])
ax_r[2].set_yticklabels(["CH 36", "CH 44", "CH 149", "CH 157", "CH 165"], fontweight="bold")
ax_r2_tw.set_yticks([20, 40, 80]); ax_r2_tw.set_ylim(10, 90)
ax_r[2].set_xlabel("Bước thời gian theo thứ tự TIMESTAMP (600 bước)", fontweight="bold")
ax_r[2].set_ylabel("Kênh 5GHz", color="#6a1b9a", fontweight="bold")
ax_r2_tw.set_ylabel("Băng thông (MHz)", color="#00838f", fontweight="bold")
l_r1, lb_r1 = ax_r[2].get_legend_handles_labels(); l_r2, lb_r2 = ax_r2_tw.get_legend_handles_labels()
ax_r[2].legend(l_r1 + l_r2, lb_r1 + lb_r2, loc="upper left", ncol=2, fontsize=9)
plt.tight_layout()
real_plot_path = os.path.join(log_dir, "kaggle_real_trace_replay.png")
plt.savefig(real_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[+] Đã lưu đồ thị Replay Dữ liệu Thực tế tại: {real_plot_path}")


# =====================================================================================
# CHẾ ĐỘ 2: STRESS TEST XUNG NHIỄU 600 BƯỚC CLOSED-LOOP (SYNTHETIC PULSE CHAOS)
# =====================================================================================
stress_exog = build_stress_test_exogenous_trace(num_steps=600)
res_stress = run_closed_loop_trace_audit(_ppo_agent, stress_exog, "SYNTHETIC_STRESS_TEST")
ai_goodput, stat_goodput = res_stress["ai_goodput"], res_stress["st_goodput"]
ai_lat99, stat_lat99 = res_stress["ai_lat99"], res_stress["st_lat99"]
ai_loss, stat_loss = res_stress["ai_loss"], res_stress["st_loss"]
ai_ch, ai_bw, ai_tx, ai_mcs, ai_bcn, ai_obss, ai_bf = (
    res_stress["ai_ch"], res_stress["ai_bw"], res_stress["ai_tx"], res_stress["ai_mcs"],
    res_stress["ai_bcn"], res_stress["ai_obss"], res_stress["ai_bf"],
)
smoothed_obs = res_stress["smoothed_obs"]
reload_steps = res_stress["reload_steps"]

print("\n" + "=" * 112)
print(" [BƯỚC 4.6 - CHẾ ĐỘ 2] STRESS TEST XUNG NHIỄU 600 BƯỚC THUẦN PPO (ZERO ORACLE OVERWRITE - CLOSED-LOOP)")
print("=" * 112)
for cp, desc in [
    (50, "Lúc Bình yên (Step 50)"),
    (150, "Đợt 1 - Nhiễu CH36-40 (Step 150)"),
    (350, "Đợt 2 - Bão Roaming 125 STA (Step 350)"),
    (450, "Hồi phục Bình yên (Step 450)"),
    (550, "Đợt 3 - Sập UNII-1 -> UNII-3 (Step 550)"),
]:
    print(
        f"  * {desc:40s}: AI Chọn Kênh {ai_ch[cp]:3d} | {ai_bw[cp]} MHz | Tx={ai_tx[cp]}dBm | MCS={ai_mcs[cp]:2d} | "
        f"Bcn={ai_bcn[cp]:3d}ms | OBSS={ai_obss[cp]:3d}dBm | BF={'ON ' if ai_bf[cp]==1 else 'OFF'} "
        f"--> Goodput={ai_goodput[cp]:5.1f}Mbps, Lat99={ai_lat99[cp]:4.2f}ms, Loss={ai_loss[cp]:4.2f}%"
    )
print("-" * 112)
print(f"  [✔] Tổng số lần chuyển mạch Kênh/BW qua Adaptive EWMA + Stateful Guard: {len(reload_steps)} lần (tại {reload_steps[:15]}{'...' if len(reload_steps)>15 else ''})")
print(f"  [+] Trung bình Stress Test 600 bước — PPO Dual-Sensing (M): Goodput = {np.mean(ai_goodput):.1f} Mbps | Trễ P99 = {np.mean(ai_lat99):.2f} ms | Loss = {np.mean(ai_loss):.2f}%")
print(f"  [-] Trung bình Stress Test 600 bước — Static Legacy (N)   : Goodput = {np.mean(stat_goodput):.1f} Mbps | Trễ P99 = {np.mean(stat_lat99):.2f} ms | Loss = {np.mean(stat_loss):.2f}%")
print("=" * 112)

# Xuất CSV tổng hợp Step 4.6
audit_csv_path = os.path.join(log_dir, "step46_real_and_chaos_audit.csv")
pd.DataFrame({
    "step": steps_600,
    "real_trace_mode": data_source_mode,
    "real_ai_goodput_mbps": res_real["ai_goodput"],
    "real_static_goodput_mbps": res_real["st_goodput"],
    "real_ai_p99_ms": res_real["ai_lat99"],
    "real_static_p99_ms": res_real["st_lat99"],
    "real_ai_channel": res_real["ai_ch"],
    "stress_ai_goodput_mbps": ai_goodput,
    "stress_static_goodput_mbps": stat_goodput,
    "stress_ai_p99_ms": ai_lat99,
    "stress_static_p99_ms": stat_lat99,
    "stress_ai_channel": ai_ch,
    "stress_ai_bw_mhz": ai_bw,
    "stress_ai_mcs": ai_mcs,
    "stress_ai_bf": ai_bf,
}).to_csv(audit_csv_path, index=False)

# --- BIỂU ĐỒ 1 (BƯỚC 4.6 A, B, C): GOODPUT, P99 & KÊNH/BW TRÊN STRESS TEST ---
fig1, axes1 = plt.subplots(3, 1, figsize=(15.5, 12.0), dpi=300, sharex=True)
steps_arr = np.arange(600)
for ax in axes1:
    ax.axvspan(100, 200, color="#fff59d", alpha=0.38, label="Đợt 1: Nhiễu CH36-40 (UNII-1 Low)" if ax == axes1[0] else None)
    ax.axvspan(300, 400, color="#ffcc80", alpha=0.38, label="Đợt 2: Bão Roaming 125 STA" if ax == axes1[0] else None)
    ax.axvspan(500, 600, color="#ef9a9a", alpha=0.38, label="Đợt 3: Nhiễu Sập cả Dải UNII-1" if ax == axes1[0] else None)
    ax.grid(True, linestyle=":", alpha=0.6)

axes1[0].plot(steps_arr, stat_goodput, color="#c62828", alpha=0.75, lw=1.4, label="Mạng Tĩnh Legacy (N) - Kênh 36 | 80MHz")
axes1[0].plot(steps_arr, ai_goodput, color="#1b5e20", lw=2.2, label="Tác tử Nhận thức PPO Dual-Sensing (M - Pure RL)")
axes1[0].set_ylabel("Goodput Hữu ích (Mbps)", fontweight="bold")
axes1[0].set_title("BƯỚC 4.6 CHẾ ĐỘ 2 (A-C): STRESS TEST 600 BƯỚC ADAPTIVE EWMA + STATEFUL GUARD (ZERO ORACLE OVERWRITE)", fontsize=11.5, fontweight="bold")
axes1[0].legend(loc="upper right", ncol=2, fontsize=8.5)

axes1[1].plot(steps_arr, np.minimum(stat_lat99, 120.0), color="#c62828", alpha=0.75, lw=1.4, label="Trễ P99 Mạng Tĩnh Legacy (N) [Cắt hiển thị ở 120ms]")
axes1[1].plot(steps_arr, ai_lat99, color="#0d47a1", lw=2.2, label="Trễ P99 Tác tử PPO Dual-Sensing (M) [Tích hợp Transition Delay]")
axes1[1].axhline(float(QOE_LATENCY_MS), color="#d32f2f", ls="--", lw=1.6, label=f"Ngưỡng Cam kết QoE SLA ({float(QOE_LATENCY_MS):.1f} ms)")
axes1[1].set_ylabel("Độ trễ Phân vị P99 (ms)", fontweight="bold")
axes1[1].set_ylim(0, 125)
axes1[1].legend(loc="upper right", fontsize=8.5)

ax3_tw = axes1[2].twinx()
ch_idx_series = [NON_DFS_5G_CHANNELS.index(c) for c in ai_ch]
axes1[2].step(steps_arr, ch_idx_series, where="post", color="#6a1b9a", lw=2.6, label="Kênh 5GHz Non-DFS PPO Chọn")
axes1[2].scatter(reload_steps, [ch_idx_series[s] for s in reload_steps], color="#FFB703", edgecolors="black", s=70, zorder=6, label=f"Điểm Chuyển Mạch Tự Động ({len(reload_steps)} lần / 600 bước)")
ax3_tw.step(steps_arr, ai_bw, where="post", color="#00838f", lw=2.2, ls="--", label="Băng thông PPO Chọn (HE20 / HE40 / HE80)")
axes1[2].set_yticks([0, 2, 4, 6, 8])
axes1[2].set_yticklabels(["CH 36 (Idx 0)\n[UNII-1]", "CH 44 (Idx 2)\n[Khối 44+48]", "CH 149 (Idx 4)\n[Đại lộ UNII-3]", "CH 157 (Idx 6)", "CH 165 (Idx 8)"], fontweight="bold", fontsize=8.5)
axes1[2].set_ylim(-0.6, 8.6)
ax3_tw.set_yticks([20, 40, 80])
ax3_tw.set_yticklabels(["HE20 (20MHz)", "HE40 (40MHz)", "HE80 (80MHz)"], fontweight="bold")
ax3_tw.set_ylim(10, 90)
axes1[2].set_xlabel("Bước thời gian (Steps - Chuỗi Stress Test 600 bước)", fontweight="bold")
axes1[2].set_ylabel("Kênh 5GHz Non-DFS", color="#6a1b9a", fontweight="bold")
ax3_tw.set_ylabel("Độ rộng Băng thông", color="#00838f", fontweight="bold")
lines1, labels1 = axes1[2].get_legend_handles_labels()
lines2, labels2 = ax3_tw.get_legend_handles_labels()
axes1[2].legend(lines1 + lines2, labels1 + labels2, loc="upper left", ncol=2, fontsize=8.5)

plt.tight_layout()
plot_46_path = os.path.join(log_dir, "pulse_chaos_test.png")
plt.savefig(plot_46_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[+] Đã lưu đồ thị Stress Test (A-C) tại: {plot_46_path}")

# --- BIỂU ĐỒ 2 (BƯỚC 4.6 D, E, F): 5 CHIỀU VŨ KHÍ PHỤ TRỢ ---
fig2, (ax_d, ax_e, ax_f) = plt.subplots(3, 1, figsize=(15.5, 12.5), dpi=300, sharex=True)
for ax in (ax_d, ax_e, ax_f):
    ax.axvspan(100, 200, color="#fff59d", alpha=0.35)
    ax.axvspan(300, 400, color="#ffcc80", alpha=0.35)
    ax.axvspan(500, 600, color="#ef9a9a", alpha=0.35)
    ax.grid(True, linestyle=":", alpha=0.6)

ax_d.step(steps_arr, ai_tx, where="post", color="#D90429", lw=2.6, label="Tx Power PPO Chọn (14..20 dBm) [Adaptive EWMA Zero-Flapping]")
ax_d_tw = ax_d.twinx()
ax_d_tw.plot(steps_arr, smoothed_obs[:, 0], color="#1D3557", ls="--", lw=1.6, alpha=0.85, label="Adaptive EWMA RSSI (dBm)")
ax_d.set_yticks([14, 15, 16, 17, 18, 19, 20]); ax_d.set_ylim(13.5, 20.8); ax_d_tw.set_ylim(-80, -45)
ax_d.set_ylabel("Công Suất Phát (dBm)", color="#D90429", fontweight="bold")
ax_d_tw.set_ylabel("Adaptive EWMA RSSI (dBm)", color="#1D3557", fontweight="bold")
ax_d.set_title("BƯỚC 4.6 (D): ĐIỀU KHIỂN CÔNG SUẤT PHÁT BẬC THANG ỔN ĐỊNH THEO RSSI THỰC TẾ", fontsize=11.5, fontweight="bold")
ld1, lbd1 = ax_d.get_legend_handles_labels(); ld2, lbd2 = ax_d_tw.get_legend_handles_labels()
ax_d.legend(ld1 + ld2, lbd1 + lbd2, loc="upper left", ncol=2, fontsize=8.5)

snr_series = smoothed_obs[:, 0] - smoothed_obs[:, 1]
ax_e.step(steps_arr, ai_mcs, where="post", color="#0077B6", lw=2.6, label="Trần Điều Chế PPO Chọn (HE-MCS 0..11) [Link Adaptation]")
ax_e_tw = ax_e.twinx()
ax_e_tw.plot(steps_arr, snr_series, color="#023E8A", ls=":", lw=1.5, alpha=0.8, label="Tỷ số SNR Kênh Đang Chọn (avg_rssi - noise_on_chosen_ch dB)")
ax_e.set_yticks([1, 3, 5, 7, 9, 11]); ax_e.set_ylim(0, 12.2)
ax_e.set_ylabel("Trần Điều Chế (HE-MCS)", color="#0077B6", fontweight="bold")
ax_e_tw.set_ylabel("SNR Kênh Đang Chọn (dB)", color="#023E8A", fontweight="bold")
ax_e.set_title("BƯỚC 4.6 (E): THÍCH ỨNG TRẦN ĐIỀU CHẾ (HE-MCS CEILING) THUẦN PPO THEO SNR ĐÓNG", fontsize=11.5, fontweight="bold")
le1, lbe1 = ax_e.get_legend_handles_labels(); le2, lbe2 = ax_e_tw.get_legend_handles_labels()
ax_e.legend(le1 + le2, lbe1 + lbe2, loc="upper left", ncol=2, fontsize=8.5)

ax_f.step(steps_arr, ai_obss, where="post", color="#2A9D8F", lw=2.6, label="Ngưỡng OBSS-PD PPO Chọn (-82..-62 dBm) [Spatial Reuse]")
ax_f.scatter(steps_arr[::12], [-61.2 if b == 1 else -83.8 for b in ai_bf[::12]], color="#7209B7", marker="^", s=38, label="Beamforming PPO Chọn (ON=-61dBm / OFF=-84dBm)")
ax_f_tw = ax_f.twinx()
ax_f_tw.step(steps_arr, ai_bcn, where="post", color="#E76F51", ls="--", lw=2.2, label="Chu kỳ Beacon PPO Chọn (80ms / 300ms)")
ax_f.set_yticks([-82, -77, -72, -67, -62]); ax_f.set_ylim(-85, -59.5)
ax_f_tw.set_yticks([80, 300]); ax_f_tw.set_ylim(40, 350)
ax_f.set_ylabel("Ngưỡng OBSS-PD (dBm)", color="#2A9D8F", fontweight="bold")
ax_f_tw.set_ylabel("Chu Kỳ Beacon (ms)", color="#E76F51", fontweight="bold")
ax_f.set_xlabel("Bước thời gian (Steps - Chuỗi Stress Test 600 bước)", fontweight="bold")
ax_f.set_title("BƯỚC 4.6 (F): ĐIỀU PHỐI OBSS-PD, CHU KỲ BEACON & BEAMFORMING", fontsize=11.5, fontweight="bold")
lf1, lbf1 = ax_f.get_legend_handles_labels(); lf2, lbf2 = ax_f_tw.get_legend_handles_labels()
ax_f.legend(lf1 + lf2, lbf1 + lbf2, loc="lower left", ncol=2, fontsize=8.5)

plt.tight_layout()
aux_plot_path = os.path.join(log_dir, "auxiliary_5_actions_audit.png")
plt.savefig(aux_plot_path, dpi=300, bbox_inches="tight")
plt.show()
print(f"[+] Đã lưu đồ thị Stress Test (D-F) tại: {aux_plot_path}")


[BƯỚC 4.6 - CHẾ ĐỘ 1] KIỂM ĐỊNH CHUỖI THỜI GIAN THỰC TẾ: data_source_mode = 'REAL_KAGGLE_CSV'
   -> Nguồn wifi_dataset_cleaned.csv: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/data_train_digital_twin/Wi-Fi 2.4GHz RF Telemetry & Optimizer/wifi_dataset_cleaned.csv
   -> Nguồn TrainingData.csv        : /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/data_train_digital_twin/UjiIndoorLoc An indoor localization dataset/TrainingData.csv
   -> Nguồn ValidationData.csv      : /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/data_train_digital_twin/UjiIndoorLoc An indoor localization dataset/ValidationData.csv


  [+] PPO Dual-Sensing (REAL_KAGGLE_CSV): Goodput = 76.87 Mbps | P99 TB = 3.52 ms | PooledStep-P99 = 19.58 ms | Loss = 0.29% | QoE SLA = 96.8% | Kênh dùng = [36, 44, 48, 149, 153, 157, 161, 165]
  [-] Static Legacy N  (REAL_KAGGLE_CSV): Goodput = 1.07 Mbps | P99 TB = 3581.97 ms | PooledStep-P99 = 4191.48 ms | Loss = 95.38% | QoE SLA = 0.0%


[+] Đã lưu đồ thị Replay Dữ liệu Thực tế tại: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/wifi_rl_logs/kaggle_real_trace_replay.png



 [BƯỚC 4.6 - CHẾ ĐỘ 2] STRESS TEST XUNG NHIỄU 600 BƯỚC THUẦN PPO (ZERO ORACLE OVERWRITE - CLOSED-LOOP)
  * Lúc Bình yên (Step 50)                  : AI Chọn Kênh  36 | 80 MHz | Tx=16dBm | MCS= 8 | Bcn=300ms | OBSS=-62dBm | BF=ON  --> Goodput=144.1Mbps, Lat99=2.19ms, Loss=0.00%
  * Đợt 1 - Nhiễu CH36-40 (Step 150)        : AI Chọn Kênh 161 | 80 MHz | Tx=18dBm | MCS= 6 | Bcn=300ms | OBSS=-77dBm | BF=ON  --> Goodput= 74.8Mbps, Lat99=2.52ms, Loss=0.00%
  * Đợt 2 - Bão Roaming 125 STA (Step 350)  : AI Chọn Kênh 161 | 20 MHz | Tx=20dBm | MCS= 3 | Bcn= 80ms | OBSS=-82dBm | BF=OFF --> Goodput= 12.4Mbps, Lat99=6.66ms, Loss=0.00%
  * Hồi phục Bình yên (Step 450)            : AI Chọn Kênh 161 | 80 MHz | Tx=16dBm | MCS= 8 | Bcn=300ms | OBSS=-62dBm | BF=ON  --> Goodput=143.9Mbps, Lat99=2.18ms, Loss=0.00%
  * Đợt 3 - Sập UNII-1 -> UNII-3 (Step 550) : AI Chọn Kênh 161 | 40 MHz | Tx=20dBm | MCS= 2 | Bcn= 80ms | OBSS=-82dBm | BF=OFF --> Goodput= 22.8Mbps, Lat99=8.72ms, Loss=0.69%
---------------------

[+] Đã lưu đồ thị Stress Test (A-C) tại: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/wifi_rl_logs/pulse_chaos_test.png


[+] Đã lưu đồ thị Stress Test (D-F) tại: /home/chigguaymeovang300475/KHKT_2026_WIFI_RL/notebooks/KHKT_training_digital_twin_8DState_7DAct/wifi_rl_logs/auxiliary_5_actions_audit.png
